# Crystallized World Models

*Prospective validity, homeostatic recovery, and MDL-gated symbolic revision*

This cleaned notebook preserves every nonempty experiment from `Untitled18.ipynb` in its original order. The archive was a sequence of standalone, single-cell programs rather than a top-to-bottom analysis. Each section below remains independently runnable after the setup cell.

## Reproducibility contract

- Original SHA-256: `f7e3d9e60f7a49dcc393ccef28260d291b2cdeb10a1547b25cdd7c09eaeeb7bd`. Original cell indices are recorded in every code cell's metadata and section header.
- Recorded outputs were removed to reduce file size; all positive, negative, null, failed, and warning outcomes are preserved in the accompanying research record and website diary.
- Concrete `/content/...` and `outputs/...` paths were moved to unique `outputs/<experiment>/` directories. Algorithms, seeds, sample sizes, thresholds, and metrics are otherwise preserved except for the explicitly listed source repairs.
- Run one experiment section at a time. Several full settings require a T4/A100-class GPU or hours of CPU/GPU time; fast/dev environment variables are smoke tests, not replacements for registered full runs.
- The source archive is historical evidence, not a preregistration. An observed output may have been produced by an earlier source revision when the output and current cell disagree; such cases are called out in the research record.


In [ ]:
from pathlib import Path
import platform

_experiment_output_dirs = ['01-original-crystallization-positive-control', '02-corrected-exact-horizon-planning-protocol', '03-prospective-validity-v2', '04-homeostatic-symbolic-recovery-v3', '05-runtime-mdl-refinement-v4', '06-calibrated-compression-gate-v5', '07-mdl-gated-symbolic-rule-invention-v6']
for _experiment in _experiment_output_dirs:
    Path("outputs", _experiment).mkdir(parents=True, exist_ok=True)

print(f"Python {platform.python_version()}")
print(f"Artifact root: {Path('outputs').resolve()}")


## Experiment 1: Original crystallization positive control

Original cell `0`.

**Audit note.** Historical easy planning protocol; use experiment 2 for the corrected planning claim.


In [ ]:
"""
CRYSTALLIZED WORLD MODEL — T4-OPTIMIZED, SINGLE-CELL EXPERIMENT

Paste this entire file into one Google Colab cell and run it with a T4 GPU.

Primary question
----------------
Can a deterministic finite transition system, queried from a learned recurrent
world model, retain accurate long-horizon dynamics and planning after the
neural model's free-running rollouts begin to drift? Can neural/finite-machine
disagreement detect an unmodelled dynamics change online?

Interpretation boundary
-----------------------
This is a positive-control study of *crystallization*, not autonomous state
discovery. The abstract physical state (quantized position and velocity) and
action alphabet are supplied as supervised labels. The transition table is
queried only from the trained neural model; the true transition function is
used solely for data generation and evaluation.

The cell saves:
  - config.json
  - results.json
  - rollout_metrics.csv
  - training_history.csv
  - crystallized_machine.npz
  - world_model.pt
  - summary.png
  - run_bundle.zip

For a quick CPU/GPU path check, set FAST_DEV_RUN = True below.
"""

from __future__ import annotations

import contextlib
import csv
import dataclasses
import json
import math
import os
import random
import shutil
import time
from collections import deque
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# =============================================================================
# 0. FROZEN CONFIGURATION
# =============================================================================

FAST_DEV_RUN = os.environ.get("CRYSTAL_FAST_DEV_RUN", "0") == "1"


@dataclass(frozen=True)
class Config:
    seed: int = 1729

    # Controlled, quantized 1-D mechanics.
    n_pos: int = 13
    v_max: int = 2
    n_actions: int = 3                 # acceleration in {-1, 0, +1}

    # Observation: clean mechanics channels plus an irrelevant noise channel.
    image_size: int = 16
    distractor_probability: float = 0.10
    distractor_amplitude: float = 1.0

    # Neural world model.
    hidden_size: int = 128
    action_embed_dim: int = 24
    train_seq_len: int = 14
    train_steps: int = 1800
    train_batch_size: int = 768
    learning_rate: float = 2.0e-3
    weight_decay: float = 1.0e-4
    grad_clip: float = 1.0
    initial_state_loss_weight: float = 0.35

    # Crystallization queries: nuisance variants per abstract state/action.
    extraction_nuisance_samples: int = 96

    # Long-horizon evaluation.
    rollout_batch_size: int = 2048
    rollout_horizons: Tuple[int, ...] = (1, 2, 4, 8, 16, 32, 64, 128)

    # Planning. Neural beam search is used instead of serial MCTS because it
    # batches the model calls efficiently on a T4. Both planners are open-loop.
    planning_cases: int = 160
    planning_max_depth: int = 28
    neural_beam_width: int = 128

    # Validity monitoring. At a hidden changepoint, persistent unmodelled wind
    # (+1 or -1 acceleration) alters the mechanics.
    validity_batch_size: int = 2048
    validity_horizon: int = 28
    validity_nominal_fraction: float = 0.50
    validity_threshold_quantile: float = 0.99
    validity_encode_chunk: int = 8192

    # Runtime/output.
    use_amp: bool = True
    allow_tf32: bool = True
    output_dir: str = "outputs/01-original-crystallization-positive-control/crystallized_world_model_results"

    @property
    def n_vel(self) -> int:
        return 2 * self.v_max + 1

    @property
    def n_states(self) -> int:
        return self.n_pos * self.n_vel


cfg = Config()
if FAST_DEV_RUN:
    cfg = dataclasses.replace(
        cfg,
        hidden_size=48,
        train_seq_len=5,
        train_steps=12,
        train_batch_size=32,
        extraction_nuisance_samples=4,
        rollout_batch_size=48,
        rollout_horizons=(1, 2, 4, 8),
        planning_cases=8,
        planning_max_depth=8,
        neural_beam_width=12,
        validity_batch_size=48,
        validity_horizon=8,
        validity_encode_chunk=256,
        output_dir=str(Path.cwd() / "crystallized_world_model_smoke"),
    )


# =============================================================================
# 1. REPRODUCIBILITY AND T4 RUNTIME SETUP
# =============================================================================

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
is_cuda = device.type == "cuda"

if is_cuda:
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = cfg.allow_tf32
    torch.backends.cudnn.allow_tf32 = cfg.allow_tf32
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

amp_enabled = bool(cfg.use_amp and is_cuda)
amp_context = (
    (lambda: torch.autocast(device_type="cuda", dtype=torch.float16))
    if amp_enabled
    else contextlib.nullcontext
)

try:
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

print("=" * 88)
print("CRYSTALLIZED WORLD MODEL — finite dynamics inside a recurrent neural model")
print("=" * 88)
print(f"device={device} | torch={torch.__version__} | AMP={amp_enabled} | fast_dev={FAST_DEV_RUN}")
if is_cuda:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU={props.name} | VRAM={props.total_memory / 2**30:.1f} GiB")
else:
    print("WARNING: full defaults are intended for a Colab T4. Set CRYSTAL_FAST_DEV_RUN=1 on CPU.")


# =============================================================================
# 2. TRUE ENVIRONMENT (GENERATION AND EVALUATION ONLY)
# =============================================================================

ACTION_VALUES = torch.tensor([-1, 0, 1], dtype=torch.long, device=device)


def encode_state(position: torch.Tensor, velocity: torch.Tensor) -> torch.Tensor:
    """Map (position, velocity) to one integer abstract state."""
    return position.long() * cfg.n_vel + (velocity.long() + cfg.v_max)


def decode_state(state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    position = torch.div(state.long(), cfg.n_vel, rounding_mode="floor")
    velocity = torch.remainder(state.long(), cfg.n_vel) - cfg.v_max
    return position, velocity


def true_step(
    state: torch.Tensor,
    action_index: torch.Tensor,
    exogenous_wind: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """
    Controlled mechanics:
        v' = clip(v + action + wind, -v_max, v_max)
        x' = x + v'
    Reflecting walls reverse velocity. With v_max << n_pos, one reflection is enough.
    The persistent wind is withheld from the learned model and creates a validity exit.
    """
    position, velocity = decode_state(state)
    acceleration = ACTION_VALUES[action_index.long()]
    if exogenous_wind is None:
        exogenous_wind = torch.zeros_like(acceleration)

    velocity_next = torch.clamp(
        velocity + acceleration + exogenous_wind.long(),
        min=-cfg.v_max,
        max=cfg.v_max,
    )
    position_next = position + velocity_next

    hit_left = position_next < 0
    position_next = torch.where(hit_left, -position_next, position_next)
    velocity_next = torch.where(hit_left, -velocity_next, velocity_next)

    hit_right = position_next >= cfg.n_pos
    position_next = torch.where(
        hit_right,
        2 * (cfg.n_pos - 1) - position_next,
        position_next,
    )
    velocity_next = torch.where(hit_right, -velocity_next, velocity_next)

    if not bool(((position_next >= 0) & (position_next < cfg.n_pos)).all()):
        raise RuntimeError("Reflection invariant failed: position escaped the grid.")
    return encode_state(position_next, velocity_next)


def random_states(batch: int) -> torch.Tensor:
    return torch.randint(0, cfg.n_states, (batch,), device=device)


def random_actions(batch: int, horizon: int) -> torch.Tensor:
    return torch.randint(0, cfg.n_actions, (batch, horizon), device=device)


def rollout_true(
    initial_state: torch.Tensor,
    actions: torch.Tensor,
    winds: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Return q_0 ... q_T with shape [batch, T+1]."""
    states = [initial_state]
    current = initial_state
    for t in range(actions.shape[1]):
        wind_t = None if winds is None else winds[:, t]
        current = true_step(current, actions[:, t], wind_t)
        states.append(current)
    return torch.stack(states, dim=1)


def render_observation(
    state: torch.Tensor,
    distractor_probability: Optional[float] = None,
) -> torch.Tensor:
    """
    Render three channels without CPU transfer:
      0. ball position,
      1. velocity gauge,
      2. iid distractor pixels unrelated to the mechanics.
    """
    if distractor_probability is None:
        distractor_probability = cfg.distractor_probability

    state = state.reshape(-1)
    batch = state.shape[0]
    size = cfg.image_size
    obs = torch.zeros((batch, 3, size, size), device=device, dtype=torch.float32)
    position, velocity = decode_state(state)

    # Map the discrete track to interior pixel columns.
    pixel_x = 1 + torch.round(
        position.float() * float(size - 3) / float(cfg.n_pos - 1)
    ).long()
    center_y = size // 2
    batch_index = torch.arange(batch, device=device)

    # A small 3-pixel ball makes the physical signal nontrivial but visible.
    obs[batch_index, 0, center_y, pixel_x] = 1.0
    obs[batch_index, 0, center_y - 1, pixel_x] = 0.75
    obs[batch_index, 0, center_y + 1, pixel_x] = 0.75

    # The velocity gauge makes (x,v) observable from one frame. It is a supplied
    # grounding interface, not a claim of unsupervised causal-state discovery.
    gauge_x = (size // 2 - cfg.v_max) + (velocity + cfg.v_max)
    obs[batch_index, 1, size - 2, gauge_x] = 1.0
    obs[batch_index, 1, size - 3, size // 2] = 0.35

    if distractor_probability > 0.0:
        mask = torch.rand((batch, size, size), device=device) < distractor_probability
        values = torch.rand((batch, size, size), device=device) * cfg.distractor_amplitude
        obs[:, 2] = mask.float() * values

    return obs.contiguous(memory_format=torch.channels_last)


# Exhaustive true table is evaluation-only. It is never passed to extraction.
with torch.no_grad():
    _all_q = torch.arange(cfg.n_states, device=device).repeat_interleave(cfg.n_actions)
    _all_a = torch.arange(cfg.n_actions, device=device).repeat(cfg.n_states)
    TRUE_TABLE = true_step(_all_q, _all_a).reshape(cfg.n_states, cfg.n_actions)


# =============================================================================
# 3. NEURAL RECURRENT WORLD MODEL
# =============================================================================

class NeuralWorldModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.SiLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.SiLU(inplace=True),
            nn.Conv2d(64, 96, 3, stride=2, padding=1),
            nn.SiLU(inplace=True),
        )
        encoded_side = math.ceil(cfg.image_size / 8)
        self.encoder_proj = nn.Sequential(
            nn.Flatten(),
            # Preserve the final 2-D layout. Global pooling would erase the very
            # position variable the mechanics requires.
            nn.Linear(96 * encoded_side * encoded_side, cfg.hidden_size),
            nn.Tanh(),
        )
        self.action_embedding = nn.Embedding(cfg.n_actions, cfg.action_embed_dim)
        self.gru = nn.GRU(cfg.action_embed_dim, cfg.hidden_size, batch_first=True)
        self.state_head = nn.Linear(cfg.hidden_size, cfg.n_states)

    def encode_hidden(self, observation: torch.Tensor) -> torch.Tensor:
        return self.encoder_proj(self.encoder_conv(observation))

    def initial_logits(self, observation: torch.Tensor) -> torch.Tensor:
        return self.state_head(self.encode_hidden(observation))

    def rollout_from_hidden(
        self,
        hidden: torch.Tensor,
        actions: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        embedded = self.action_embedding(actions)
        outputs, final_hidden = self.gru(embedded, hidden.unsqueeze(0))
        return self.state_head(outputs), final_hidden.squeeze(0)

    def forward(
        self,
        initial_observation: torch.Tensor,
        actions: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        hidden = self.encode_hidden(initial_observation)
        initial_logits = self.state_head(hidden)
        rollout_logits, _ = self.rollout_from_hidden(hidden, actions)
        return initial_logits, rollout_logits


model = NeuralWorldModel().to(device)
if is_cuda:
    model.encoder_conv = model.encoder_conv.to(memory_format=torch.channels_last)

optimizer_kwargs = dict(lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
try:
    optimizer = torch.optim.AdamW(model.parameters(), fused=is_cuda, **optimizer_kwargs)
except (TypeError, RuntimeError):
    optimizer = torch.optim.AdamW(model.parameters(), **optimizer_kwargs)

parameter_count = sum(p.numel() for p in model.parameters())
print(f"world-model parameters={parameter_count:,}")


# =============================================================================
# 4. TRAINING
# =============================================================================

training_history: List[Dict[str, float]] = []
model.train()
train_start = time.perf_counter()

for step in range(1, cfg.train_steps + 1):
    q0 = random_states(cfg.train_batch_size)
    action_seq = random_actions(cfg.train_batch_size, cfg.train_seq_len)
    target_states = rollout_true(q0, action_seq)
    obs0 = render_observation(q0)

    optimizer.zero_grad(set_to_none=True)
    with amp_context():
        initial_logits, future_logits = model(obs0, action_seq)
        loss_initial = F.cross_entropy(initial_logits.float(), target_states[:, 0])
        loss_future = F.cross_entropy(
            future_logits.float().reshape(-1, cfg.n_states),
            target_states[:, 1:].reshape(-1),
        )
        loss = loss_future + cfg.initial_state_loss_weight * loss_initial

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
    scaler.step(optimizer)
    scaler.update()

    if step == 1 or step % max(1, cfg.train_steps // 40) == 0 or step == cfg.train_steps:
        with torch.no_grad():
            initial_acc = (initial_logits.argmax(-1) == target_states[:, 0]).float().mean().item()
            final_acc = (future_logits[:, -1].argmax(-1) == target_states[:, -1]).float().mean().item()
        row = {
            "step": float(step),
            "loss": float(loss.item()),
            "initial_accuracy": float(initial_acc),
            "train_horizon_accuracy": float(final_acc),
        }
        training_history.append(row)
        print(
            f"train {step:4d}/{cfg.train_steps} | loss={row['loss']:.4f} | "
            f"q0={100*initial_acc:5.1f}% | t{cfg.train_seq_len}={100*final_acc:5.1f}%"
        )

train_seconds = time.perf_counter() - train_start
model.eval()
print(f"training elapsed={train_seconds:.1f}s")


# =============================================================================
# 5. QUERY AND CRYSTALLIZE A DETERMINISTIC FINITE TRANSITION SYSTEM
# =============================================================================

@torch.inference_mode()
def extract_machine() -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    For each supplied abstract state and action, query the *neural model* across
    nuisance-rendered observations. Majority/mean-probability crystallization
    produces one deterministic successor. No true transition is read here.
    """
    transitions = torch.empty((cfg.n_states, cfg.n_actions), dtype=torch.long, device=device)
    confidence = torch.empty_like(transitions, dtype=torch.float32)
    stability = torch.empty_like(transitions, dtype=torch.float32)

    for q in range(cfg.n_states):
        q_batch = torch.full(
            (cfg.extraction_nuisance_samples,), q, dtype=torch.long, device=device
        )
        observations = render_observation(q_batch)
        hidden = model.encode_hidden(observations)
        for action in range(cfg.n_actions):
            actions = torch.full(
                (cfg.extraction_nuisance_samples, 1),
                action,
                dtype=torch.long,
                device=device,
            )
            logits, _ = model.rollout_from_hidden(hidden, actions)
            probabilities = logits[:, 0].float().softmax(-1)
            mean_probability = probabilities.mean(0)
            successor = mean_probability.argmax()
            transitions[q, action] = successor
            confidence[q, action] = mean_probability[successor]
            stability[q, action] = (
                probabilities.argmax(-1) == successor
            ).float().mean()
    return transitions, confidence, stability


MACHINE_TABLE, MACHINE_CONFIDENCE, MACHINE_STABILITY = extract_machine()
machine_transition_accuracy = (MACHINE_TABLE == TRUE_TABLE).float().mean().item()
machine_min_confidence = MACHINE_CONFIDENCE.min().item()
machine_mean_confidence = MACHINE_CONFIDENCE.mean().item()
machine_min_stability = MACHINE_STABILITY.min().item()

print("\nCRYSTALLIZATION AUDIT")
print("-" * 88)
print(f"finite states={cfg.n_states} | actions={cfg.n_actions} | transitions={cfg.n_states*cfg.n_actions}")
print(f"transition accuracy against withheld simulator={100*machine_transition_accuracy:.2f}%")
print(f"mean/min neural confidence={machine_mean_confidence:.4f}/{machine_min_confidence:.4f}")
print(f"minimum nuisance stability={machine_min_stability:.4f}")


# =============================================================================
# 6. LONG-HORIZON ROLLOUT: RECURRENT MODEL VS CRYSTALLIZED MACHINE
# =============================================================================

@torch.inference_mode()
def rollout_machine(initial_state: torch.Tensor, actions: torch.Tensor) -> torch.Tensor:
    states = []
    current = initial_state
    for t in range(actions.shape[1]):
        current = MACHINE_TABLE[current, actions[:, t]]
        states.append(current)
    return torch.stack(states, dim=1)


@torch.inference_mode()
def evaluate_rollouts() -> List[Dict[str, float]]:
    max_horizon = max(cfg.rollout_horizons)
    q0 = random_states(cfg.rollout_batch_size)
    actions = random_actions(cfg.rollout_batch_size, max_horizon)
    truth = rollout_true(q0, actions)
    obs0 = render_observation(q0)

    with amp_context():
        initial_logits, neural_logits = model(obs0, actions)
    neural_pred = neural_logits.float().argmax(-1)
    q0_hat = initial_logits.float().argmax(-1)
    machine_pred = rollout_machine(q0_hat, actions)
    perception_correct = q0_hat == q0

    rows: List[Dict[str, float]] = []
    for horizon in cfg.rollout_horizons:
        target = truth[:, horizon]
        neural_q = neural_pred[:, horizon - 1]
        machine_q = machine_pred[:, horizon - 1]
        target_p, _ = decode_state(target)
        neural_p, _ = decode_state(neural_q)
        machine_p, _ = decode_state(machine_q)

        if bool(perception_correct.any()):
            neural_cond = (neural_q[perception_correct] == target[perception_correct]).float().mean()
            machine_cond = (machine_q[perception_correct] == target[perception_correct]).float().mean()
        else:
            neural_cond = torch.tensor(float("nan"), device=device)
            machine_cond = torch.tensor(float("nan"), device=device)

        rows.append(
            {
                "horizon": float(horizon),
                "neural_state_accuracy": float((neural_q == target).float().mean().item()),
                "machine_state_accuracy": float((machine_q == target).float().mean().item()),
                "neural_state_accuracy_given_correct_q0": float(neural_cond.item()),
                "machine_state_accuracy_given_correct_q0": float(machine_cond.item()),
                "neural_position_mae": float((neural_p - target_p).abs().float().mean().item()),
                "machine_position_mae": float((machine_p - target_p).abs().float().mean().item()),
            }
        )
    return rows


rollout_rows = evaluate_rollouts()
print("\nLONG-HORIZON ROLLOUT")
print("-" * 88)
print(" horizon | neural state acc | machine state acc | neural pos MAE | machine pos MAE")
for row in rollout_rows:
    print(
        f" {int(row['horizon']):7d} |"
        f" {100*row['neural_state_accuracy']:15.2f}% |"
        f" {100*row['machine_state_accuracy']:16.2f}% |"
        f" {row['neural_position_mae']:14.3f} |"
        f" {row['machine_position_mae']:15.3f}"
    )

crossover_horizon: Optional[int] = None
for row in rollout_rows:
    if row["machine_state_accuracy"] > row["neural_state_accuracy"] + 1e-9:
        crossover_horizon = int(row["horizon"])
        break


# =============================================================================
# 7. PLANNING: MACHINE BFS VS BATCHED NEURAL BEAM SEARCH
# =============================================================================

def machine_bfs_plan(start_state: int, target_position: int) -> Optional[List[int]]:
    """Shortest open-loop plan in the crystallized finite graph."""
    if start_state // cfg.n_vel == target_position:
        return []
    queue = deque([start_state])
    parent: Dict[int, Tuple[int, int]] = {}
    visited = {start_state}

    found: Optional[int] = None
    while queue:
        q = queue.popleft()
        for action in range(cfg.n_actions):
            q_next = int(MACHINE_TABLE[q, action].item())
            if q_next in visited:
                continue
            visited.add(q_next)
            parent[q_next] = (q, action)
            if q_next // cfg.n_vel == target_position:
                found = q_next
                queue.clear()
                break
            queue.append(q_next)

    if found is None:
        return None

    actions: List[int] = []
    cursor = found
    while cursor != start_state:
        previous, action = parent[cursor]
        actions.append(action)
        cursor = previous
    actions.reverse()
    return actions


@torch.inference_mode()
def neural_beam_plan(start_state: int, target_position: int) -> Optional[List[int]]:
    """
    GPU-batched beam search over recurrent hidden states. This is intentionally
    not called MCTS: it is the T4-efficient neural-rollout planner in this cell.
    """
    q_tensor = torch.tensor([start_state], device=device)
    obs = render_observation(q_tensor)
    with amp_context():
        hidden = model.encode_hidden(obs)

    sequences = torch.empty((1, 0), dtype=torch.long, device=device)
    cumulative_log_confidence = torch.zeros(1, device=device)

    for depth in range(1, cfg.planning_max_depth + 1):
        beam_count = hidden.shape[0]
        expanded_hidden = hidden.repeat_interleave(cfg.n_actions, dim=0)
        expanded_actions = torch.arange(cfg.n_actions, device=device).repeat(beam_count)

        with amp_context():
            logits, next_hidden = model.rollout_from_hidden(
                expanded_hidden,
                expanded_actions[:, None],
            )
        probabilities = logits[:, 0].float().softmax(-1)
        predicted_q = probabilities.argmax(-1)
        confidence = probabilities.gather(1, predicted_q[:, None]).squeeze(1).clamp_min(1e-8)
        predicted_position, _ = decode_state(predicted_q)

        expanded_sequences = torch.cat(
            [
                sequences.repeat_interleave(cfg.n_actions, dim=0),
                expanded_actions[:, None],
            ],
            dim=1,
        )
        expanded_log_confidence = (
            cumulative_log_confidence.repeat_interleave(cfg.n_actions)
            + confidence.log()
        )

        reaches_target = predicted_position == target_position
        if bool(reaches_target.any()):
            candidate_ids = torch.nonzero(reaches_target, as_tuple=False).squeeze(1)
            best = candidate_ids[expanded_log_confidence[candidate_ids].argmax()]
            return expanded_sequences[best].detach().cpu().tolist()

        # Distance drives progress; confidence breaks ties and suppresses paths
        # that the neural dynamics itself considers implausible.
        distance = (predicted_position - target_position).abs().float()
        score = -distance + 0.03 * expanded_log_confidence
        keep = min(cfg.neural_beam_width, score.numel())
        selected = torch.topk(score, k=keep, largest=True).indices
        hidden = next_hidden[selected]
        sequences = expanded_sequences[selected]
        cumulative_log_confidence = expanded_log_confidence[selected]

    return None


@torch.inference_mode()
def execute_plan(start_state: int, actions: Optional[Sequence[int]], target_position: int) -> bool:
    if actions is None:
        return False
    q = torch.tensor([start_state], dtype=torch.long, device=device)
    if int(decode_state(q)[0].item()) == target_position:
        return True
    for action in actions:
        a = torch.tensor([action], dtype=torch.long, device=device)
        q = true_step(q, a)
        if int(decode_state(q)[0].item()) == target_position:
            return True
    return False


planning_rng = np.random.default_rng(cfg.seed + 7)
machine_successes: List[float] = []
neural_successes: List[float] = []
machine_lengths: List[float] = []
neural_lengths: List[float] = []

planning_start = time.perf_counter()
for _ in range(cfg.planning_cases):
    start_state = int(planning_rng.integers(0, cfg.n_states))
    start_position = start_state // cfg.n_vel
    possible_targets = [p for p in range(cfg.n_pos) if p != start_position]
    target_position = int(planning_rng.choice(possible_targets))

    machine_plan = machine_bfs_plan(start_state, target_position)
    neural_plan = neural_beam_plan(start_state, target_position)

    machine_successes.append(float(execute_plan(start_state, machine_plan, target_position)))
    neural_successes.append(float(execute_plan(start_state, neural_plan, target_position)))
    if machine_plan is not None:
        machine_lengths.append(float(len(machine_plan)))
    if neural_plan is not None:
        neural_lengths.append(float(len(neural_plan)))

planning_seconds = time.perf_counter() - planning_start
planning_results = {
    "cases": cfg.planning_cases,
    "machine_success_rate": float(np.mean(machine_successes)),
    "neural_beam_success_rate": float(np.mean(neural_successes)),
    "machine_median_plan_length": float(np.median(machine_lengths)) if machine_lengths else None,
    "neural_beam_median_plan_length": float(np.median(neural_lengths)) if neural_lengths else None,
    "elapsed_seconds": planning_seconds,
}

print("\nOPEN-LOOP PLANNING")
print("-" * 88)
print(
    f"machine BFS success={100*planning_results['machine_success_rate']:.2f}% | "
    f"median length={planning_results['machine_median_plan_length']}"
)
print(
    f"neural beam success={100*planning_results['neural_beam_success_rate']:.2f}% | "
    f"median length={planning_results['neural_beam_median_plan_length']}"
)


# =============================================================================
# 8. ONLINE VALIDITY MONITORING UNDER A HIDDEN DYNAMICS CHANGE
# =============================================================================

def binary_auroc(labels: np.ndarray, scores: np.ndarray) -> float:
    labels = labels.astype(np.int64)
    positives = int(labels.sum())
    negatives = int((1 - labels).sum())
    if positives == 0 or negatives == 0:
        return float("nan")
    order = np.argsort(-scores, kind="mergesort")
    y = labels[order]
    tpr = np.r_[0.0, np.cumsum(y) / positives, 1.0]
    fpr = np.r_[0.0, np.cumsum(1 - y) / negatives, 1.0]
    if hasattr(np, "trapezoid"):
        return float(np.trapezoid(tpr, fpr))
    return float(np.trapz(tpr, fpr))


def binary_auprc(labels: np.ndarray, scores: np.ndarray) -> float:
    labels = labels.astype(np.int64)
    positives = int(labels.sum())
    if positives == 0:
        return float("nan")
    order = np.argsort(-scores, kind="mergesort")
    y = labels[order]
    tp = np.cumsum(y)
    fp = np.cumsum(1 - y)
    recall = tp / positives
    precision = tp / np.maximum(tp + fp, 1)
    recall = np.r_[0.0, recall]
    precision = np.r_[1.0, precision]
    return float(np.sum((recall[1:] - recall[:-1]) * precision[1:]))


@torch.inference_mode()
def encode_state_probabilities(observations: torch.Tensor, chunk: int) -> torch.Tensor:
    outputs = []
    for start in range(0, observations.shape[0], chunk):
        with amp_context():
            logits = model.initial_logits(observations[start : start + chunk])
        outputs.append(logits.float().softmax(-1))
    return torch.cat(outputs, dim=0)


@torch.inference_mode()
def evaluate_validity() -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    batch = cfg.validity_batch_size
    horizon = cfg.validity_horizon
    q0 = random_states(batch)
    actions = random_actions(batch, horizon)

    nominal_count = int(round(batch * cfg.validity_nominal_fraction))
    has_exit = torch.zeros(batch, dtype=torch.bool, device=device)
    has_exit[nominal_count:] = True
    # Shuffle which sequences contain exits.
    has_exit = has_exit[torch.randperm(batch, device=device)]

    min_event = min(3, max(1, horizon // 3))
    max_event_exclusive = max(min_event + 1, horizon - 2)
    event_time = torch.randint(min_event, max_event_exclusive, (batch,), device=device)
    wind_direction = torch.where(
        torch.rand(batch, device=device) < 0.5,
        -torch.ones(batch, dtype=torch.long, device=device),
        torch.ones(batch, dtype=torch.long, device=device),
    )

    times = torch.arange(horizon, device=device)[None, :]
    active = has_exit[:, None] & (times >= event_time[:, None])
    winds = active.long() * wind_direction[:, None]
    actual = rollout_true(q0, actions, winds)

    # Counterfactual nominal successor from each actual current state. This asks
    # whether the hidden wind changes the next outcome on this particular step.
    nominal_next = true_step(
        actual[:, :-1].reshape(-1),
        actions.reshape(-1),
    ).reshape(batch, horizon)
    counterexample = actual[:, 1:] != nominal_next

    current_obs = render_observation(actual[:, :-1].reshape(-1))
    next_obs = render_observation(actual[:, 1:].reshape(-1))
    current_probs = encode_state_probabilities(current_obs, cfg.validity_encode_chunk)
    next_probs = encode_state_probabilities(next_obs, cfg.validity_encode_chunk)
    current_hat = current_probs.argmax(-1)
    predicted_machine_next = MACHINE_TABLE[current_hat, actions.reshape(-1)]

    # Surprise is bounded [0,1] and calibrated entirely on nominal steps.
    assigned_probability = next_probs.gather(1, predicted_machine_next[:, None]).squeeze(1)
    surprise = (1.0 - assigned_probability).reshape(batch, horizon)

    nominal_scores = surprise[~has_exit].detach().cpu().numpy().reshape(-1)
    threshold = float(np.quantile(nominal_scores, cfg.validity_threshold_quantile))
    alarms = surprise > threshold

    domain_labels = active.detach().cpu().numpy().reshape(-1).astype(np.int64)
    counterexample_labels = counterexample.detach().cpu().numpy().reshape(-1).astype(np.int64)
    score_np = surprise.detach().cpu().numpy().reshape(-1)

    # Event-level latency from the dynamics switch, even if clipping temporarily
    # makes the altered and nominal transition coincide.
    latencies: List[float] = []
    detected_within_1 = 0
    detected_within_3 = 0
    exit_indices = torch.nonzero(has_exit, as_tuple=False).squeeze(1)
    for i in exit_indices.tolist():
        t0 = int(event_time[i].item())
        post_alarm = torch.nonzero(alarms[i, t0:], as_tuple=False).squeeze(1)
        if post_alarm.numel() > 0:
            latency = int(post_alarm[0].item())
            latencies.append(float(latency))
            detected_within_1 += int(latency <= 1)
            detected_within_3 += int(latency <= 3)

    exit_n = max(1, int(has_exit.sum().item()))
    results = {
        "threshold": threshold,
        "nominal_false_positive_rate": float(alarms[~has_exit].float().mean().item()),
        "domain_exit_auroc": binary_auroc(domain_labels, score_np),
        "domain_exit_auprc": binary_auprc(domain_labels, score_np),
        "counterexample_auroc": binary_auroc(counterexample_labels, score_np),
        "counterexample_auprc": binary_auprc(counterexample_labels, score_np),
        "event_detection_rate": float(len(latencies) / exit_n),
        "detected_within_1_step": float(detected_within_1 / exit_n),
        "detected_within_3_steps": float(detected_within_3 / exit_n),
        "median_detection_latency": float(np.median(latencies)) if latencies else None,
        "positive_step_prevalence": float(domain_labels.mean()),
        "counterexample_step_prevalence": float(counterexample_labels.mean()),
    }
    return results, score_np, domain_labels


validity_results, validity_scores, validity_labels = evaluate_validity()
print("\nVALIDITY MONITOR")
print("-" * 88)
print(
    f"domain-exit AUROC={validity_results['domain_exit_auroc']:.3f} | "
    f"AUPRC={validity_results['domain_exit_auprc']:.3f} | "
    f"nominal FPR={100*validity_results['nominal_false_positive_rate']:.2f}%"
)
print(
    f"counterexample AUROC={validity_results['counterexample_auroc']:.3f} | "
    f"AUPRC={validity_results['counterexample_auprc']:.3f}"
)
print(
    f"event detection={100*validity_results['event_detection_rate']:.2f}% | "
    f"within 1 step={100*validity_results['detected_within_1_step']:.2f}% | "
    f"median latency={validity_results['median_detection_latency']}"
)


# =============================================================================
# 9. SAVE ARTIFACTS AND SUMMARY FIGURE
# =============================================================================

config_json = asdict(cfg)
config_json["rollout_horizons"] = list(cfg.rollout_horizons)

results = {
    "interpretation_boundary": (
        "The abstract (position, velocity) state labels and action alphabet were supplied. "
        "This tests neural-to-finite crystallization, long-horizon dynamics, planning, and "
        "validity monitoring; it does not test autonomous state/coarse-graining discovery."
    ),
    "runtime": {
        "device": str(device),
        "gpu": torch.cuda.get_device_name(0) if is_cuda else None,
        "torch_version": torch.__version__,
        "amp": amp_enabled,
        "parameter_count": parameter_count,
        "train_seconds": train_seconds,
    },
    "crystallization": {
        "n_states": cfg.n_states,
        "n_actions": cfg.n_actions,
        "transition_accuracy": machine_transition_accuracy,
        "mean_confidence": machine_mean_confidence,
        "min_confidence": machine_min_confidence,
        "min_nuisance_stability": machine_min_stability,
    },
    "rollout": {
        "rows": rollout_rows,
        "first_strict_machine_crossover_horizon": crossover_horizon,
    },
    "planning": planning_results,
    "validity": validity_results,
}

with (out_dir / "config.json").open("w", encoding="utf-8") as f:
    json.dump(config_json, f, indent=2)
with (out_dir / "results.json").open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

with (out_dir / "training_history.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(training_history[0].keys()))
    writer.writeheader()
    writer.writerows(training_history)

with (out_dir / "rollout_metrics.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rollout_rows[0].keys()))
    writer.writeheader()
    writer.writerows(rollout_rows)

np.savez_compressed(
    out_dir / "crystallized_machine.npz",
    transition_table=MACHINE_TABLE.detach().cpu().numpy(),
    query_confidence=MACHINE_CONFIDENCE.detach().cpu().numpy(),
    nuisance_stability=MACHINE_STABILITY.detach().cpu().numpy(),
)
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": config_json,
    },
    out_dir / "world_model.pt",
)

fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)

axes[0, 0].plot(
    [r["step"] for r in training_history],
    [r["loss"] for r in training_history],
    color="#4C78A8",
    linewidth=2,
)
axes[0, 0].set_title("Neural world-model training")
axes[0, 0].set_xlabel("optimizer step")
axes[0, 0].set_ylabel("loss")
axes[0, 0].grid(alpha=0.25)

h = np.array([r["horizon"] for r in rollout_rows])
axes[0, 1].plot(
    h,
    [r["neural_state_accuracy"] for r in rollout_rows],
    "o-",
    label="neural recurrent rollout",
    color="#E45756",
)
axes[0, 1].plot(
    h,
    [r["machine_state_accuracy"] for r in rollout_rows],
    "o-",
    label="crystallized machine",
    color="#54A24B",
)
axes[0, 1].set_xscale("log", base=2)
axes[0, 1].set_ylim(-0.03, 1.03)
axes[0, 1].set_title("Exact state accuracy vs rollout depth")
axes[0, 1].set_xlabel("horizon")
axes[0, 1].set_ylabel("accuracy")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.25)

axes[1, 0].plot(
    h,
    [r["neural_position_mae"] for r in rollout_rows],
    "o-",
    label="neural recurrent rollout",
    color="#E45756",
)
axes[1, 0].plot(
    h,
    [r["machine_position_mae"] for r in rollout_rows],
    "o-",
    label="crystallized machine",
    color="#54A24B",
)
axes[1, 0].set_xscale("log", base=2)
axes[1, 0].set_title("Position error vs rollout depth")
axes[1, 0].set_xlabel("horizon")
axes[1, 0].set_ylabel("mean absolute position error")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.25)

nominal_scores = validity_scores[validity_labels == 0]
exit_scores = validity_scores[validity_labels == 1]
bins = np.linspace(0.0, 1.0, 45)
axes[1, 1].hist(nominal_scores, bins=bins, density=True, alpha=0.65, label="nominal")
axes[1, 1].hist(exit_scores, bins=bins, density=True, alpha=0.65, label="domain exit")
axes[1, 1].axvline(validity_results["threshold"], color="black", linestyle="--", label="calibrated threshold")
axes[1, 1].set_title("Machine/observation disagreement")
axes[1, 1].set_xlabel("surprise = 1 - P(observed successor)")
axes[1, 1].set_ylabel("density")
axes[1, 1].legend()

fig.suptitle(
    "Crystallized finite dynamics inside a neural world model\n"
    f"transition accuracy={100*machine_transition_accuracy:.1f}% | "
    f"planning: machine={100*planning_results['machine_success_rate']:.1f}%, "
    f"neural={100*planning_results['neural_beam_success_rate']:.1f}%",
    fontsize=14,
)
fig.savefig(out_dir / "summary.png", dpi=180)
plt.show()

archive_path = out_dir / "run_bundle.zip"
if archive_path.exists():
    archive_path.unlink()
# Build outside the directory being archived; otherwise the ZIP can include its
# own growing byte stream on some shutil/zipfile versions.
temporary_archive_base = out_dir.parent / f".{out_dir.name}_run_bundle_tmp"
temporary_archive_path = temporary_archive_base.with_suffix(".zip")
if temporary_archive_path.exists():
    temporary_archive_path.unlink()
made_archive = shutil.make_archive(
    str(temporary_archive_base),
    "zip",
    root_dir=out_dir,
    base_dir=".",
)
shutil.move(made_archive, archive_path)

print("\n" + "=" * 88)
print("FINAL SUMMARY")
print("=" * 88)
print(f"machine transition accuracy: {100*machine_transition_accuracy:.2f}%")
print(f"first strict machine rollout crossover: {crossover_horizon}")
print(
    f"planning success — machine: {100*planning_results['machine_success_rate']:.2f}% | "
    f"neural beam: {100*planning_results['neural_beam_success_rate']:.2f}%"
)
print(
    f"validity AUROC: {validity_results['domain_exit_auroc']:.3f} | "
    f"counterexample AUROC: {validity_results['counterexample_auroc']:.3f}"
)
print(f"artifacts: {out_dir}")
print(f"bundle: {archive_path}")

if machine_transition_accuracy < 0.95:
    print(
        "\nAUDIT WARNING: the extracted one-step machine is below 95% transition accuracy. "
        "Do not interpret long-horizon differences as drift removal alone; rerun with more "
        "training or report this as a failed crystallization condition."
    )


## Experiment 2: Corrected exact-horizon planning protocol

Original cell `1`.

**Audit note.** Corrected primary planning protocol: exact terminal state at an exact sampled horizon.


In [ ]:
"""
CRYSTALLIZED WORLD MODEL — T4-OPTIMIZED, SINGLE-CELL EXPERIMENT

Paste this entire file into one Google Colab cell and run it with a T4 GPU.

Primary question
----------------
Can a deterministic finite transition system, queried from a learned recurrent
world model, retain accurate long-horizon dynamics and planning after the
neural model's free-running rollouts begin to drift? Can neural/finite-machine
disagreement detect an unmodelled dynamics change online?

Interpretation boundary
-----------------------
This is a positive-control study of *crystallization*, not autonomous state
discovery. The abstract physical state (quantized position and velocity) and
action alphabet are supplied as supervised labels. The transition table is
queried only from the trained neural model; the true transition function is
used solely for data generation and evaluation.

The cell saves:
  - config.json
  - results.json
  - rollout_metrics.csv
  - training_history.csv
  - crystallized_machine.npz
  - world_model.pt
  - summary.png
  - run_bundle.zip

For a quick CPU/GPU path check, set FAST_DEV_RUN = True below.
"""

from __future__ import annotations

import contextlib
import csv
import dataclasses
import json
import math
import os
import random
import shutil
import time
from collections import deque
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# =============================================================================
# 0. FROZEN CONFIGURATION
# =============================================================================

FAST_DEV_RUN = os.environ.get("CRYSTAL_FAST_DEV_RUN", "0") == "1"


@dataclass(frozen=True)
class Config:
    seed: int = 1729

    # Controlled, quantized 1-D mechanics.
    n_pos: int = 13
    v_max: int = 2
    n_actions: int = 3                 # acceleration in {-1, 0, +1}

    # Observation: clean mechanics channels plus an irrelevant noise channel.
    image_size: int = 16
    distractor_probability: float = 0.10
    distractor_amplitude: float = 1.0

    # Neural world model.
    hidden_size: int = 128
    action_embed_dim: int = 24
    train_seq_len: int = 14
    train_steps: int = 1800
    train_batch_size: int = 768
    learning_rate: float = 2.0e-3
    weight_decay: float = 1.0e-4
    grad_clip: float = 1.0
    initial_state_loss_weight: float = 0.35

    # Crystallization queries: nuisance variants per abstract state/action.
    extraction_nuisance_samples: int = 96

    # Long-horizon evaluation.
    rollout_batch_size: int = 2048
    rollout_horizons: Tuple[int, ...] = (1, 2, 4, 8, 16, 32, 64, 128)

    # Planning. Neural beam search is used instead of serial MCTS because it
    # batches the model calls efficiently on a T4. The machine uses exact graph
    # search in a time-expanded state space. Both planners are open-loop.
    planning_cases: int = 160
    planning_max_depth: int = 28
    neural_beam_width: int = 128

    # Validity monitoring. At a hidden changepoint, persistent unmodelled wind
    # (+1 or -1 acceleration) alters the mechanics.
    validity_batch_size: int = 2048
    validity_horizon: int = 28
    validity_nominal_fraction: float = 0.50
    validity_threshold_quantile: float = 0.99
    validity_encode_chunk: int = 8192

    # Runtime/output.
    use_amp: bool = True
    allow_tf32: bool = True
    output_dir: str = "outputs/02-corrected-exact-horizon-planning-protocol/crystallized_world_model_results"

    @property
    def n_vel(self) -> int:
        return 2 * self.v_max + 1

    @property
    def n_states(self) -> int:
        return self.n_pos * self.n_vel


cfg = Config()
if FAST_DEV_RUN:
    cfg = dataclasses.replace(
        cfg,
        hidden_size=48,
        train_seq_len=5,
        train_steps=int(os.environ.get("CRYSTAL_FAST_STEPS", "12")),
        train_batch_size=int(os.environ.get("CRYSTAL_FAST_BATCH", "32")),
        extraction_nuisance_samples=4,
        rollout_batch_size=48,
        rollout_horizons=(1, 2, 4, 8),
        planning_cases=int(os.environ.get("CRYSTAL_FAST_PLANNING_CASES", "8")),
        planning_max_depth=8,
        neural_beam_width=12,
        validity_batch_size=48,
        validity_horizon=8,
        validity_encode_chunk=256,
        output_dir=str(Path.cwd() / "crystallized_world_model_smoke"),
    )


# =============================================================================
# 1. REPRODUCIBILITY AND T4 RUNTIME SETUP
# =============================================================================

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
is_cuda = device.type == "cuda"

if is_cuda:
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = cfg.allow_tf32
    torch.backends.cudnn.allow_tf32 = cfg.allow_tf32
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

amp_enabled = bool(cfg.use_amp and is_cuda)
amp_context = (
    (lambda: torch.autocast(device_type="cuda", dtype=torch.float16))
    if amp_enabled
    else contextlib.nullcontext
)

try:
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

print("=" * 88)
print("CRYSTALLIZED WORLD MODEL — finite dynamics inside a recurrent neural model")
print("=" * 88)
print(f"device={device} | torch={torch.__version__} | AMP={amp_enabled} | fast_dev={FAST_DEV_RUN}")
if is_cuda:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU={props.name} | VRAM={props.total_memory / 2**30:.1f} GiB")
else:
    print("WARNING: full defaults are intended for a Colab T4. Set CRYSTAL_FAST_DEV_RUN=1 on CPU.")


# =============================================================================
# 2. TRUE ENVIRONMENT (GENERATION AND EVALUATION ONLY)
# =============================================================================

ACTION_VALUES = torch.tensor([-1, 0, 1], dtype=torch.long, device=device)


def encode_state(position: torch.Tensor, velocity: torch.Tensor) -> torch.Tensor:
    """Map (position, velocity) to one integer abstract state."""
    return position.long() * cfg.n_vel + (velocity.long() + cfg.v_max)


def decode_state(state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    position = torch.div(state.long(), cfg.n_vel, rounding_mode="floor")
    velocity = torch.remainder(state.long(), cfg.n_vel) - cfg.v_max
    return position, velocity


def true_step(
    state: torch.Tensor,
    action_index: torch.Tensor,
    exogenous_wind: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """
    Controlled mechanics:
        v' = clip(v + action + wind, -v_max, v_max)
        x' = x + v'
    Reflecting walls reverse velocity. With v_max << n_pos, one reflection is enough.
    The persistent wind is withheld from the learned model and creates a validity exit.
    """
    position, velocity = decode_state(state)
    acceleration = ACTION_VALUES[action_index.long()]
    if exogenous_wind is None:
        exogenous_wind = torch.zeros_like(acceleration)

    velocity_next = torch.clamp(
        velocity + acceleration + exogenous_wind.long(),
        min=-cfg.v_max,
        max=cfg.v_max,
    )
    position_next = position + velocity_next

    hit_left = position_next < 0
    position_next = torch.where(hit_left, -position_next, position_next)
    velocity_next = torch.where(hit_left, -velocity_next, velocity_next)

    hit_right = position_next >= cfg.n_pos
    position_next = torch.where(
        hit_right,
        2 * (cfg.n_pos - 1) - position_next,
        position_next,
    )
    velocity_next = torch.where(hit_right, -velocity_next, velocity_next)

    if not bool(((position_next >= 0) & (position_next < cfg.n_pos)).all()):
        raise RuntimeError("Reflection invariant failed: position escaped the grid.")
    return encode_state(position_next, velocity_next)


def random_states(batch: int) -> torch.Tensor:
    return torch.randint(0, cfg.n_states, (batch,), device=device)


def random_actions(batch: int, horizon: int) -> torch.Tensor:
    return torch.randint(0, cfg.n_actions, (batch, horizon), device=device)


def rollout_true(
    initial_state: torch.Tensor,
    actions: torch.Tensor,
    winds: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Return q_0 ... q_T with shape [batch, T+1]."""
    states = [initial_state]
    current = initial_state
    for t in range(actions.shape[1]):
        wind_t = None if winds is None else winds[:, t]
        current = true_step(current, actions[:, t], wind_t)
        states.append(current)
    return torch.stack(states, dim=1)


def render_observation(
    state: torch.Tensor,
    distractor_probability: Optional[float] = None,
) -> torch.Tensor:
    """
    Render three channels without CPU transfer:
      0. ball position,
      1. velocity gauge,
      2. iid distractor pixels unrelated to the mechanics.
    """
    if distractor_probability is None:
        distractor_probability = cfg.distractor_probability

    state = state.reshape(-1)
    batch = state.shape[0]
    size = cfg.image_size
    obs = torch.zeros((batch, 3, size, size), device=device, dtype=torch.float32)
    position, velocity = decode_state(state)

    # Map the discrete track to interior pixel columns.
    pixel_x = 1 + torch.round(
        position.float() * float(size - 3) / float(cfg.n_pos - 1)
    ).long()
    center_y = size // 2
    batch_index = torch.arange(batch, device=device)

    # A small 3-pixel ball makes the physical signal nontrivial but visible.
    obs[batch_index, 0, center_y, pixel_x] = 1.0
    obs[batch_index, 0, center_y - 1, pixel_x] = 0.75
    obs[batch_index, 0, center_y + 1, pixel_x] = 0.75

    # The velocity gauge makes (x,v) observable from one frame. It is a supplied
    # grounding interface, not a claim of unsupervised causal-state discovery.
    gauge_x = (size // 2 - cfg.v_max) + (velocity + cfg.v_max)
    obs[batch_index, 1, size - 2, gauge_x] = 1.0
    obs[batch_index, 1, size - 3, size // 2] = 0.35

    if distractor_probability > 0.0:
        mask = torch.rand((batch, size, size), device=device) < distractor_probability
        values = torch.rand((batch, size, size), device=device) * cfg.distractor_amplitude
        obs[:, 2] = mask.float() * values

    return obs.contiguous(memory_format=torch.channels_last)


# Exhaustive true table is evaluation-only. It is never passed to extraction.
with torch.no_grad():
    _all_q = torch.arange(cfg.n_states, device=device).repeat_interleave(cfg.n_actions)
    _all_a = torch.arange(cfg.n_actions, device=device).repeat(cfg.n_states)
    TRUE_TABLE = true_step(_all_q, _all_a).reshape(cfg.n_states, cfg.n_actions)


# =============================================================================
# 3. NEURAL RECURRENT WORLD MODEL
# =============================================================================

class NeuralWorldModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.SiLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.SiLU(inplace=True),
            nn.Conv2d(64, 96, 3, stride=2, padding=1),
            nn.SiLU(inplace=True),
        )
        encoded_side = math.ceil(cfg.image_size / 8)
        self.encoder_proj = nn.Sequential(
            nn.Flatten(),
            # Preserve the final 2-D layout. Global pooling would erase the very
            # position variable the mechanics requires.
            nn.Linear(96 * encoded_side * encoded_side, cfg.hidden_size),
            nn.Tanh(),
        )
        self.action_embedding = nn.Embedding(cfg.n_actions, cfg.action_embed_dim)
        self.gru = nn.GRU(cfg.action_embed_dim, cfg.hidden_size, batch_first=True)
        self.state_head = nn.Linear(cfg.hidden_size, cfg.n_states)

    def encode_hidden(self, observation: torch.Tensor) -> torch.Tensor:
        return self.encoder_proj(self.encoder_conv(observation))

    def initial_logits(self, observation: torch.Tensor) -> torch.Tensor:
        return self.state_head(self.encode_hidden(observation))

    def rollout_from_hidden(
        self,
        hidden: torch.Tensor,
        actions: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        embedded = self.action_embedding(actions)
        outputs, final_hidden = self.gru(embedded, hidden.unsqueeze(0))
        return self.state_head(outputs), final_hidden.squeeze(0)

    def forward(
        self,
        initial_observation: torch.Tensor,
        actions: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        hidden = self.encode_hidden(initial_observation)
        initial_logits = self.state_head(hidden)
        rollout_logits, _ = self.rollout_from_hidden(hidden, actions)
        return initial_logits, rollout_logits


model = NeuralWorldModel().to(device)
if is_cuda:
    model.encoder_conv = model.encoder_conv.to(memory_format=torch.channels_last)

optimizer_kwargs = dict(lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
try:
    optimizer = torch.optim.AdamW(model.parameters(), fused=is_cuda, **optimizer_kwargs)
except (TypeError, RuntimeError):
    optimizer = torch.optim.AdamW(model.parameters(), **optimizer_kwargs)

parameter_count = sum(p.numel() for p in model.parameters())
print(f"world-model parameters={parameter_count:,}")


# =============================================================================
# 4. TRAINING
# =============================================================================

training_history: List[Dict[str, float]] = []
model.train()
train_start = time.perf_counter()

for step in range(1, cfg.train_steps + 1):
    q0 = random_states(cfg.train_batch_size)
    action_seq = random_actions(cfg.train_batch_size, cfg.train_seq_len)
    target_states = rollout_true(q0, action_seq)
    obs0 = render_observation(q0)

    optimizer.zero_grad(set_to_none=True)
    with amp_context():
        initial_logits, future_logits = model(obs0, action_seq)
        loss_initial = F.cross_entropy(initial_logits.float(), target_states[:, 0])
        loss_future = F.cross_entropy(
            future_logits.float().reshape(-1, cfg.n_states),
            target_states[:, 1:].reshape(-1),
        )
        loss = loss_future + cfg.initial_state_loss_weight * loss_initial

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
    scaler.step(optimizer)
    scaler.update()

    if step == 1 or step % max(1, cfg.train_steps // 40) == 0 or step == cfg.train_steps:
        with torch.no_grad():
            initial_acc = (initial_logits.argmax(-1) == target_states[:, 0]).float().mean().item()
            final_acc = (future_logits[:, -1].argmax(-1) == target_states[:, -1]).float().mean().item()
        row = {
            "step": float(step),
            "loss": float(loss.item()),
            "initial_accuracy": float(initial_acc),
            "train_horizon_accuracy": float(final_acc),
        }
        training_history.append(row)
        print(
            f"train {step:4d}/{cfg.train_steps} | loss={row['loss']:.4f} | "
            f"q0={100*initial_acc:5.1f}% | t{cfg.train_seq_len}={100*final_acc:5.1f}%"
        )

train_seconds = time.perf_counter() - train_start
model.eval()
print(f"training elapsed={train_seconds:.1f}s")


# =============================================================================
# 5. QUERY AND CRYSTALLIZE A DETERMINISTIC FINITE TRANSITION SYSTEM
# =============================================================================

@torch.inference_mode()
def extract_machine() -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    For each supplied abstract state and action, query the *neural model* across
    nuisance-rendered observations. Majority/mean-probability crystallization
    produces one deterministic successor. No true transition is read here.
    """
    transitions = torch.empty((cfg.n_states, cfg.n_actions), dtype=torch.long, device=device)
    confidence = torch.empty_like(transitions, dtype=torch.float32)
    stability = torch.empty_like(transitions, dtype=torch.float32)

    for q in range(cfg.n_states):
        q_batch = torch.full(
            (cfg.extraction_nuisance_samples,), q, dtype=torch.long, device=device
        )
        observations = render_observation(q_batch)
        hidden = model.encode_hidden(observations)
        for action in range(cfg.n_actions):
            actions = torch.full(
                (cfg.extraction_nuisance_samples, 1),
                action,
                dtype=torch.long,
                device=device,
            )
            logits, _ = model.rollout_from_hidden(hidden, actions)
            probabilities = logits[:, 0].float().softmax(-1)
            mean_probability = probabilities.mean(0)
            successor = mean_probability.argmax()
            transitions[q, action] = successor
            confidence[q, action] = mean_probability[successor]
            stability[q, action] = (
                probabilities.argmax(-1) == successor
            ).float().mean()
    return transitions, confidence, stability


MACHINE_TABLE, MACHINE_CONFIDENCE, MACHINE_STABILITY = extract_machine()
machine_transition_accuracy = (MACHINE_TABLE == TRUE_TABLE).float().mean().item()
machine_min_confidence = MACHINE_CONFIDENCE.min().item()
machine_mean_confidence = MACHINE_CONFIDENCE.mean().item()
machine_min_stability = MACHINE_STABILITY.min().item()

print("\nCRYSTALLIZATION AUDIT")
print("-" * 88)
print(f"finite states={cfg.n_states} | actions={cfg.n_actions} | transitions={cfg.n_states*cfg.n_actions}")
print(f"transition accuracy against withheld simulator={100*machine_transition_accuracy:.2f}%")
print(f"mean/min neural confidence={machine_mean_confidence:.4f}/{machine_min_confidence:.4f}")
print(f"minimum nuisance stability={machine_min_stability:.4f}")


# =============================================================================
# 6. LONG-HORIZON ROLLOUT: RECURRENT MODEL VS CRYSTALLIZED MACHINE
# =============================================================================

@torch.inference_mode()
def rollout_machine(initial_state: torch.Tensor, actions: torch.Tensor) -> torch.Tensor:
    states = []
    current = initial_state
    for t in range(actions.shape[1]):
        current = MACHINE_TABLE[current, actions[:, t]]
        states.append(current)
    return torch.stack(states, dim=1)


@torch.inference_mode()
def evaluate_rollouts() -> List[Dict[str, float]]:
    max_horizon = max(cfg.rollout_horizons)
    q0 = random_states(cfg.rollout_batch_size)
    actions = random_actions(cfg.rollout_batch_size, max_horizon)
    truth = rollout_true(q0, actions)
    obs0 = render_observation(q0)

    with amp_context():
        initial_logits, neural_logits = model(obs0, actions)
    neural_pred = neural_logits.float().argmax(-1)
    q0_hat = initial_logits.float().argmax(-1)
    machine_pred = rollout_machine(q0_hat, actions)
    perception_correct = q0_hat == q0

    rows: List[Dict[str, float]] = []
    for horizon in cfg.rollout_horizons:
        target = truth[:, horizon]
        neural_q = neural_pred[:, horizon - 1]
        machine_q = machine_pred[:, horizon - 1]
        target_p, _ = decode_state(target)
        neural_p, _ = decode_state(neural_q)
        machine_p, _ = decode_state(machine_q)

        if bool(perception_correct.any()):
            neural_cond = (neural_q[perception_correct] == target[perception_correct]).float().mean()
            machine_cond = (machine_q[perception_correct] == target[perception_correct]).float().mean()
        else:
            neural_cond = torch.tensor(float("nan"), device=device)
            machine_cond = torch.tensor(float("nan"), device=device)

        rows.append(
            {
                "horizon": float(horizon),
                "neural_state_accuracy": float((neural_q == target).float().mean().item()),
                "machine_state_accuracy": float((machine_q == target).float().mean().item()),
                "neural_state_accuracy_given_correct_q0": float(neural_cond.item()),
                "machine_state_accuracy_given_correct_q0": float(machine_cond.item()),
                "neural_position_mae": float((neural_p - target_p).abs().float().mean().item()),
                "machine_position_mae": float((machine_p - target_p).abs().float().mean().item()),
            }
        )
    return rows


rollout_rows = evaluate_rollouts()
print("\nLONG-HORIZON ROLLOUT")
print("-" * 88)
print(" horizon | neural state acc | machine state acc | neural pos MAE | machine pos MAE")
for row in rollout_rows:
    print(
        f" {int(row['horizon']):7d} |"
        f" {100*row['neural_state_accuracy']:15.2f}% |"
        f" {100*row['machine_state_accuracy']:16.2f}% |"
        f" {row['neural_position_mae']:14.3f} |"
        f" {row['machine_position_mae']:15.3f}"
    )

crossover_horizon: Optional[int] = None
for row in rollout_rows:
    if row["machine_state_accuracy"] > row["neural_state_accuracy"] + 1e-9:
        crossover_horizon = int(row["horizon"])
        break


# =============================================================================
# 7. PLANNING: MACHINE BFS VS BATCHED NEURAL BEAM SEARCH
# =============================================================================

def machine_fixed_horizon_plan(
    start_state: int,
    target_state: int,
    horizon: int,
) -> Optional[List[int]]:
    """
    Exact graph search in the time-expanded crystallized machine. Histories that
    reach the same finite state at the same depth are merged, so the search is
    O(horizon * |Q| * |A|), not exponential in horizon.
    """
    frontier: Dict[int, List[int]] = {start_state: []}
    for _ in range(horizon):
        next_frontier: Dict[int, List[int]] = {}
        for q, prefix in frontier.items():
            for action in range(cfg.n_actions):
                q_next = int(MACHINE_TABLE[q, action].item())
                if q_next not in next_frontier:
                    next_frontier[q_next] = prefix + [action]
        frontier = next_frontier
    return frontier.get(target_state)


@torch.inference_mode()
def neural_beam_plan(
    start_state: int,
    target_state: int,
    horizon: int,
) -> Optional[List[int]]:
    """
    GPU-batched beam search over recurrent hidden states. This is intentionally
    not called MCTS: it is the T4-efficient neural-rollout planner in this cell.
    """
    q_tensor = torch.tensor([start_state], device=device)
    target_q_tensor = torch.tensor([target_state], device=device)
    target_position, target_velocity = decode_state(target_q_tensor)
    obs = render_observation(q_tensor)
    with amp_context():
        hidden = model.encode_hidden(obs)

    sequences = torch.empty((1, 0), dtype=torch.long, device=device)
    cumulative_log_confidence = torch.zeros(1, device=device)

    for depth in range(1, horizon + 1):
        beam_count = hidden.shape[0]
        expanded_hidden = hidden.repeat_interleave(cfg.n_actions, dim=0)
        expanded_actions = torch.arange(cfg.n_actions, device=device).repeat(beam_count)

        with amp_context():
            logits, next_hidden = model.rollout_from_hidden(
                expanded_hidden,
                expanded_actions[:, None],
            )
        probabilities = logits[:, 0].float().softmax(-1)
        predicted_q = probabilities.argmax(-1)
        confidence = probabilities.gather(1, predicted_q[:, None]).squeeze(1).clamp_min(1e-8)
        predicted_position, _ = decode_state(predicted_q)

        expanded_sequences = torch.cat(
            [
                sequences.repeat_interleave(cfg.n_actions, dim=0),
                expanded_actions[:, None],
            ],
            dim=1,
        )
        expanded_log_confidence = (
            cumulative_log_confidence.repeat_interleave(cfg.n_actions)
            + confidence.log()
        )

        reaches_target = predicted_q == target_state
        if depth == horizon and bool(reaches_target.any()):
            candidate_ids = torch.nonzero(reaches_target, as_tuple=False).squeeze(1)
            best = candidate_ids[expanded_log_confidence[candidate_ids].argmax()]
            return expanded_sequences[best].detach().cpu().tolist()

        if depth == horizon:
            return None

        # Distance drives progress; confidence breaks ties and suppresses paths
        # that the neural dynamics itself considers implausible.
        _, predicted_velocity = decode_state(predicted_q)
        distance = (
            (predicted_position - target_position).abs().float()
            + 0.5 * (predicted_velocity - target_velocity).abs().float()
        )
        score = -distance + 0.03 * expanded_log_confidence
        keep = min(cfg.neural_beam_width, score.numel())
        selected = torch.topk(score, k=keep, largest=True).indices
        hidden = next_hidden[selected]
        sequences = expanded_sequences[selected]
        cumulative_log_confidence = expanded_log_confidence[selected]

    return None


@torch.inference_mode()
def execute_plan(start_state: int, actions: Optional[Sequence[int]], target_state: int) -> bool:
    if actions is None:
        return False
    q = torch.tensor([start_state], dtype=torch.long, device=device)
    for action in actions:
        a = torch.tensor([action], dtype=torch.long, device=device)
        q = true_step(q, a)
    return int(q.item()) == target_state


planning_rng = np.random.default_rng(cfg.seed + 7)
machine_successes: List[float] = []
neural_successes: List[float] = []
machine_lengths: List[float] = []
neural_lengths: List[float] = []
planning_horizons: List[float] = []

planning_start = time.perf_counter()
for _ in range(cfg.planning_cases):
    start_state = int(planning_rng.integers(0, cfg.n_states))
    minimum_horizon = max(4, cfg.planning_max_depth // 2)
    planning_horizon = int(
        planning_rng.integers(minimum_horizon, cfg.planning_max_depth + 1)
    )
    # Generate a guaranteed-reachable exact terminal state, then discard the
    # generating action sequence. Both planners must independently recover one.
    witness_actions = torch.tensor(
        planning_rng.integers(0, cfg.n_actions, size=(1, planning_horizon)),
        dtype=torch.long,
        device=device,
    )
    target_state = int(
        rollout_true(
            torch.tensor([start_state], dtype=torch.long, device=device),
            witness_actions,
        )[0, -1].item()
    )

    machine_plan = machine_fixed_horizon_plan(
        start_state, target_state, planning_horizon
    )
    neural_plan = neural_beam_plan(
        start_state, target_state, planning_horizon
    )

    machine_successes.append(float(execute_plan(start_state, machine_plan, target_state)))
    neural_successes.append(float(execute_plan(start_state, neural_plan, target_state)))
    planning_horizons.append(float(planning_horizon))
    if machine_plan is not None:
        machine_lengths.append(float(len(machine_plan)))
    if neural_plan is not None:
        neural_lengths.append(float(len(neural_plan)))

planning_seconds = time.perf_counter() - planning_start
planning_results = {
    "cases": cfg.planning_cases,
    "machine_success_rate": float(np.mean(machine_successes)),
    "neural_beam_success_rate": float(np.mean(neural_successes)),
    "machine_median_plan_length": float(np.median(machine_lengths)) if machine_lengths else None,
    "neural_beam_median_plan_length": float(np.median(neural_lengths)) if neural_lengths else None,
    "median_required_horizon": float(np.median(planning_horizons)),
    "elapsed_seconds": planning_seconds,
}

print("\nEXACT-HORIZON OPEN-LOOP PLANNING")
print("-" * 88)
print(
    f"machine graph-search success={100*planning_results['machine_success_rate']:.2f}% | "
    f"median length={planning_results['machine_median_plan_length']}"
)
print(
    f"neural beam success={100*planning_results['neural_beam_success_rate']:.2f}% | "
    f"median length={planning_results['neural_beam_median_plan_length']}"
)
print(f"median exact-arrival horizon={planning_results['median_required_horizon']}")


# =============================================================================
# 8. ONLINE VALIDITY MONITORING UNDER A HIDDEN DYNAMICS CHANGE
# =============================================================================

def binary_auroc(labels: np.ndarray, scores: np.ndarray) -> float:
    labels = labels.astype(np.int64)
    positives = int(labels.sum())
    negatives = int((1 - labels).sum())
    if positives == 0 or negatives == 0:
        return float("nan")
    order = np.argsort(-scores, kind="mergesort")
    y = labels[order]
    tpr = np.r_[0.0, np.cumsum(y) / positives, 1.0]
    fpr = np.r_[0.0, np.cumsum(1 - y) / negatives, 1.0]
    if hasattr(np, "trapezoid"):
        return float(np.trapezoid(tpr, fpr))
    return float(np.trapz(tpr, fpr))


def binary_auprc(labels: np.ndarray, scores: np.ndarray) -> float:
    labels = labels.astype(np.int64)
    positives = int(labels.sum())
    if positives == 0:
        return float("nan")
    order = np.argsort(-scores, kind="mergesort")
    y = labels[order]
    tp = np.cumsum(y)
    fp = np.cumsum(1 - y)
    recall = tp / positives
    precision = tp / np.maximum(tp + fp, 1)
    recall = np.r_[0.0, recall]
    precision = np.r_[1.0, precision]
    return float(np.sum((recall[1:] - recall[:-1]) * precision[1:]))


@torch.inference_mode()
def encode_state_probabilities(observations: torch.Tensor, chunk: int) -> torch.Tensor:
    outputs = []
    for start in range(0, observations.shape[0], chunk):
        with amp_context():
            logits = model.initial_logits(observations[start : start + chunk])
        outputs.append(logits.float().softmax(-1))
    return torch.cat(outputs, dim=0)


@torch.inference_mode()
def evaluate_validity() -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    batch = cfg.validity_batch_size
    horizon = cfg.validity_horizon
    q0 = random_states(batch)
    actions = random_actions(batch, horizon)

    nominal_count = int(round(batch * cfg.validity_nominal_fraction))
    has_exit = torch.zeros(batch, dtype=torch.bool, device=device)
    has_exit[nominal_count:] = True
    # Shuffle which sequences contain exits.
    has_exit = has_exit[torch.randperm(batch, device=device)]

    min_event = min(3, max(1, horizon // 3))
    max_event_exclusive = max(min_event + 1, horizon - 2)
    event_time = torch.randint(min_event, max_event_exclusive, (batch,), device=device)
    wind_direction = torch.where(
        torch.rand(batch, device=device) < 0.5,
        -torch.ones(batch, dtype=torch.long, device=device),
        torch.ones(batch, dtype=torch.long, device=device),
    )

    times = torch.arange(horizon, device=device)[None, :]
    active = has_exit[:, None] & (times >= event_time[:, None])
    winds = active.long() * wind_direction[:, None]
    actual = rollout_true(q0, actions, winds)

    # Counterfactual nominal successor from each actual current state. This asks
    # whether the hidden wind changes the next outcome on this particular step.
    nominal_next = true_step(
        actual[:, :-1].reshape(-1),
        actions.reshape(-1),
    ).reshape(batch, horizon)
    counterexample = actual[:, 1:] != nominal_next

    current_obs = render_observation(actual[:, :-1].reshape(-1))
    next_obs = render_observation(actual[:, 1:].reshape(-1))
    current_probs = encode_state_probabilities(current_obs, cfg.validity_encode_chunk)
    next_probs = encode_state_probabilities(next_obs, cfg.validity_encode_chunk)
    current_hat = current_probs.argmax(-1)
    predicted_machine_next = MACHINE_TABLE[current_hat, actions.reshape(-1)]

    # Surprise is bounded [0,1] and calibrated entirely on nominal steps.
    assigned_probability = next_probs.gather(1, predicted_machine_next[:, None]).squeeze(1)
    surprise = (1.0 - assigned_probability).reshape(batch, horizon)

    nominal_scores = surprise[~has_exit].detach().cpu().numpy().reshape(-1)
    threshold = float(np.quantile(nominal_scores, cfg.validity_threshold_quantile))
    alarms = surprise > threshold

    domain_labels = active.detach().cpu().numpy().reshape(-1).astype(np.int64)
    counterexample_labels = counterexample.detach().cpu().numpy().reshape(-1).astype(np.int64)
    score_np = surprise.detach().cpu().numpy().reshape(-1)

    # Event-level latency from the dynamics switch, even if clipping temporarily
    # makes the altered and nominal transition coincide.
    latencies: List[float] = []
    detected_within_1 = 0
    detected_within_3 = 0
    exit_indices = torch.nonzero(has_exit, as_tuple=False).squeeze(1)
    for i in exit_indices.tolist():
        t0 = int(event_time[i].item())
        post_alarm = torch.nonzero(alarms[i, t0:], as_tuple=False).squeeze(1)
        if post_alarm.numel() > 0:
            latency = int(post_alarm[0].item())
            latencies.append(float(latency))
            detected_within_1 += int(latency <= 1)
            detected_within_3 += int(latency <= 3)

    exit_n = max(1, int(has_exit.sum().item()))
    results = {
        "threshold": threshold,
        "nominal_false_positive_rate": float(alarms[~has_exit].float().mean().item()),
        "domain_exit_auroc": binary_auroc(domain_labels, score_np),
        "domain_exit_auprc": binary_auprc(domain_labels, score_np),
        "counterexample_auroc": binary_auroc(counterexample_labels, score_np),
        "counterexample_auprc": binary_auprc(counterexample_labels, score_np),
        "event_detection_rate": float(len(latencies) / exit_n),
        "detected_within_1_step": float(detected_within_1 / exit_n),
        "detected_within_3_steps": float(detected_within_3 / exit_n),
        "median_detection_latency": float(np.median(latencies)) if latencies else None,
        "positive_step_prevalence": float(domain_labels.mean()),
        "counterexample_step_prevalence": float(counterexample_labels.mean()),
    }
    return results, score_np, domain_labels


validity_results, validity_scores, validity_labels = evaluate_validity()
print("\nVALIDITY MONITOR")
print("-" * 88)
print(
    f"domain-exit AUROC={validity_results['domain_exit_auroc']:.3f} | "
    f"AUPRC={validity_results['domain_exit_auprc']:.3f} | "
    f"nominal FPR={100*validity_results['nominal_false_positive_rate']:.2f}%"
)
print(
    f"counterexample AUROC={validity_results['counterexample_auroc']:.3f} | "
    f"AUPRC={validity_results['counterexample_auprc']:.3f}"
)
print(
    f"event detection={100*validity_results['event_detection_rate']:.2f}% | "
    f"within 1 step={100*validity_results['detected_within_1_step']:.2f}% | "
    f"median latency={validity_results['median_detection_latency']}"
)


# =============================================================================
# 9. SAVE ARTIFACTS AND SUMMARY FIGURE
# =============================================================================

config_json = asdict(cfg)
config_json["rollout_horizons"] = list(cfg.rollout_horizons)

results = {
    "interpretation_boundary": (
        "The abstract (position, velocity) state labels and action alphabet were supplied. "
        "This tests neural-to-finite crystallization, long-horizon dynamics, planning, and "
        "validity monitoring; it does not test autonomous state/coarse-graining discovery."
    ),
    "runtime": {
        "device": str(device),
        "gpu": torch.cuda.get_device_name(0) if is_cuda else None,
        "torch_version": torch.__version__,
        "amp": amp_enabled,
        "parameter_count": parameter_count,
        "train_seconds": train_seconds,
    },
    "crystallization": {
        "n_states": cfg.n_states,
        "n_actions": cfg.n_actions,
        "transition_accuracy": machine_transition_accuracy,
        "mean_confidence": machine_mean_confidence,
        "min_confidence": machine_min_confidence,
        "min_nuisance_stability": machine_min_stability,
    },
    "rollout": {
        "rows": rollout_rows,
        "first_strict_machine_crossover_horizon": crossover_horizon,
    },
    "planning": planning_results,
    "validity": validity_results,
}

with (out_dir / "config.json").open("w", encoding="utf-8") as f:
    json.dump(config_json, f, indent=2)
with (out_dir / "results.json").open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

with (out_dir / "training_history.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(training_history[0].keys()))
    writer.writeheader()
    writer.writerows(training_history)

with (out_dir / "rollout_metrics.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rollout_rows[0].keys()))
    writer.writeheader()
    writer.writerows(rollout_rows)

np.savez_compressed(
    out_dir / "crystallized_machine.npz",
    transition_table=MACHINE_TABLE.detach().cpu().numpy(),
    query_confidence=MACHINE_CONFIDENCE.detach().cpu().numpy(),
    nuisance_stability=MACHINE_STABILITY.detach().cpu().numpy(),
)
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": config_json,
    },
    out_dir / "world_model.pt",
)

fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)

axes[0, 0].plot(
    [r["step"] for r in training_history],
    [r["loss"] for r in training_history],
    color="#4C78A8",
    linewidth=2,
)
axes[0, 0].set_title("Neural world-model training")
axes[0, 0].set_xlabel("optimizer step")
axes[0, 0].set_ylabel("loss")
axes[0, 0].grid(alpha=0.25)

h = np.array([r["horizon"] for r in rollout_rows])
axes[0, 1].plot(
    h,
    [r["neural_state_accuracy"] for r in rollout_rows],
    "o-",
    label="neural recurrent rollout",
    color="#E45756",
)
axes[0, 1].plot(
    h,
    [r["machine_state_accuracy"] for r in rollout_rows],
    "o-",
    label="crystallized machine",
    color="#54A24B",
)
axes[0, 1].set_xscale("log", base=2)
axes[0, 1].set_ylim(-0.03, 1.03)
axes[0, 1].set_title("Exact state accuracy vs rollout depth")
axes[0, 1].set_xlabel("horizon")
axes[0, 1].set_ylabel("accuracy")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.25)

axes[1, 0].plot(
    h,
    [r["neural_position_mae"] for r in rollout_rows],
    "o-",
    label="neural recurrent rollout",
    color="#E45756",
)
axes[1, 0].plot(
    h,
    [r["machine_position_mae"] for r in rollout_rows],
    "o-",
    label="crystallized machine",
    color="#54A24B",
)
axes[1, 0].set_xscale("log", base=2)
axes[1, 0].set_title("Position error vs rollout depth")
axes[1, 0].set_xlabel("horizon")
axes[1, 0].set_ylabel("mean absolute position error")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.25)

nominal_scores = validity_scores[validity_labels == 0]
exit_scores = validity_scores[validity_labels == 1]
bins = np.linspace(0.0, 1.0, 45)
axes[1, 1].hist(nominal_scores, bins=bins, density=True, alpha=0.65, label="nominal")
axes[1, 1].hist(exit_scores, bins=bins, density=True, alpha=0.65, label="domain exit")
axes[1, 1].axvline(validity_results["threshold"], color="black", linestyle="--", label="calibrated threshold")
axes[1, 1].set_title("Machine/observation disagreement")
axes[1, 1].set_xlabel("surprise = 1 - P(observed successor)")
axes[1, 1].set_ylabel("density")
axes[1, 1].legend()

fig.suptitle(
    "Crystallized finite dynamics inside a neural world model\n"
    f"transition accuracy={100*machine_transition_accuracy:.1f}% | "
    f"planning: machine={100*planning_results['machine_success_rate']:.1f}%, "
    f"neural={100*planning_results['neural_beam_success_rate']:.1f}%",
    fontsize=14,
)
fig.savefig(out_dir / "summary.png", dpi=180)
plt.show()

archive_path = out_dir / "run_bundle.zip"
if archive_path.exists():
    archive_path.unlink()
# Build outside the directory being archived; otherwise the ZIP can include its
# own growing byte stream on some shutil/zipfile versions.
temporary_archive_base = out_dir.parent / f".{out_dir.name}_run_bundle_tmp"
temporary_archive_path = temporary_archive_base.with_suffix(".zip")
if temporary_archive_path.exists():
    temporary_archive_path.unlink()
made_archive = shutil.make_archive(
    str(temporary_archive_base),
    "zip",
    root_dir=out_dir,
    base_dir=".",
)
shutil.move(made_archive, archive_path)

print("\n" + "=" * 88)
print("FINAL SUMMARY")
print("=" * 88)
print(f"machine transition accuracy: {100*machine_transition_accuracy:.2f}%")
print(f"first strict machine rollout crossover: {crossover_horizon}")
print(
    f"planning success — machine: {100*planning_results['machine_success_rate']:.2f}% | "
    f"neural beam: {100*planning_results['neural_beam_success_rate']:.2f}%"
)
print(
    f"validity AUROC: {validity_results['domain_exit_auroc']:.3f} | "
    f"counterexample AUROC: {validity_results['counterexample_auroc']:.3f}"
)
print(f"artifacts: {out_dir}")
print(f"bundle: {archive_path}")

if machine_transition_accuracy < 0.95:
    print(
        "\nAUDIT WARNING: the extracted one-step machine is below 95% transition accuracy. "
        "Do not interpret long-horizon differences as drift removal alone; rerun with more "
        "training or report this as a failed crystallization condition."
    )


## Experiment 3: Prospective validity V2

Original cell `2`.


In [ ]:
"""
PROSPECTIVE VALIDITY V2 — T4-OPTIMIZED, SINGLE-CELL COLAB EXPERIMENT

Paste this entire file into one Google Colab cell and run it with a T4 GPU.

FROZEN QUESTION
---------------
Can a small neural monitor predict whether a proposed action sequence will leave
the validity domain of a crystallized finite world model *before* the first
machine/world disagreement, and can that forecast route planning toward a safer
machine-equivalent plan?

ENVIRONMENT AND INTERPRETATION BOUNDARY
---------------------------------------
The finite state q=(quantized position, velocity) and action alphabet are
supplied. A nominal recurrent world model is trained with visible context stripes
that do not affect its nominal dynamics, then queried into a deterministic finite
transition table. In V2, one stripe becomes a wind zone: it is visible in the raw
observation but deliberately omitted from q. Entering it invalidates the nominal
machine; wind affects subsequent dynamics.

This tests a learned boundary around a supplied abstraction. It does not test
autonomous discovery of the abstraction. Exact machine planning is guaranteed
only conditional on validity; the prospective monitor supplies statistical risk,
not a formal proof that validity will persist.

PREDECLARED CLAIMS
------------------
P1. On held-out wind-zone locations, observation+action monitoring outperforms
    observation-only and action-only monitoring for exit-within-k prediction.
P2. For paired plans from the same observation, the joint monitor ranks the plan
    that enters the zone as riskier than the avoiding plan.
P3. At a validation-calibrated 5% safe-plan false-positive target, prospective
    alerts have negative latency relative to domain exit, whereas postdictive
    disagreement cannot alert before the first manifest counterexample.
P4. When two plans reach the same target state at the same horizon in the nominal
    machine, validity-aware selection chooses the avoiding plan more often and
    reduces domain exits versus a machine-only random tie-break.
P5. For an unobservable random switch whose time is independent of observation
    and action, the same prospective architecture remains at chance AUROC. This
    is the information-theoretic negative control.

PRIMARY TIMESTAMPS
------------------
tau_exit: first state time at which the nominal trajectory enters the visible
          wind zone, so the nominal machine's preconditions no longer hold.
tau_cex:  first later state time at which the true and nominal states disagree.

AUDIT GATES
-----------
1. The crystallized nominal transition table must be exact before primary V2
   interpretation (relaxed only in FAST_DEV_RUN).
2. Train/validation/test paired-plan corpora use independent seeds.
3. Wind-zone starts {2,7} are held out from monitor training; start 5 is used
   only for validation thresholding.
4. Every paired sample shares observation/context and differs only in actions:
   one plan exits by k=12 and one does not.
5. Planning challenge pairs reach the same nominal target at the same horizon.
6. The alert threshold is chosen from validation safe plans only.
7. The hidden-switch label is sampled independently of all model inputs.

SAVED OUTPUTS
-------------
config.json, results.json, horizon_metrics.csv, planning_cases.csv,
crystallized_machine.npz, model checkpoints, summary.png, and run_bundle.zip.

Set environment variable PROSPECTIVE_V2_FAST_DEV_RUN=1 for a reduced path test.
"""

from __future__ import annotations

import contextlib
import csv
import dataclasses
import json
import math
import os
import random
import shutil
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# =============================================================================
# 0. FROZEN CONFIGURATION
# =============================================================================

FAST_DEV_RUN = os.environ.get("PROSPECTIVE_V2_FAST_DEV_RUN", "0") == "1"


@dataclass(frozen=True)
class Config:
    seed: int = 2601

    # Supplied finite mechanics.
    n_pos: int = 13
    v_max: int = 2
    n_actions: int = 3
    image_size: int = 16
    zone_width: int = 3
    distractor_probability: float = 0.08

    # Held-out spatial split for the prospective boundary monitor.
    train_zone_starts: Tuple[int, ...] = (0, 1, 3, 4, 6, 8, 9, 10)
    validation_zone_starts: Tuple[int, ...] = (5,)
    test_zone_starts: Tuple[int, ...] = (2, 7)

    # Nominal neural world model and crystallization.
    world_hidden_size: int = 128
    world_action_embed_dim: int = 24
    world_train_seq_len: int = 14
    world_train_steps: int = 700
    world_batch_size: int = 768
    world_learning_rate: float = 2.0e-3
    world_initial_loss_weight: float = 0.35
    extraction_nuisance_samples: int = 96

    # Paired-plan corpus. Each pair shares q/context and contains one risky and
    # one safe action sequence.
    max_prediction_horizon: int = 12
    reported_horizons: Tuple[int, ...] = (1, 3, 5, 8, 12)
    candidate_plans_per_context: int = 48
    monitor_train_pairs: int = 20_000
    monitor_validation_pairs: int = 4_000
    monitor_test_pairs: int = 6_000

    # Prospective monitors.
    monitor_obs_dim: int = 96
    monitor_action_hidden: int = 64
    monitor_action_embed: int = 20
    monitor_batch_size: int = 768
    joint_train_steps: int = 1_000
    ablation_train_steps: int = 700
    hidden_control_train_steps: int = 700
    monitor_learning_rate: float = 2.0e-3
    safe_plan_fpr_target: float = 0.05
    evaluation_nuisance_averages: int = 3

    # Information-theoretic hidden-switch negative control.
    hidden_train_samples: int = 40_000
    hidden_test_samples: int = 12_000

    # Counterfactual validity-aware planning challenge.
    planning_cases: int = 400
    planning_min_horizon: int = 8
    planning_max_horizon: int = 12

    # Runtime.
    use_amp: bool = True
    allow_tf32: bool = True
    output_dir: str = "outputs/03-prospective-validity-v2/prospective_validity_v2_results"

    @property
    def n_vel(self) -> int:
        return 2 * self.v_max + 1

    @property
    def n_states(self) -> int:
        return self.n_pos * self.n_vel


cfg = Config()
if FAST_DEV_RUN:
    cfg = dataclasses.replace(
        cfg,
        world_hidden_size=48,
        world_train_seq_len=5,
        world_train_steps=int(os.environ.get("PROSPECTIVE_V2_FAST_WORLD_STEPS", "12")),
        world_batch_size=48,
        extraction_nuisance_samples=4,
        candidate_plans_per_context=16,
        monitor_train_pairs=96,
        monitor_validation_pairs=48,
        monitor_test_pairs=64,
        monitor_obs_dim=40,
        monitor_action_hidden=32,
        monitor_batch_size=32,
        joint_train_steps=int(os.environ.get("PROSPECTIVE_V2_FAST_MONITOR_STEPS", "10")),
        ablation_train_steps=int(os.environ.get("PROSPECTIVE_V2_FAST_MONITOR_STEPS", "10")),
        hidden_control_train_steps=int(os.environ.get("PROSPECTIVE_V2_FAST_MONITOR_STEPS", "10")),
        hidden_train_samples=192,
        hidden_test_samples=96,
        planning_cases=24,
        evaluation_nuisance_averages=1,
        output_dir=str(Path.cwd() / "prospective_validity_v2_smoke"),
    )


# =============================================================================
# 1. RUNTIME AND REPRODUCIBILITY
# =============================================================================

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
is_cuda = device.type == "cuda"
if is_cuda:
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = cfg.allow_tf32
    torch.backends.cudnn.allow_tf32 = cfg.allow_tf32
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

amp_enabled = bool(cfg.use_amp and is_cuda)
amp_context = (
    (lambda: torch.autocast(device_type="cuda", dtype=torch.float16))
    if amp_enabled
    else contextlib.nullcontext
)


def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=amp_enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=amp_enabled)


out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

print("=" * 96)
print("PROSPECTIVE VALIDITY V2 — action-conditional boundary prediction")
print("=" * 96)
print(f"device={device} | torch={torch.__version__} | AMP={amp_enabled} | fast_dev={FAST_DEV_RUN}")
if is_cuda:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU={props.name} | VRAM={props.total_memory / 2**30:.1f} GiB")
else:
    print("WARNING: full defaults are designed for a Colab T4.")


# =============================================================================
# 2. NOMINAL AND HAZARD DYNAMICS
# =============================================================================

ACTION_VALUES = torch.tensor([-1, 0, 1], dtype=torch.long, device=device)
ALL_ZONE_STARTS = tuple(range(cfg.n_pos - cfg.zone_width + 1))


def encode_state(position: torch.Tensor, velocity: torch.Tensor) -> torch.Tensor:
    return position.long() * cfg.n_vel + (velocity.long() + cfg.v_max)


def decode_state(state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    position = torch.div(state.long(), cfg.n_vel, rounding_mode="floor")
    velocity = torch.remainder(state.long(), cfg.n_vel) - cfg.v_max
    return position, velocity


def nominal_step(
    state: torch.Tensor,
    action_index: torch.Tensor,
    exogenous_wind: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    position, velocity = decode_state(state)
    acceleration = ACTION_VALUES[action_index.long()]
    if exogenous_wind is None:
        exogenous_wind = torch.zeros_like(acceleration)
    velocity_next = torch.clamp(
        velocity + acceleration + exogenous_wind.long(),
        -cfg.v_max,
        cfg.v_max,
    )
    position_next = position + velocity_next

    hit_left = position_next < 0
    position_next = torch.where(hit_left, -position_next, position_next)
    velocity_next = torch.where(hit_left, -velocity_next, velocity_next)
    hit_right = position_next >= cfg.n_pos
    position_next = torch.where(
        hit_right,
        2 * (cfg.n_pos - 1) - position_next,
        position_next,
    )
    velocity_next = torch.where(hit_right, -velocity_next, velocity_next)
    return encode_state(position_next, velocity_next)


def in_zone(state: torch.Tensor, zone_start: torch.Tensor) -> torch.Tensor:
    position, _ = decode_state(state)
    return (position >= zone_start) & (position < zone_start + cfg.zone_width)


def hazard_step(
    state: torch.Tensor,
    action_index: torch.Tensor,
    zone_start: torch.Tensor,
    wind_direction: torch.Tensor,
) -> torch.Tensor:
    active_wind = in_zone(state, zone_start).long() * wind_direction.long()
    return nominal_step(state, action_index, active_wind)


def rollout_nominal(initial_state: torch.Tensor, actions: torch.Tensor) -> torch.Tensor:
    states = [initial_state]
    current = initial_state
    for t in range(actions.shape[1]):
        current = nominal_step(current, actions[:, t])
        states.append(current)
    return torch.stack(states, dim=1)


def rollout_hazard(
    initial_state: torch.Tensor,
    actions: torch.Tensor,
    zone_start: torch.Tensor,
    wind_direction: torch.Tensor,
) -> torch.Tensor:
    states = [initial_state]
    current = initial_state
    for t in range(actions.shape[1]):
        current = hazard_step(
            current,
            actions[:, t],
            zone_start,
            wind_direction,
        )
        states.append(current)
    return torch.stack(states, dim=1)


def sample_from(values: Sequence[int], count: int, generator: torch.Generator) -> torch.Tensor:
    value_tensor = torch.tensor(values, dtype=torch.long, device=device)
    indices = torch.randint(
        0,
        len(values),
        (count,),
        generator=generator,
        device=device,
    )
    return value_tensor[indices]


def sample_states_outside_zone(
    count: int,
    zone_start: torch.Tensor,
    generator: torch.Generator,
) -> torch.Tensor:
    q = torch.randint(0, cfg.n_states, (count,), generator=generator, device=device)
    inside = in_zone(q, zone_start)
    attempts = 0
    while bool(inside.any()):
        q[inside] = torch.randint(
            0,
            cfg.n_states,
            (int(inside.sum().item()),),
            generator=generator,
            device=device,
        )
        inside = in_zone(q, zone_start)
        attempts += 1
        if attempts > 50:
            raise RuntimeError("Could not sample states outside the wind zone.")
    return q


def state_to_pixel(position: torch.Tensor) -> torch.Tensor:
    return 1 + torch.round(
        position.float() * float(cfg.image_size - 3) / float(cfg.n_pos - 1)
    ).long()


def render_observation(
    state: torch.Tensor,
    zone_start: Optional[torch.Tensor] = None,
    wind_direction: Optional[torch.Tensor] = None,
    visible_context: bool = True,
    distractor_probability: Optional[float] = None,
) -> torch.Tensor:
    """Channels: ball, velocity gauge, visible context, irrelevant distractors."""
    if distractor_probability is None:
        distractor_probability = cfg.distractor_probability
    state = state.reshape(-1)
    batch = state.shape[0]
    size = cfg.image_size
    obs = torch.zeros((batch, 4, size, size), dtype=torch.float32, device=device)
    position, velocity = decode_state(state)
    px = state_to_pixel(position)
    bi = torch.arange(batch, device=device)
    cy = size // 2

    obs[bi, 0, cy, px] = 1.0
    obs[bi, 0, cy - 1, px] = 0.75
    obs[bi, 0, cy + 1, px] = 0.75
    gauge_x = (size // 2 - cfg.v_max) + (velocity + cfg.v_max)
    obs[bi, 1, size - 2, gauge_x] = 1.0
    obs[bi, 1, size - 3, size // 2] = 0.35

    if visible_context and zone_start is not None:
        if wind_direction is None:
            wind_direction = torch.ones(batch, dtype=torch.long, device=device)
        zone_rows = torch.arange(2, size - 4, device=device)
        for offset in range(cfg.zone_width):
            zone_position = zone_start + offset
            zone_px = state_to_pixel(zone_position)
            obs[bi[:, None], 2, zone_rows[None, :], zone_px[:, None]] = 0.55
        # Direction is visible but is not required merely to predict entry.
        marker_row = torch.where(wind_direction > 0, 1, size - 4)
        center_position = zone_start + cfg.zone_width // 2
        center_px = state_to_pixel(center_position)
        obs[bi, 2, marker_row, center_px] = 1.0

    if distractor_probability > 0:
        mask = torch.rand((batch, size, size), device=device) < distractor_probability
        obs[:, 3] = mask.float() * torch.rand((batch, size, size), device=device)
    return obs.contiguous(memory_format=torch.channels_last)


with torch.no_grad():
    _q = torch.arange(cfg.n_states, device=device).repeat_interleave(cfg.n_actions)
    _a = torch.arange(cfg.n_actions, device=device).repeat(cfg.n_states)
    TRUE_NOMINAL_TABLE = nominal_step(_q, _a).reshape(cfg.n_states, cfg.n_actions)


# =============================================================================
# 3. NOMINAL RECURRENT WORLD MODEL AND CRYSTALLIZATION
# =============================================================================

class NominalWorldModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(4, 32, 3, stride=2, padding=1),
            nn.SiLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.SiLU(inplace=True),
            nn.Conv2d(64, 96, 3, stride=2, padding=1),
            nn.SiLU(inplace=True),
        )
        encoded_side = math.ceil(cfg.image_size / 8)
        self.encoder_proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(96 * encoded_side * encoded_side, cfg.world_hidden_size),
            nn.Tanh(),
        )
        self.action_embedding = nn.Embedding(cfg.n_actions, cfg.world_action_embed_dim)
        self.gru = nn.GRU(
            cfg.world_action_embed_dim,
            cfg.world_hidden_size,
            batch_first=True,
        )
        self.state_head = nn.Linear(cfg.world_hidden_size, cfg.n_states)

    def encode_hidden(self, observation: torch.Tensor) -> torch.Tensor:
        return self.encoder_proj(self.encoder_conv(observation))

    def initial_logits(self, observation: torch.Tensor) -> torch.Tensor:
        return self.state_head(self.encode_hidden(observation))

    def rollout_from_hidden(
        self,
        hidden: torch.Tensor,
        actions: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        output, final_hidden = self.gru(
            self.action_embedding(actions),
            hidden.unsqueeze(0),
        )
        return self.state_head(output), final_hidden.squeeze(0)

    def forward(
        self,
        initial_observation: torch.Tensor,
        actions: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        hidden = self.encode_hidden(initial_observation)
        future_logits, _ = self.rollout_from_hidden(hidden, actions)
        return self.state_head(hidden), future_logits


world_model = NominalWorldModel().to(device)
if is_cuda:
    world_model.encoder_conv = world_model.encoder_conv.to(memory_format=torch.channels_last)
try:
    world_optimizer = torch.optim.AdamW(
        world_model.parameters(),
        lr=cfg.world_learning_rate,
        weight_decay=1.0e-4,
        fused=is_cuda,
    )
except (TypeError, RuntimeError):
    world_optimizer = torch.optim.AdamW(
        world_model.parameters(),
        lr=cfg.world_learning_rate,
        weight_decay=1.0e-4,
    )
world_scaler = make_scaler()
world_history: List[Dict[str, float]] = []
world_generator = torch.Generator(device=device).manual_seed(cfg.seed + 1)

print(f"nominal world-model parameters={sum(p.numel() for p in world_model.parameters()):,}")
world_model.train()
world_start_time = time.perf_counter()
for step in range(1, cfg.world_train_steps + 1):
    q0 = torch.randint(
        0,
        cfg.n_states,
        (cfg.world_batch_size,),
        generator=world_generator,
        device=device,
    )
    actions = torch.randint(
        0,
        cfg.n_actions,
        (cfg.world_batch_size, cfg.world_train_seq_len),
        generator=world_generator,
        device=device,
    )
    targets = rollout_nominal(q0, actions)
    zones = sample_from(ALL_ZONE_STARTS, cfg.world_batch_size, world_generator)
    winds = torch.where(
        torch.rand(cfg.world_batch_size, generator=world_generator, device=device) < 0.5,
        -torch.ones(cfg.world_batch_size, dtype=torch.long, device=device),
        torch.ones(cfg.world_batch_size, dtype=torch.long, device=device),
    )
    # Context is visible but causally irrelevant during nominal training. The
    # finite projection is explicitly trained to ignore it.
    obs0 = render_observation(q0, zones, winds, visible_context=True)

    world_optimizer.zero_grad(set_to_none=True)
    with amp_context():
        q0_logits, future_logits = world_model(obs0, actions)
        initial_loss = F.cross_entropy(q0_logits.float(), targets[:, 0])
        future_loss = F.cross_entropy(
            future_logits.float().reshape(-1, cfg.n_states),
            targets[:, 1:].reshape(-1),
        )
        loss = future_loss + cfg.world_initial_loss_weight * initial_loss
    world_scaler.scale(loss).backward()
    world_scaler.unscale_(world_optimizer)
    nn.utils.clip_grad_norm_(world_model.parameters(), 1.0)
    world_scaler.step(world_optimizer)
    world_scaler.update()

    report_every = max(1, cfg.world_train_steps // 20)
    if step == 1 or step % report_every == 0 or step == cfg.world_train_steps:
        with torch.no_grad():
            q0_acc = (q0_logits.argmax(-1) == targets[:, 0]).float().mean().item()
            final_acc = (
                future_logits[:, -1].argmax(-1) == targets[:, -1]
            ).float().mean().item()
        row = {
            "step": float(step),
            "loss": float(loss.item()),
            "q0_accuracy": q0_acc,
            "train_horizon_accuracy": final_acc,
        }
        world_history.append(row)
        print(
            f"world {step:4d}/{cfg.world_train_steps} | loss={loss.item():.4f} | "
            f"q0={100*q0_acc:5.1f}% | t{cfg.world_train_seq_len}={100*final_acc:5.1f}%"
        )
world_train_seconds = time.perf_counter() - world_start_time
world_model.eval()


@torch.inference_mode()
def crystallize_machine() -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    table = torch.empty((cfg.n_states, cfg.n_actions), dtype=torch.long, device=device)
    confidence = torch.empty_like(table, dtype=torch.float32)
    stability = torch.empty_like(table, dtype=torch.float32)
    generator = torch.Generator(device=device).manual_seed(cfg.seed + 2)
    for q_value in range(cfg.n_states):
        q_batch = torch.full(
            (cfg.extraction_nuisance_samples,),
            q_value,
            dtype=torch.long,
            device=device,
        )
        zones = sample_from(ALL_ZONE_STARTS, cfg.extraction_nuisance_samples, generator)
        winds = torch.where(
            torch.rand(cfg.extraction_nuisance_samples, generator=generator, device=device) < 0.5,
            -torch.ones(cfg.extraction_nuisance_samples, dtype=torch.long, device=device),
            torch.ones(cfg.extraction_nuisance_samples, dtype=torch.long, device=device),
        )
        obs = render_observation(q_batch, zones, winds, visible_context=True)
        hidden = world_model.encode_hidden(obs)
        for action in range(cfg.n_actions):
            action_batch = torch.full(
                (cfg.extraction_nuisance_samples, 1),
                action,
                dtype=torch.long,
                device=device,
            )
            logits, _ = world_model.rollout_from_hidden(hidden, action_batch)
            probabilities = logits[:, 0].float().softmax(-1)
            mean_probability = probabilities.mean(0)
            successor = mean_probability.argmax()
            table[q_value, action] = successor
            confidence[q_value, action] = mean_probability[successor]
            stability[q_value, action] = (
                probabilities.argmax(-1) == successor
            ).float().mean()
    return table, confidence, stability


MACHINE_TABLE, MACHINE_CONFIDENCE, MACHINE_STABILITY = crystallize_machine()
machine_transition_accuracy = (
    MACHINE_TABLE == TRUE_NOMINAL_TABLE
).float().mean().item()
print("\nCRYSTALLIZATION AUDIT")
print("-" * 96)
print(f"states={cfg.n_states} | actions={cfg.n_actions} | transitions={cfg.n_states*cfg.n_actions}")
print(f"withheld nominal transition accuracy={100*machine_transition_accuracy:.2f}%")
print(
    f"confidence mean/min={MACHINE_CONFIDENCE.mean().item():.4f}/"
    f"{MACHINE_CONFIDENCE.min().item():.4f} | "
    f"minimum context+nuisance stability={MACHINE_STABILITY.min().item():.4f}"
)
if not FAST_DEV_RUN and machine_transition_accuracy < 1.0:
    raise RuntimeError(
        "Primary V2 audit failed: the crystallized nominal transition table is not exact. "
        "Increase world_train_steps or report a failed crystallization condition."
    )


# =============================================================================
# 4. PAIRED ACTION-CONDITIONAL DATASETS
# =============================================================================

def labels_for_plan(
    q0: torch.Tensor,
    actions: torch.Tensor,
    zone_start: torch.Tensor,
) -> torch.Tensor:
    nominal_states = rollout_nominal(q0, actions)
    future_positions, _ = decode_state(nominal_states[:, 1:])
    entered = (
        (future_positions >= zone_start[:, None])
        & (future_positions < (zone_start + cfg.zone_width)[:, None])
    )
    return entered.long().cumsum(dim=1).clamp_max(1).float()


def generate_paired_dataset(
    n_pairs: int,
    allowed_zone_starts: Sequence[int],
    seed: int,
) -> Dict[str, torch.Tensor]:
    """Interleaved order is [risky_0, safe_0, risky_1, safe_1, ...]."""
    generator = torch.Generator(device=device).manual_seed(seed)
    collected: Dict[str, List[torch.Tensor]] = {
        "q": [],
        "zone": [],
        "wind": [],
        "actions": [],
        "labels": [],
    }
    pairs_collected = 0
    attempts = 0
    context_batch = 512 if not FAST_DEV_RUN else 64

    while pairs_collected < n_pairs:
        zones = sample_from(allowed_zone_starts, context_batch, generator)
        q0 = sample_states_outside_zone(context_batch, zones, generator)
        winds = torch.where(
            torch.rand(context_batch, generator=generator, device=device) < 0.5,
            -torch.ones(context_batch, dtype=torch.long, device=device),
            torch.ones(context_batch, dtype=torch.long, device=device),
        )
        candidates = torch.randint(
            0,
            cfg.n_actions,
            (
                context_batch,
                cfg.candidate_plans_per_context,
                cfg.max_prediction_horizon,
            ),
            generator=generator,
            device=device,
        )
        flat_actions = candidates.reshape(-1, cfg.max_prediction_horizon)
        flat_q = q0[:, None].expand(-1, cfg.candidate_plans_per_context).reshape(-1)
        flat_zone = zones[:, None].expand(-1, cfg.candidate_plans_per_context).reshape(-1)
        final_labels = labels_for_plan(flat_q, flat_actions, flat_zone)[:, -1].bool()
        final_labels = final_labels.reshape(context_batch, cfg.candidate_plans_per_context)
        has_risky = final_labels.any(dim=1)
        has_safe = (~final_labels).any(dim=1)
        usable = has_risky & has_safe
        if bool(usable.any()):
            usable_ids = torch.nonzero(usable, as_tuple=False).squeeze(1)
            risky_index = final_labels[usable].float().argmax(dim=1)
            safe_index = (~final_labels[usable]).float().argmax(dim=1)
            candidate_subset = candidates[usable]
            row_ids = torch.arange(candidate_subset.shape[0], device=device)
            risky_actions = candidate_subset[row_ids, risky_index]
            safe_actions = candidate_subset[row_ids, safe_index]
            paired_actions = torch.stack([risky_actions, safe_actions], dim=1).reshape(
                -1, cfg.max_prediction_horizon
            )
            paired_q = q0[usable_ids].repeat_interleave(2)
            paired_zone = zones[usable_ids].repeat_interleave(2)
            paired_wind = winds[usable_ids].repeat_interleave(2)
            paired_labels = labels_for_plan(paired_q, paired_actions, paired_zone)

            collected["q"].append(paired_q.cpu())
            collected["zone"].append(paired_zone.cpu())
            collected["wind"].append(paired_wind.cpu())
            collected["actions"].append(paired_actions.cpu())
            collected["labels"].append(paired_labels.cpu())
            pairs_collected += usable_ids.numel()

        attempts += 1
        if attempts > 10_000:
            raise RuntimeError("Paired-plan generator could not find enough safe/risky contexts.")

    result = {key: torch.cat(value, dim=0)[: 2 * n_pairs] for key, value in collected.items()}
    # Audit interleaving and action-only difference.
    if not bool((result["labels"][0::2, -1] == 1).all()):
        raise RuntimeError("Paired-data audit failed: risky item missing positive label.")
    if not bool((result["labels"][1::2, -1] == 0).all()):
        raise RuntimeError("Paired-data audit failed: safe item has an exit label.")
    if bool((result["actions"][0::2] == result["actions"][1::2]).all(dim=1).any()):
        raise RuntimeError("Paired-data audit failed: a pair has identical actions.")
    return result


dataset_start = time.perf_counter()
train_data = generate_paired_dataset(
    cfg.monitor_train_pairs,
    cfg.train_zone_starts,
    cfg.seed + 10,
)
validation_data = generate_paired_dataset(
    cfg.monitor_validation_pairs,
    cfg.validation_zone_starts,
    cfg.seed + 11,
)
test_data = generate_paired_dataset(
    cfg.monitor_test_pairs,
    cfg.test_zone_starts,
    cfg.seed + 12,
)
dataset_seconds = time.perf_counter() - dataset_start
print("\nPAIRED-DATA AUDIT")
print("-" * 96)
print(
    f"train/validation/test samples={len(train_data['q'])}/"
    f"{len(validation_data['q'])}/{len(test_data['q'])} | "
    f"generation={dataset_seconds:.1f}s"
)
print(
    f"zone starts train={cfg.train_zone_starts} | validation={cfg.validation_zone_starts} | "
    f"held-out test={cfg.test_zone_starts}"
)


def generate_hidden_switch_dataset(n_samples: int, seed: int) -> Dict[str, torch.Tensor]:
    """Exit time is independent of q, observation, and proposed actions."""
    generator = torch.Generator(device=device).manual_seed(seed)
    q0 = torch.randint(0, cfg.n_states, (n_samples,), generator=generator, device=device)
    actions = torch.randint(
        0,
        cfg.n_actions,
        (n_samples, cfg.max_prediction_horizon),
        generator=generator,
        device=device,
    )
    has_event = torch.rand(n_samples, generator=generator, device=device) < 0.5
    event_time = torch.randint(
        1,
        cfg.max_prediction_horizon + 1,
        (n_samples,),
        generator=generator,
        device=device,
    )
    event_time = torch.where(
        has_event,
        event_time,
        torch.full_like(event_time, cfg.max_prediction_horizon + 1),
    )
    steps = torch.arange(1, cfg.max_prediction_horizon + 1, device=device)[None, :]
    labels = (steps >= event_time[:, None]).float()
    # Stored context values are decoys and are hidden at rendering time.
    zones = sample_from(ALL_ZONE_STARTS, n_samples, generator)
    winds = torch.ones(n_samples, dtype=torch.long, device=device)
    return {
        "q": q0.cpu(),
        "zone": zones.cpu(),
        "wind": winds.cpu(),
        "actions": actions.cpu(),
        "labels": labels.cpu(),
    }


hidden_train_data = generate_hidden_switch_dataset(
    cfg.hidden_train_samples,
    cfg.seed + 13,
)
hidden_test_data = generate_hidden_switch_dataset(
    cfg.hidden_test_samples,
    cfg.seed + 14,
)


# =============================================================================
# 5. PROSPECTIVE MONITORS AND ABLATIONS
# =============================================================================

class ProspectiveValidityMonitor(nn.Module):
    def __init__(self, mode: str) -> None:
        super().__init__()
        if mode not in {"obs_action", "obs_only", "action_only"}:
            raise ValueError(f"Unknown monitor mode: {mode}")
        self.mode = mode
        self.obs_encoder = nn.Sequential(
            nn.Conv2d(4, 24, 3, stride=2, padding=1),
            nn.SiLU(inplace=True),
            nn.Conv2d(24, 48, 3, stride=2, padding=1),
            nn.SiLU(inplace=True),
            nn.Conv2d(48, 64, 3, stride=2, padding=1),
            nn.SiLU(inplace=True),
            nn.Flatten(),
            nn.Linear(64 * math.ceil(cfg.image_size / 8) ** 2, cfg.monitor_obs_dim),
            nn.SiLU(inplace=True),
        )
        self.action_embedding = nn.Embedding(cfg.n_actions, cfg.monitor_action_embed)
        self.action_gru = nn.GRU(
            cfg.monitor_action_embed,
            cfg.monitor_action_hidden,
            batch_first=True,
        )
        self.horizon_embedding = nn.Embedding(cfg.max_prediction_horizon, 16)

        feature_dim = 16
        if mode in {"obs_action", "obs_only"}:
            feature_dim += cfg.monitor_obs_dim
        if mode in {"obs_action", "action_only"}:
            feature_dim += cfg.monitor_action_hidden
        self.head = nn.Sequential(
            nn.Linear(feature_dim, 96),
            nn.SiLU(inplace=True),
            nn.Dropout(0.05),
            nn.Linear(96, 1),
        )

    def forward(self, observation: torch.Tensor, actions: torch.Tensor) -> torch.Tensor:
        batch = actions.shape[0]
        action_output, _ = self.action_gru(self.action_embedding(actions))
        horizon_ids = torch.arange(cfg.max_prediction_horizon, device=actions.device)
        horizon_features = self.horizon_embedding(horizon_ids)[None].expand(batch, -1, -1)
        features = [horizon_features]
        if self.mode in {"obs_action", "obs_only"}:
            obs_features = self.obs_encoder(observation)
            features.append(obs_features[:, None].expand(-1, cfg.max_prediction_horizon, -1))
        if self.mode in {"obs_action", "action_only"}:
            features.append(action_output)
        return self.head(torch.cat(features, dim=-1)).squeeze(-1)


def make_monitor_optimizer(model: nn.Module):
    try:
        return torch.optim.AdamW(
            model.parameters(),
            lr=cfg.monitor_learning_rate,
            weight_decay=1.0e-4,
            fused=is_cuda,
        )
    except (TypeError, RuntimeError):
        return torch.optim.AdamW(
            model.parameters(),
            lr=cfg.monitor_learning_rate,
            weight_decay=1.0e-4,
        )


def render_dataset_batch(
    data: Dict[str, torch.Tensor],
    indices: torch.Tensor,
    visible_context: bool,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    ids = indices.cpu()
    q = data["q"][ids].to(device, non_blocking=True)
    zones = data["zone"][ids].to(device, non_blocking=True)
    winds = data["wind"][ids].to(device, non_blocking=True)
    actions = data["actions"][ids].to(device, non_blocking=True)
    labels = data["labels"][ids].to(device, non_blocking=True)
    obs = render_observation(q, zones, winds, visible_context=visible_context)
    return obs, actions, labels


def train_monitor(
    mode: str,
    data: Dict[str, torch.Tensor],
    steps: int,
    seed: int,
    visible_context: bool,
) -> Tuple[ProspectiveValidityMonitor, List[float]]:
    torch.manual_seed(seed)
    model = ProspectiveValidityMonitor(mode).to(device)
    if is_cuda:
        model.obs_encoder = model.obs_encoder.to(memory_format=torch.channels_last)
    optimizer = make_monitor_optimizer(model)
    scaler = make_scaler()
    labels = data["labels"].float()
    positives = labels.sum(dim=0).clamp_min(1.0)
    negatives = labels.shape[0] - positives
    pos_weight = (negatives / positives).clamp(0.5, 25.0).to(device)
    history: List[float] = []
    generator = torch.Generator().manual_seed(seed + 1)
    model.train()

    for step in range(1, steps + 1):
        ids = torch.randint(
            0,
            len(data["q"]),
            (cfg.monitor_batch_size,),
            generator=generator,
        )
        obs, actions, target = render_dataset_batch(data, ids, visible_context)
        optimizer.zero_grad(set_to_none=True)
        with amp_context():
            logits = model(obs, actions)
            loss = F.binary_cross_entropy_with_logits(
                logits.float(),
                target,
                pos_weight=pos_weight,
            )
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        history.append(float(loss.item()))

        report_every = max(1, steps // 5)
        if step == 1 or step % report_every == 0 or step == steps:
            print(f"monitor {mode:11s} {step:4d}/{steps} | loss={loss.item():.4f}")
    model.eval()
    return model, history


print("\nTRAINING PROSPECTIVE MONITORS")
print("-" * 96)
joint_monitor, joint_history = train_monitor(
    "obs_action",
    train_data,
    cfg.joint_train_steps,
    cfg.seed + 20,
    visible_context=True,
)
obs_monitor, obs_history = train_monitor(
    "obs_only",
    train_data,
    cfg.ablation_train_steps,
    cfg.seed + 21,
    visible_context=True,
)
action_monitor, action_history = train_monitor(
    "action_only",
    train_data,
    cfg.ablation_train_steps,
    cfg.seed + 22,
    visible_context=True,
)
hidden_monitor, hidden_history = train_monitor(
    "obs_action",
    hidden_train_data,
    cfg.hidden_control_train_steps,
    cfg.seed + 23,
    visible_context=False,
)


@torch.inference_mode()
def predict_monitor(
    model: ProspectiveValidityMonitor,
    data: Dict[str, torch.Tensor],
    visible_context: bool,
    nuisance_averages: int,
) -> np.ndarray:
    outputs: List[torch.Tensor] = []
    chunk = 2048 if not FAST_DEV_RUN else 128
    for start in range(0, len(data["q"]), chunk):
        end = min(start + chunk, len(data["q"]))
        ids = torch.arange(start, end)
        probability_sum = None
        for _ in range(nuisance_averages):
            obs, actions, _ = render_dataset_batch(data, ids, visible_context)
            with amp_context():
                probabilities = model(obs, actions).float().sigmoid()
            probability_sum = probabilities if probability_sum is None else probability_sum + probabilities
        outputs.append((probability_sum / nuisance_averages).cpu())
    return torch.cat(outputs, dim=0).numpy()


def binary_auroc(labels: np.ndarray, scores: np.ndarray) -> float:
    labels = labels.astype(np.int64)
    positives = int(labels.sum())
    negatives = int((1 - labels).sum())
    if positives == 0 or negatives == 0:
        return float("nan")
    order = np.argsort(-scores, kind="mergesort")
    y = labels[order]
    tpr = np.r_[0.0, np.cumsum(y) / positives, 1.0]
    fpr = np.r_[0.0, np.cumsum(1 - y) / negatives, 1.0]
    if hasattr(np, "trapezoid"):
        return float(np.trapezoid(tpr, fpr))
    return float(np.trapz(tpr, fpr))


def binary_auprc(labels: np.ndarray, scores: np.ndarray) -> float:
    labels = labels.astype(np.int64)
    positives = int(labels.sum())
    if positives == 0:
        return float("nan")
    order = np.argsort(-scores, kind="mergesort")
    y = labels[order]
    tp = np.cumsum(y)
    fp = np.cumsum(1 - y)
    recall = tp / positives
    precision = tp / np.maximum(tp + fp, 1)
    recall = np.r_[0.0, recall]
    precision = np.r_[1.0, precision]
    return float(np.sum((recall[1:] - recall[:-1]) * precision[1:]))


def horizon_metrics(
    name: str,
    labels: np.ndarray,
    probabilities: np.ndarray,
) -> List[Dict[str, float]]:
    rows: List[Dict[str, float]] = []
    for horizon in cfg.reported_horizons:
        index = horizon - 1
        y = labels[:, index]
        p = probabilities[:, index]
        rows.append(
            {
                "monitor": name,
                "horizon": float(horizon),
                "prevalence": float(y.mean()),
                "auroc": binary_auroc(y, p),
                "auprc": binary_auprc(y, p),
                "brier": float(np.mean((p - y) ** 2)),
            }
        )
    return rows


test_labels_np = test_data["labels"].numpy()
hidden_labels_np = hidden_test_data["labels"].numpy()
joint_test_probs = predict_monitor(
    joint_monitor,
    test_data,
    True,
    cfg.evaluation_nuisance_averages,
)
obs_test_probs = predict_monitor(
    obs_monitor,
    test_data,
    True,
    cfg.evaluation_nuisance_averages,
)
action_test_probs = predict_monitor(
    action_monitor,
    test_data,
    True,
    cfg.evaluation_nuisance_averages,
)
hidden_test_probs = predict_monitor(
    hidden_monitor,
    hidden_test_data,
    False,
    cfg.evaluation_nuisance_averages,
)

metric_rows: List[Dict[str, float]] = []
metric_rows += horizon_metrics("observation+actions", test_labels_np, joint_test_probs)
metric_rows += horizon_metrics("observation-only", test_labels_np, obs_test_probs)
metric_rows += horizon_metrics("actions-only", test_labels_np, action_test_probs)
metric_rows += horizon_metrics("unobservable-switch", hidden_labels_np, hidden_test_probs)


def paired_ranking_accuracy(probabilities: np.ndarray) -> float:
    risky = probabilities[0::2, -1]
    safe = probabilities[1::2, -1]
    return float(np.mean((risky > safe).astype(float) + 0.5 * (risky == safe)))


paired_ranking = {
    "observation+actions": paired_ranking_accuracy(joint_test_probs),
    "observation-only": paired_ranking_accuracy(obs_test_probs),
    "actions-only": paired_ranking_accuracy(action_test_probs),
}

print("\nHELD-OUT ZONE PREDICTION")
print("-" * 96)
print(" monitor               | horizon | AUROC | AUPRC | Brier")
for row in metric_rows:
    print(
        f" {row['monitor']:21s} | {int(row['horizon']):7d} | "
        f"{row['auroc']:.3f} | {row['auprc']:.3f} | {row['brier']:.4f}"
    )
print("paired risky>safe ranking:", {k: round(v, 3) for k, v in paired_ranking.items()})


# =============================================================================
# 6. PROSPECTIVE LEAD TIME VS POSTDICTIVE COUNTEREXAMPLE LATENCY
# =============================================================================

joint_validation_probs = predict_monitor(
    joint_monitor,
    validation_data,
    True,
    cfg.evaluation_nuisance_averages,
)
validation_safe_scores = joint_validation_probs[1::2, -1]
alert_threshold = float(
    np.quantile(validation_safe_scores, 1.0 - cfg.safe_plan_fpr_target)
)

risky_q = test_data["q"][0::2].to(device)
risky_zone = test_data["zone"][0::2].to(device)
risky_wind = test_data["wind"][0::2].to(device)
risky_actions = test_data["actions"][0::2].to(device)
nominal_risky_states = rollout_nominal(risky_q, risky_actions)
actual_risky_states = rollout_hazard(
    risky_q,
    risky_actions,
    risky_zone,
    risky_wind,
)
entered_labels = test_data["labels"][0::2].bool()
exit_time = entered_labels.long().argmax(dim=1).numpy() + 1
counterexample_mask = (actual_risky_states[:, 1:] != nominal_risky_states[:, 1:]).cpu().numpy()
has_counterexample = counterexample_mask.any(axis=1)
cex_time = np.where(
    has_counterexample,
    counterexample_mask.argmax(axis=1) + 1,
    cfg.max_prediction_horizon + 1,
)

prospective_scores = joint_test_probs[0::2, -1]
safe_test_scores = joint_test_probs[1::2, -1]
prospective_detected = prospective_scores > alert_threshold
prospective_latency = -exit_time[prospective_detected]
postdictive_latency = cex_time[has_counterexample] - exit_time[has_counterexample]

latency_results = {
    "validation_threshold": alert_threshold,
    "safe_plan_false_positive_rate": float(np.mean(safe_test_scores > alert_threshold)),
    "prospective_detection_rate": float(np.mean(prospective_detected)),
    "prospective_median_latency_detected": (
        float(np.median(prospective_latency)) if prospective_latency.size else None
    ),
    "prospective_fraction_alerted_before_exit": float(
        np.mean(prospective_detected)
    ),
    "manifest_counterexample_fraction": float(np.mean(has_counterexample)),
    "postdictive_median_latency_manifest": (
        float(np.median(postdictive_latency)) if postdictive_latency.size else None
    ),
    "postdictive_minimum_latency_manifest": (
        float(np.min(postdictive_latency)) if postdictive_latency.size else None
    ),
}

print("\nLEAD-TIME CONTRAST")
print("-" * 96)
print(
    f"threshold={alert_threshold:.4f} | held-out safe-plan FPR="
    f"{100*latency_results['safe_plan_false_positive_rate']:.2f}%"
)
print(
    f"prospective detection={100*latency_results['prospective_detection_rate']:.2f}% | "
    f"median alert-exit latency={latency_results['prospective_median_latency_detected']} steps"
)
print(
    f"manifest counterexamples={100*latency_results['manifest_counterexample_fraction']:.2f}% | "
    f"postdictive median/min latency={latency_results['postdictive_median_latency_manifest']}/"
    f"{latency_results['postdictive_minimum_latency_manifest']} steps"
)


# =============================================================================
# 7. COUNTERFACTUAL VALIDITY-AWARE PLANNING
# =============================================================================

def generate_planning_cases(n_cases: int, seed: int) -> List[Dict[str, object]]:
    """
    Find safe and risky action sequences that reach the same target at the same
    horizon under the nominal crystallized machine. A risky path is retained
    with its earliest discovered zone entry to make the boundary consequential.
    """
    rng = np.random.default_rng(seed)
    # FAST_DEV_RUN deliberately undertrains the world model; use the withheld
    # nominal table only to exercise downstream planning code in that smoke path.
    # Full V2 always uses the audited crystallized table.
    planning_table_tensor = (
        TRUE_NOMINAL_TABLE
        if FAST_DEV_RUN and machine_transition_accuracy < 1.0
        else MACHINE_TABLE
    )
    table = planning_table_tensor.detach().cpu().numpy()
    cases: List[Dict[str, object]] = []
    attempts = 0
    while len(cases) < n_cases:
        zone = int(rng.choice(cfg.test_zone_starts))
        wind = int(rng.choice([-1, 1]))
        start = int(rng.integers(0, cfg.n_states))
        start_position = start // cfg.n_vel
        if zone <= start_position < zone + cfg.zone_width:
            continue
        horizon = int(
            rng.integers(cfg.planning_min_horizon, cfg.planning_max_horizon + 1)
        )
        # value: (action list, first entry time or None)
        frontier: Dict[Tuple[int, bool], Tuple[List[int], Optional[int]]] = {
            (start, False): ([], None)
        }
        for depth in range(1, horizon + 1):
            next_frontier: Dict[Tuple[int, bool], Tuple[List[int], Optional[int]]] = {}
            for (q, visited), (prefix, first_entry) in frontier.items():
                for action in range(cfg.n_actions):
                    q_next = int(table[q, action])
                    p_next = q_next // cfg.n_vel
                    enters = zone <= p_next < zone + cfg.zone_width
                    visited_next = visited or enters
                    entry_next = first_entry if first_entry is not None else (depth if enters else None)
                    key = (q_next, visited_next)
                    candidate = (prefix + [action], entry_next)
                    if key not in next_frontier:
                        next_frontier[key] = candidate
                    elif visited_next:
                        old_entry = next_frontier[key][1]
                        if old_entry is None or (
                            entry_next is not None and entry_next < old_entry
                        ):
                            next_frontier[key] = candidate
            frontier = next_frontier

        targets = [
            q
            for q in range(cfg.n_states)
            if (q, False) in frontier
            and (q, True) in frontier
            and frontier[(q, True)][1] is not None
            and int(frontier[(q, True)][1]) < horizon
        ]
        if targets:
            target = int(rng.choice(targets))
            safe_plan = frontier[(target, False)][0]
            risky_plan, first_entry = frontier[(target, True)]
            cases.append(
                {
                    "start_state": start,
                    "target_state": target,
                    "zone_start": zone,
                    "wind_direction": wind,
                    "horizon": horizon,
                    "safe_actions": safe_plan,
                    "risky_actions": risky_plan,
                    "risky_exit_time": int(first_entry),
                }
            )
        attempts += 1
        if attempts > 100_000:
            raise RuntimeError("Could not generate enough counterfactual planning cases.")
    return cases


planning_cases = generate_planning_cases(cfg.planning_cases, cfg.seed + 30)


def planning_tensor_data(cases: List[Dict[str, object]]) -> Dict[str, torch.Tensor]:
    q_values: List[int] = []
    zones: List[int] = []
    winds: List[int] = []
    actions: List[List[int]] = []
    labels: List[List[float]] = []
    for case in cases:
        horizon = int(case["horizon"])
        for is_risky, plan in [(0, case["safe_actions"]), (1, case["risky_actions"])]:
            padded = list(plan) + [1] * (cfg.max_prediction_horizon - horizon)
            cumulative = [0.0] * cfg.max_prediction_horizon
            if is_risky:
                exit_index = int(case["risky_exit_time"]) - 1
                for i in range(exit_index, cfg.max_prediction_horizon):
                    cumulative[i] = 1.0
            q_values.append(int(case["start_state"]))
            zones.append(int(case["zone_start"]))
            winds.append(int(case["wind_direction"]))
            actions.append(padded)
            labels.append(cumulative)
    return {
        "q": torch.tensor(q_values, dtype=torch.long),
        "zone": torch.tensor(zones, dtype=torch.long),
        "wind": torch.tensor(winds, dtype=torch.long),
        "actions": torch.tensor(actions, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.float32),
    }


planning_data = planning_tensor_data(planning_cases)
planning_joint_probs = predict_monitor(joint_monitor, planning_data, True, 1)
planning_obs_probs = predict_monitor(obs_monitor, planning_data, True, 1)
planning_action_probs = predict_monitor(action_monitor, planning_data, True, 1)

planning_rng = np.random.default_rng(cfg.seed + 31)
method_names = [
    "machine-random-tie",
    "observation-only",
    "actions-only",
    "observation+actions",
    "oracle-boundary",
]
planning_summary: Dict[str, Dict[str, float]] = {}
planning_case_rows: List[Dict[str, object]] = []


def choose_safe_from_scores(safe_score: float, risky_score: float) -> bool:
    if safe_score < risky_score:
        return True
    if safe_score > risky_score:
        return False
    return bool(planning_rng.integers(0, 2))


choices_by_method: Dict[str, List[bool]] = {name: [] for name in method_names}
risky_goal_success: List[bool] = []

for i, case in enumerate(planning_cases):
    horizon = int(case["horizon"])
    score_index = horizon - 1
    safe_row = 2 * i
    risky_row = safe_row + 1

    q0 = torch.tensor([int(case["start_state"])], dtype=torch.long, device=device)
    zone = torch.tensor([int(case["zone_start"])], dtype=torch.long, device=device)
    wind = torch.tensor([int(case["wind_direction"])], dtype=torch.long, device=device)
    risky_actions_tensor = torch.tensor(
        [case["risky_actions"]], dtype=torch.long, device=device
    )
    risky_actual_final = int(
        rollout_hazard(q0, risky_actions_tensor, zone, wind)[0, -1].item()
    )
    risky_succeeds = risky_actual_final == int(case["target_state"])
    risky_goal_success.append(risky_succeeds)

    method_choices = {
        "machine-random-tie": bool(planning_rng.integers(0, 2)),
        "observation-only": choose_safe_from_scores(
            float(planning_obs_probs[safe_row, score_index]),
            float(planning_obs_probs[risky_row, score_index]),
        ),
        "actions-only": choose_safe_from_scores(
            float(planning_action_probs[safe_row, score_index]),
            float(planning_action_probs[risky_row, score_index]),
        ),
        "observation+actions": choose_safe_from_scores(
            float(planning_joint_probs[safe_row, score_index]),
            float(planning_joint_probs[risky_row, score_index]),
        ),
        "oracle-boundary": True,
    }
    for method, chose_safe in method_choices.items():
        choices_by_method[method].append(chose_safe)

    planning_case_rows.append(
        {
            "case_id": i,
            "start_state": int(case["start_state"]),
            "target_state": int(case["target_state"]),
            "zone_start": int(case["zone_start"]),
            "wind_direction": int(case["wind_direction"]),
            "horizon": horizon,
            "risky_exit_time": int(case["risky_exit_time"]),
            "risky_true_goal_success": int(risky_succeeds),
            "joint_safe_score": float(planning_joint_probs[safe_row, score_index]),
            "joint_risky_score": float(planning_joint_probs[risky_row, score_index]),
            **{f"{method}_chose_safe": int(choice) for method, choice in method_choices.items()},
        }
    )

for method in method_names:
    chose_safe = np.asarray(choices_by_method[method], dtype=bool)
    risky_success_np = np.asarray(risky_goal_success, dtype=bool)
    goal_success = chose_safe | ((~chose_safe) & risky_success_np)
    planning_summary[method] = {
        "safe_plan_selection_rate": float(chose_safe.mean()),
        "domain_exit_rate": float((~chose_safe).mean()),
        "true_goal_success_rate": float(goal_success.mean()),
    }

print("\nVALIDITY-AWARE COUNTERFACTUAL PLANNING")
print("-" * 96)
print(" method                | safe selection | domain exits | true goal success")
for method in method_names:
    row = planning_summary[method]
    print(
        f" {method:21s} | {100*row['safe_plan_selection_rate']:12.2f}% | "
        f"{100*row['domain_exit_rate']:10.2f}% | {100*row['true_goal_success_rate']:16.2f}%"
    )


# =============================================================================
# 8. SAVE RESULTS AND FIGURE
# =============================================================================

config_json = asdict(cfg)
for key in [
    "train_zone_starts",
    "validation_zone_starts",
    "test_zone_starts",
    "reported_horizons",
]:
    config_json[key] = list(config_json[key])

results = {
    "interpretation_boundary": (
        "The finite state and actions are supplied. The machine is exact only conditional on "
        "validity. The neural monitor estimates statistical boundary risk; autonomous state "
        "discovery and formal certification of the boundary are not tested."
    ),
    "runtime": {
        "device": str(device),
        "gpu": torch.cuda.get_device_name(0) if is_cuda else None,
        "torch_version": torch.__version__,
        "amp": amp_enabled,
        "world_train_seconds": world_train_seconds,
        "paired_dataset_seconds": dataset_seconds,
    },
    "crystallization": {
        "transition_accuracy": machine_transition_accuracy,
        "mean_confidence": float(MACHINE_CONFIDENCE.mean().item()),
        "minimum_confidence": float(MACHINE_CONFIDENCE.min().item()),
        "minimum_context_nuisance_stability": float(MACHINE_STABILITY.min().item()),
    },
    "horizon_metrics": metric_rows,
    "paired_plan_ranking": paired_ranking,
    "latency": latency_results,
    "planning": planning_summary,
}

with (out_dir / "config.json").open("w", encoding="utf-8") as f:
    json.dump(config_json, f, indent=2)
with (out_dir / "results.json").open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

with (out_dir / "horizon_metrics.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(metric_rows[0].keys()))
    writer.writeheader()
    writer.writerows(metric_rows)
with (out_dir / "planning_cases.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(planning_case_rows[0].keys()))
    writer.writeheader()
    writer.writerows(planning_case_rows)

np.savez_compressed(
    out_dir / "crystallized_machine.npz",
    transition_table=MACHINE_TABLE.detach().cpu().numpy(),
    confidence=MACHINE_CONFIDENCE.detach().cpu().numpy(),
    stability=MACHINE_STABILITY.detach().cpu().numpy(),
)
torch.save(
    {"state_dict": world_model.state_dict(), "config": config_json},
    out_dir / "nominal_world_model.pt",
)
torch.save(
    {"state_dict": joint_monitor.state_dict(), "config": config_json},
    out_dir / "prospective_joint_monitor.pt",
)
torch.save(
    {"state_dict": obs_monitor.state_dict(), "config": config_json},
    out_dir / "observation_only_monitor.pt",
)
torch.save(
    {"state_dict": action_monitor.state_dict(), "config": config_json},
    out_dir / "action_only_monitor.pt",
)
torch.save(
    {"state_dict": hidden_monitor.state_dict(), "config": config_json},
    out_dir / "hidden_switch_control_monitor.pt",
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

colors = {
    "observation+actions": "#54A24B",
    "observation-only": "#4C78A8",
    "actions-only": "#F58518",
    "unobservable-switch": "#B279A2",
}
for monitor_name in colors:
    rows = [row for row in metric_rows if row["monitor"] == monitor_name]
    axes[0, 0].plot(
        [row["horizon"] for row in rows],
        [row["auroc"] for row in rows],
        "o-",
        label=monitor_name,
        color=colors[monitor_name],
    )
axes[0, 0].axhline(0.5, color="black", linestyle="--", linewidth=1)
axes[0, 0].set_ylim(0.45, 1.02)
axes[0, 0].set_title("Exit-within-k prediction on held-out zone locations")
axes[0, 0].set_xlabel("prediction horizon k")
axes[0, 0].set_ylabel("AUROC")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.25)

planning_methods = method_names
safe_rates = [planning_summary[m]["safe_plan_selection_rate"] for m in planning_methods]
axes[0, 1].barh(planning_methods, safe_rates, color=["#BAB0AC", "#4C78A8", "#F58518", "#54A24B", "#72B7B2"])
axes[0, 1].set_xlim(0, 1.02)
axes[0, 1].set_title("Machine-equivalent plan selection")
axes[0, 1].set_xlabel("fraction choosing the domain-preserving plan")
axes[0, 1].grid(axis="x", alpha=0.25)

latency_bins = np.arange(-cfg.max_prediction_horizon - 0.5, cfg.max_prediction_horizon + 1.5, 1)
if prospective_latency.size:
    axes[1, 0].hist(
        prospective_latency,
        bins=latency_bins,
        alpha=0.7,
        density=True,
        label="prospective alert",
        color="#54A24B",
    )
if postdictive_latency.size:
    axes[1, 0].hist(
        postdictive_latency,
        bins=latency_bins,
        alpha=0.7,
        density=True,
        label="postdictive disagreement",
        color="#E45756",
    )
axes[1, 0].axvline(0, color="black", linestyle="--")
axes[1, 0].set_title("Alert time relative to domain exit")
axes[1, 0].set_xlabel("alert time - tau_exit (negative = advance warning)")
axes[1, 0].set_ylabel("density")
axes[1, 0].legend()

axes[1, 1].hist(
    joint_test_probs[1::2, -1],
    bins=np.linspace(0, 1, 40),
    alpha=0.7,
    density=True,
    label="safe paired plans",
    color="#4C78A8",
)
axes[1, 1].hist(
    joint_test_probs[0::2, -1],
    bins=np.linspace(0, 1, 40),
    alpha=0.7,
    density=True,
    label="zone-entering paired plans",
    color="#E45756",
)
axes[1, 1].axvline(alert_threshold, color="black", linestyle="--", label="validation threshold")
axes[1, 1].set_title("Prospective risk at planning time")
axes[1, 1].set_xlabel(f"P(exit within {cfg.max_prediction_horizon} steps)")
axes[1, 1].set_ylabel("density")
axes[1, 1].legend()

fig.suptitle(
    "Prospective validity around a crystallized finite world model\n"
    f"machine transitions={100*machine_transition_accuracy:.1f}% | "
    f"joint paired ranking={100*paired_ranking['observation+actions']:.1f}% | "
    f"safe-plan selection={100*planning_summary['observation+actions']['safe_plan_selection_rate']:.1f}%",
    fontsize=14,
)
fig.savefig(out_dir / "summary.png", dpi=180)
plt.show()

archive_path = out_dir / "run_bundle.zip"
if archive_path.exists():
    archive_path.unlink()
temporary_archive_base = out_dir.parent / f".{out_dir.name}_run_bundle_tmp"
temporary_archive = temporary_archive_base.with_suffix(".zip")
if temporary_archive.exists():
    temporary_archive.unlink()
made_archive = shutil.make_archive(
    str(temporary_archive_base),
    "zip",
    root_dir=out_dir,
    base_dir=".",
)
shutil.move(made_archive, archive_path)

print("\n" + "=" * 96)
print("FINAL V2 SUMMARY")
print("=" * 96)
print(f"nominal machine transition accuracy: {100*machine_transition_accuracy:.2f}%")
print(
    f"paired risky>safe ranking — joint/obs/action: "
    f"{100*paired_ranking['observation+actions']:.2f}% / "
    f"{100*paired_ranking['observation-only']:.2f}% / "
    f"{100*paired_ranking['actions-only']:.2f}%"
)
print(
    f"prospective detection={100*latency_results['prospective_detection_rate']:.2f}% | "
    f"safe FPR={100*latency_results['safe_plan_false_positive_rate']:.2f}% | "
    f"median prospective latency={latency_results['prospective_median_latency_detected']}"
)
print(
    f"validity-aware safe-plan selection="
    f"{100*planning_summary['observation+actions']['safe_plan_selection_rate']:.2f}% | "
    f"machine-only={100*planning_summary['machine-random-tie']['safe_plan_selection_rate']:.2f}%"
)
hidden_final = [
    row for row in metric_rows
    if row["monitor"] == "unobservable-switch"
    and int(row["horizon"]) == cfg.max_prediction_horizon
][0]
print(f"unobservable-switch AUROC at k={cfg.max_prediction_horizon}: {hidden_final['auroc']:.3f}")
print(f"artifacts: {out_dir}")
print(f"bundle: {archive_path}")


## Experiment 4: Homeostatic symbolic recovery V3

Original cell `3`.


In [ ]:
"""
HOMEOSTATIC SYMBOLIC RECOVERY V3 — T4-OPTIMIZED, SINGLE-CELL COLAB EXPERIMENT

Paste this entire file into one Google Colab cell and run it with a T4 GPU.

FROZEN QUESTION
---------------
Can a crystallized finite world model act as homeostatic memory after an
unexpected intervention by (1) snapping an incorrect predicted state back to an
observed symbolic state, (2) temporarily yielding to a neural fallback while its
validity is suspended, and (3) triggering structural re-examination when the
dynamics change permanently?

FAILURE TAXONOMY
----------------
L1 — state error: one unannounced impulse; nominal dynamics resume immediately.
L2 — transient validity error: an unannounced force lasts 2–5 transitions.
L3 — model/regime error: the force begins once and remains for the episode.

At the first disagreement all three levels are observationally identical. The
runtime controller must therefore escalate through reset -> fallback -> revision.
The full true-level x maximum-escalation confusion matrix, classification latency,
bad-plan steps, unnecessary fallback, and false revisions are primary outcomes.

PREDECLARED CLAIMS
------------------
P1. After L1, re-grounding the finite state restores nominal predictive accuracy
    faster and with lower integrated post-shock error than open-loop rollout.
P2. After L2, fallback plus agreement hysteresis returns to the original machine
    without structural revision at the balanced operating point.
P3. After L3, persistent frozen-baseline surprise reaches revision depth, while
    responsive/conservative hysteresis trades under-response against over-response.
P4. With equal transition budgets, active distinct (q,a) probes produce a more
    globally accurate revised table than passive on-policy counterexamples.
P5. A neural observer given the same post-shock observation is a required control:
    the symbolic claim is exactness and integrated recovery, not exclusive access
    to observation correction.

AUDIT GATES
-----------
1. The nominal machine must recover all transitions in the full run.
2. Per-(q,a) surprise mean and scale are estimated before shocks and frozen.
3. Every intervention is forced to produce a manifest first counterexample.
4. Recovery policies see only observation-derived state probabilities, actions,
   the frozen machine, and frozen baseline statistics—not true level or duration.
5. Active and passive revision receive the same transition budget.
6. The structural candidate is intentionally only a patched transition table.
   State-split/guard proposal search and autonomous ontology revision are not
   claimed by this experiment.

OUTPUTS
-------
config.json, results.json, recovery_metrics.csv, confusion_matrices.csv,
revision_trials.csv, crystallized_machine.npz, model checkpoint, summary.png,
and run_bundle.zip.

Set HOMEOSTATIC_V3_FAST_DEV_RUN=1 for a reduced execution-path test.
"""

from __future__ import annotations

import contextlib
import csv
import dataclasses
import json
import math
import os
import random
import shutil
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# =============================================================================
# 0. FROZEN CONFIGURATION
# =============================================================================

FAST_DEV_RUN = os.environ.get("HOMEOSTATIC_V3_FAST_DEV_RUN", "0") == "1"


@dataclass(frozen=True)
class Config:
    seed: int = 3307
    n_pos: int = 13
    v_max: int = 2
    n_actions: int = 3
    image_size: int = 16
    distractor_probability: float = 0.10

    # Nominal neural model and crystallization.
    hidden_size: int = 128
    action_embed_dim: int = 24
    train_seq_len: int = 14
    train_steps: int = 700
    train_batch_size: int = 768
    learning_rate: float = 2.0e-3
    initial_loss_weight: float = 0.35
    extraction_nuisance_samples: int = 96

    # Frozen nominal surprise calibration.
    baseline_samples_per_transition: int = 64
    surprise_sigma_floor: float = 0.02
    surprise_z_threshold: float = 3.0

    # Bird-strike episodes.
    episode_horizon: int = 48
    event_time_min: int = 8
    event_time_max: int = 12
    episodes_per_level: int = 512
    level2_min_duration: int = 2
    level2_max_duration: int = 5
    recovery_streak: int = 3
    recovery_window: int = 20

    # Hysteresis operating points: name, fallback consecutive surprises,
    # revision accumulated evidence, re-entry agreement streak, evidence decay.
    operating_points: Tuple[Tuple[str, int, float, int, float], ...] = (
        ("responsive", 2, 4.0, 2, 0.50),
        ("balanced", 2, 6.0, 3, 0.75),
        ("conservative", 3, 9.0, 4, 1.00),
    )

    # Structural revision: same observation budget, different acquisition policy.
    revision_trials: int = 256
    revision_budget: int = 32
    revision_eval_trajectory: int = 128

    use_amp: bool = True
    allow_tf32: bool = True
    output_dir: str = "outputs/04-homeostatic-symbolic-recovery-v3/homeostatic_recovery_v3_results"

    @property
    def n_vel(self) -> int:
        return 2 * self.v_max + 1

    @property
    def n_states(self) -> int:
        return self.n_pos * self.n_vel


cfg = Config()
if FAST_DEV_RUN:
    cfg = dataclasses.replace(
        cfg,
        hidden_size=48,
        train_seq_len=5,
        train_steps=int(os.environ.get("HOMEOSTATIC_V3_FAST_TRAIN_STEPS", "12")),
        train_batch_size=48,
        extraction_nuisance_samples=4,
        baseline_samples_per_transition=int(os.environ.get("HOMEOSTATIC_V3_FAST_BASELINE_SAMPLES", "3")),
        episode_horizon=18,
        event_time_min=4,
        event_time_max=6,
        episodes_per_level=int(os.environ.get("HOMEOSTATIC_V3_FAST_EPISODES", "32")),
        recovery_window=8,
        revision_trials=int(os.environ.get("HOMEOSTATIC_V3_FAST_REVISION_TRIALS", "24")),
        revision_budget=int(os.environ.get("HOMEOSTATIC_V3_FAST_REVISION_BUDGET", "10")),
        revision_eval_trajectory=int(os.environ.get("HOMEOSTATIC_V3_FAST_REVISION_EVAL", "24")),
        output_dir=str(Path.cwd() / "homeostatic_recovery_v3_smoke"),
    )


# =============================================================================
# 1. RUNTIME
# =============================================================================

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
is_cuda = device.type == "cuda"
if is_cuda:
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = cfg.allow_tf32
    torch.backends.cudnn.allow_tf32 = cfg.allow_tf32
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

amp_enabled = bool(cfg.use_amp and is_cuda)
amp_context = (
    (lambda: torch.autocast(device_type="cuda", dtype=torch.float16))
    if amp_enabled
    else contextlib.nullcontext
)


def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=amp_enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=amp_enabled)


out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

print("=" * 96)
print("HOMEOSTATIC SYMBOLIC RECOVERY V3 — reset, fallback, revision")
print("=" * 96)
print(f"device={device} | torch={torch.__version__} | AMP={amp_enabled} | fast_dev={FAST_DEV_RUN}")
if is_cuda:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU={props.name} | VRAM={props.total_memory / 2**30:.1f} GiB")
else:
    print("WARNING: full defaults are intended for a Colab T4.")


# =============================================================================
# 2. FINITE MECHANICS AND OBSERVATIONS
# =============================================================================

ACTION_VALUES = torch.tensor([-1, 0, 1], dtype=torch.long, device=device)


def encode_state(position: torch.Tensor, velocity: torch.Tensor) -> torch.Tensor:
    return position.long() * cfg.n_vel + (velocity.long() + cfg.v_max)


def decode_state(state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    position = torch.div(state.long(), cfg.n_vel, rounding_mode="floor")
    velocity = torch.remainder(state.long(), cfg.n_vel) - cfg.v_max
    return position, velocity


def physics_step(
    state: torch.Tensor,
    action_index: torch.Tensor,
    exogenous_force: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    position, velocity = decode_state(state)
    acceleration = ACTION_VALUES[action_index.long()]
    if exogenous_force is None:
        exogenous_force = torch.zeros_like(acceleration)
    velocity_next = torch.clamp(
        velocity + acceleration + exogenous_force.long(),
        -cfg.v_max,
        cfg.v_max,
    )
    position_next = position + velocity_next
    hit_left = position_next < 0
    position_next = torch.where(hit_left, -position_next, position_next)
    velocity_next = torch.where(hit_left, -velocity_next, velocity_next)
    hit_right = position_next >= cfg.n_pos
    position_next = torch.where(
        hit_right,
        2 * (cfg.n_pos - 1) - position_next,
        position_next,
    )
    velocity_next = torch.where(hit_right, -velocity_next, velocity_next)
    return encode_state(position_next, velocity_next)


def rollout_physics(
    initial_state: torch.Tensor,
    actions: torch.Tensor,
    forces: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    states = [initial_state]
    current = initial_state
    for t in range(actions.shape[1]):
        force_t = None if forces is None else forces[:, t]
        current = physics_step(current, actions[:, t], force_t)
        states.append(current)
    return torch.stack(states, dim=1)


def render_observation(state: torch.Tensor) -> torch.Tensor:
    state = state.reshape(-1)
    batch = state.shape[0]
    size = cfg.image_size
    obs = torch.zeros((batch, 3, size, size), dtype=torch.float32, device=device)
    position, velocity = decode_state(state)
    px = 1 + torch.round(
        position.float() * float(size - 3) / float(cfg.n_pos - 1)
    ).long()
    bi = torch.arange(batch, device=device)
    cy = size // 2
    obs[bi, 0, cy, px] = 1.0
    obs[bi, 0, cy - 1, px] = 0.75
    obs[bi, 0, cy + 1, px] = 0.75
    gauge_x = (size // 2 - cfg.v_max) + (velocity + cfg.v_max)
    obs[bi, 1, size - 2, gauge_x] = 1.0
    obs[bi, 1, size - 3, size // 2] = 0.35
    mask = torch.rand((batch, size, size), device=device) < cfg.distractor_probability
    obs[:, 2] = mask.float() * torch.rand((batch, size, size), device=device)
    return obs.contiguous(memory_format=torch.channels_last)


with torch.no_grad():
    _all_q = torch.arange(cfg.n_states, device=device).repeat_interleave(cfg.n_actions)
    _all_a = torch.arange(cfg.n_actions, device=device).repeat(cfg.n_states)
    TRUE_NOMINAL_TABLE = physics_step(_all_q, _all_a).reshape(cfg.n_states, cfg.n_actions)


# =============================================================================
# 3. NOMINAL RECURRENT WORLD MODEL AND MACHINE EXTRACTION
# =============================================================================

class NeuralWorldModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.SiLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.SiLU(inplace=True),
            nn.Conv2d(64, 96, 3, stride=2, padding=1),
            nn.SiLU(inplace=True),
        )
        side = math.ceil(cfg.image_size / 8)
        self.encoder_proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(96 * side * side, cfg.hidden_size),
            nn.Tanh(),
        )
        self.action_embedding = nn.Embedding(cfg.n_actions, cfg.action_embed_dim)
        self.gru = nn.GRU(cfg.action_embed_dim, cfg.hidden_size, batch_first=True)
        self.state_head = nn.Linear(cfg.hidden_size, cfg.n_states)

    def encode_hidden(self, observation: torch.Tensor) -> torch.Tensor:
        return self.encoder_proj(self.encoder_conv(observation))

    def initial_logits(self, observation: torch.Tensor) -> torch.Tensor:
        return self.state_head(self.encode_hidden(observation))

    def rollout_from_hidden(
        self,
        hidden: torch.Tensor,
        actions: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        output, final_hidden = self.gru(
            self.action_embedding(actions),
            hidden.unsqueeze(0),
        )
        return self.state_head(output), final_hidden.squeeze(0)

    def forward(
        self,
        observation: torch.Tensor,
        actions: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        hidden = self.encode_hidden(observation)
        future_logits, _ = self.rollout_from_hidden(hidden, actions)
        return self.state_head(hidden), future_logits


model = NeuralWorldModel().to(device)
if is_cuda:
    model.encoder_conv = model.encoder_conv.to(memory_format=torch.channels_last)
try:
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=1.0e-4,
        fused=is_cuda,
    )
except (TypeError, RuntimeError):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=1.0e-4,
    )
scaler = make_scaler()
train_history: List[Dict[str, float]] = []
generator = torch.Generator(device=device).manual_seed(cfg.seed + 1)

print(f"world-model parameters={sum(p.numel() for p in model.parameters()):,}")
model.train()
train_start = time.perf_counter()
for step in range(1, cfg.train_steps + 1):
    q0 = torch.randint(
        0,
        cfg.n_states,
        (cfg.train_batch_size,),
        generator=generator,
        device=device,
    )
    actions = torch.randint(
        0,
        cfg.n_actions,
        (cfg.train_batch_size, cfg.train_seq_len),
        generator=generator,
        device=device,
    )
    targets = rollout_physics(q0, actions)
    obs0 = render_observation(q0)
    optimizer.zero_grad(set_to_none=True)
    with amp_context():
        q0_logits, future_logits = model(obs0, actions)
        initial_loss = F.cross_entropy(q0_logits.float(), targets[:, 0])
        future_loss = F.cross_entropy(
            future_logits.float().reshape(-1, cfg.n_states),
            targets[:, 1:].reshape(-1),
        )
        loss = future_loss + cfg.initial_loss_weight * initial_loss
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    report_every = max(1, cfg.train_steps // 20)
    if step == 1 or step % report_every == 0 or step == cfg.train_steps:
        with torch.no_grad():
            q0_acc = (q0_logits.argmax(-1) == targets[:, 0]).float().mean().item()
            final_acc = (
                future_logits[:, -1].argmax(-1) == targets[:, -1]
            ).float().mean().item()
        train_history.append(
            {
                "step": float(step),
                "loss": float(loss.item()),
                "q0_accuracy": q0_acc,
                "train_horizon_accuracy": final_acc,
            }
        )
        print(
            f"train {step:4d}/{cfg.train_steps} | loss={loss.item():.4f} | "
            f"q0={100*q0_acc:5.1f}% | t{cfg.train_seq_len}={100*final_acc:5.1f}%"
        )
train_seconds = time.perf_counter() - train_start
model.eval()


@torch.inference_mode()
def extract_machine() -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    table = torch.empty((cfg.n_states, cfg.n_actions), dtype=torch.long, device=device)
    confidence = torch.empty_like(table, dtype=torch.float32)
    stability = torch.empty_like(table, dtype=torch.float32)
    for q_value in range(cfg.n_states):
        q_batch = torch.full(
            (cfg.extraction_nuisance_samples,),
            q_value,
            dtype=torch.long,
            device=device,
        )
        obs = render_observation(q_batch)
        hidden = model.encode_hidden(obs)
        for action in range(cfg.n_actions):
            action_batch = torch.full(
                (cfg.extraction_nuisance_samples, 1),
                action,
                dtype=torch.long,
                device=device,
            )
            logits, _ = model.rollout_from_hidden(hidden, action_batch)
            probabilities = logits[:, 0].float().softmax(-1)
            mean_probability = probabilities.mean(0)
            successor = mean_probability.argmax()
            table[q_value, action] = successor
            confidence[q_value, action] = mean_probability[successor]
            stability[q_value, action] = (
                probabilities.argmax(-1) == successor
            ).float().mean()
    return table, confidence, stability


MACHINE_TABLE, MACHINE_CONFIDENCE, MACHINE_STABILITY = extract_machine()
machine_accuracy = (MACHINE_TABLE == TRUE_NOMINAL_TABLE).float().mean().item()
print("\nCRYSTALLIZATION AUDIT")
print("-" * 96)
print(f"transition accuracy={100*machine_accuracy:.2f}%")
print(
    f"confidence mean/min={MACHINE_CONFIDENCE.mean().item():.4f}/"
    f"{MACHINE_CONFIDENCE.min().item():.4f} | "
    f"minimum nuisance stability={MACHINE_STABILITY.min().item():.4f}"
)
if not FAST_DEV_RUN and machine_accuracy < 1.0:
    raise RuntimeError("V3 audit failed: nominal crystallized table is not exact.")

# Downstream smoke tests use the exact table because the fast path intentionally
# undertrains the neural model. Full V3 always uses the extracted table.
RUNTIME_TABLE = (
    TRUE_NOMINAL_TABLE
    if FAST_DEV_RUN and machine_accuracy < 1.0
    else MACHINE_TABLE
)


# =============================================================================
# 4. FROZEN PER-(q,a) NOMINAL SURPRISE BASELINE
# =============================================================================

@torch.inference_mode()
def encode_probabilities(states: torch.Tensor, chunk: int = 8192) -> torch.Tensor:
    outputs = []
    flat = states.reshape(-1)
    for start in range(0, flat.shape[0], chunk):
        obs = render_observation(flat[start : start + chunk])
        with amp_context():
            logits = model.initial_logits(obs)
        outputs.append(logits.float().softmax(-1))
    return torch.cat(outputs, dim=0)


@torch.inference_mode()
def calibrate_surprise() -> Tuple[torch.Tensor, torch.Tensor]:
    means = torch.empty((cfg.n_states, cfg.n_actions), device=device)
    scales = torch.empty_like(means)
    for q in range(cfg.n_states):
        for action in range(cfg.n_actions):
            q_batch = torch.full(
                (cfg.baseline_samples_per_transition,),
                q,
                dtype=torch.long,
                device=device,
            )
            a_batch = torch.full_like(q_batch, action)
            successor = RUNTIME_TABLE[q_batch, a_batch]
            probs = encode_probabilities(successor)
            predicted = RUNTIME_TABLE[q, action]
            nll = -torch.log(probs[:, predicted].clamp_min(1.0e-8))
            means[q, action] = nll.mean()
            scales[q, action] = nll.std(unbiased=False).clamp_min(cfg.surprise_sigma_floor)
    return means, scales


BASELINE_MEAN, BASELINE_SCALE = calibrate_surprise()
frozen_baseline_checksum = float(BASELINE_MEAN.sum().item() + BASELINE_SCALE.sum().item())
print("\nFROZEN SURPRISE BASELINE")
print("-" * 96)
print(
    f"mean NLL={BASELINE_MEAN.mean().item():.5f} | "
    f"mean scale={BASELINE_SCALE.mean().item():.5f} | checksum={frozen_baseline_checksum:.6f}"
)


# =============================================================================
# 5. THREE-LEVEL INTERVENTION EPISODES
# =============================================================================

def generate_level_episodes(level: int, count: int, seed: int) -> Dict[str, torch.Tensor]:
    if level not in {1, 2, 3}:
        raise ValueError("level must be 1, 2, or 3")
    g = torch.Generator(device=device).manual_seed(seed)
    q0 = torch.randint(0, cfg.n_states, (count,), generator=g, device=device)
    actions = torch.randint(
        0,
        cfg.n_actions,
        (count, cfg.episode_horizon),
        generator=g,
        device=device,
    )
    event_time = torch.randint(
        cfg.event_time_min,
        cfg.event_time_max + 1,
        (count,),
        generator=g,
        device=device,
    )

    # Before intervention, actual and nominal are identical. Resample only the
    # event action/direction until the first forced transition is manifest.
    nominal = rollout_physics(q0, actions)
    row = torch.arange(count, device=device)
    q_event = nominal[row, event_time]
    a_event = actions[row, event_time]
    plus = torch.ones(count, dtype=torch.long, device=device)
    minus = -plus
    nominal_next = physics_step(q_event, a_event)
    plus_next = physics_step(q_event, a_event, plus)
    minus_next = physics_step(q_event, a_event, minus)
    plus_changes = plus_next != nominal_next
    minus_changes = minus_next != nominal_next
    direction = torch.where(plus_changes, plus, minus)
    impossible = ~(plus_changes | minus_changes)
    attempts = 0
    while bool(impossible.any()):
        ids = torch.nonzero(impossible, as_tuple=False).squeeze(1)
        actions[ids, event_time[ids]] = torch.randint(
            0,
            cfg.n_actions,
            (ids.numel(),),
            generator=g,
            device=device,
        )
        nominal = rollout_physics(q0, actions)
        q_event = nominal[row, event_time]
        a_event = actions[row, event_time]
        nominal_next = physics_step(q_event, a_event)
        plus_next = physics_step(q_event, a_event, plus)
        minus_next = physics_step(q_event, a_event, minus)
        plus_changes = plus_next != nominal_next
        minus_changes = minus_next != nominal_next
        direction = torch.where(plus_changes, plus, minus)
        impossible = ~(plus_changes | minus_changes)
        attempts += 1
        if attempts > 20:
            raise RuntimeError("Could not force manifest intervention transitions.")

    if level == 1:
        duration = torch.ones(count, dtype=torch.long, device=device)
    elif level == 2:
        duration = torch.randint(
            cfg.level2_min_duration,
            cfg.level2_max_duration + 1,
            (count,),
            generator=g,
            device=device,
        )
    else:
        duration = cfg.episode_horizon - event_time

    times = torch.arange(cfg.episode_horizon, device=device)[None, :]
    active = (
        (times >= event_time[:, None])
        & (times < (event_time + duration)[:, None])
    )
    forces = active.long() * direction[:, None]
    actual = rollout_physics(q0, actions, forces)
    nominal = rollout_physics(q0, actions)
    first_manifest = actual[row, event_time + 1] != nominal[row, event_time + 1]
    if not bool(first_manifest.all()):
        raise RuntimeError("Manifest-first-counterexample audit failed.")
    return {
        "level": torch.full((count,), level, dtype=torch.long),
        "q0": q0.cpu(),
        "actions": actions.cpu(),
        "states": actual.cpu(),
        "nominal_states": nominal.cpu(),
        "forces": forces.cpu(),
        "event_time": event_time.cpu(),
        "duration": duration.cpu(),
        "direction": direction.cpu(),
    }


episodes_by_level = {
    level: generate_level_episodes(
        level,
        cfg.episodes_per_level,
        cfg.seed + 10 + level,
    )
    for level in (1, 2, 3)
}


@torch.inference_mode()
def analyze_episode_predictions(data: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    states = data["states"].to(device)
    actions = data["actions"].to(device)
    batch, state_times = states.shape
    horizon = state_times - 1
    probabilities = encode_probabilities(states).reshape(batch, state_times, cfg.n_states)
    q_hat = probabilities.argmax(dim=-1)

    current_hat = q_hat[:, :-1]
    next_hat = q_hat[:, 1:]
    shadow_prediction = RUNTIME_TABLE[current_hat, actions]
    assigned = probabilities[:, 1:].gather(
        2,
        shadow_prediction[:, :, None],
    ).squeeze(2)
    nll = -torch.log(assigned.clamp_min(1.0e-8))
    mu = BASELINE_MEAN[current_hat, actions]
    scale = BASELINE_SCALE[current_hat, actions]
    z = ((nll - mu) / scale).clamp(min=0.0, max=50.0)
    mismatch = shadow_prediction != next_hat

    # Open-loop machine.
    machine_open = []
    q_machine = q_hat[:, 0]
    for t in range(horizon):
        q_machine = RUNTIME_TABLE[q_machine, actions[:, t]]
        machine_open.append(q_machine)
    machine_open = torch.stack(machine_open, dim=1)

    # Open-loop neural rollout.
    obs0 = render_observation(states[:, 0])
    with amp_context():
        _, neural_open_logits = model(obs0, actions)
    neural_open = neural_open_logits.float().argmax(-1)

    # Neural observation correction: re-encode at every state, then make exactly
    # one prediction. This is the fair control for symbolic re-grounding.
    grounded_predictions: List[torch.Tensor] = []
    chunk = 8192
    flat_current = states[:, :-1].reshape(-1)
    flat_actions = actions.reshape(-1)
    for start in range(0, flat_current.shape[0], chunk):
        obs = render_observation(flat_current[start : start + chunk])
        with amp_context():
            hidden = model.encode_hidden(obs)
            logits, _ = model.rollout_from_hidden(
                hidden,
                flat_actions[start : start + chunk, None],
            )
        grounded_predictions.append(logits[:, 0].float().argmax(-1))
    neural_grounded = torch.cat(grounded_predictions).reshape(batch, horizon)

    truth_next = states[:, 1:]
    return {
        "q_hat": q_hat.cpu(),
        "z": z.cpu(),
        "mismatch": mismatch.cpu(),
        "machine_open_correct": (machine_open == truth_next).cpu(),
        "machine_grounded_correct": (shadow_prediction == truth_next).cpu(),
        "neural_open_correct": (neural_open == truth_next).cpu(),
        "neural_grounded_correct": (neural_grounded == truth_next).cpu(),
    }


analysis_by_level = {
    level: analyze_episode_predictions(data)
    for level, data in episodes_by_level.items()
}


def recovery_time(
    correct: np.ndarray,
    intervention_end: int,
    streak: int,
) -> float:
    for t in range(intervention_end, correct.shape[0] - streak + 1):
        if bool(correct[t : t + streak].all()):
            return float(t - intervention_end)
    return float("nan")


recovery_rows: List[Dict[str, float]] = []
systems = {
    "machine-open-loop": "machine_open_correct",
    "machine-re-grounded": "machine_grounded_correct",
    "neural-open-loop": "neural_open_correct",
    "neural-re-grounded": "neural_grounded_correct",
}
for level in (1, 2, 3):
    data = episodes_by_level[level]
    analysis = analysis_by_level[level]
    events = data["event_time"].numpy()
    durations = data["duration"].numpy()
    for system_name, key in systems.items():
        correct = analysis[key].numpy()
        integrated_errors: List[float] = []
        recovery_times: List[float] = []
        for i in range(correct.shape[0]):
            start = int(events[i])
            end_window = min(correct.shape[1], start + cfg.recovery_window)
            integrated_errors.append(float((~correct[i, start:end_window]).sum()))
            if level < 3:
                intervention_end = int(events[i] + durations[i])
                rt = recovery_time(correct[i], intervention_end, cfg.recovery_streak)
                if not np.isnan(rt):
                    recovery_times.append(rt)
        recovery_rows.append(
            {
                "true_level": float(level),
                "system": system_name,
                "pre_event_accuracy": float(
                    np.mean([
                        correct[i, : int(events[i])].mean()
                        for i in range(correct.shape[0])
                    ])
                ),
                "mean_integrated_postshock_error": float(np.mean(integrated_errors)),
                "median_recovery_time": (
                    float(np.median(recovery_times)) if recovery_times else float("nan")
                ),
                "recovery_observed_fraction": float(
                    len(recovery_times) / correct.shape[0]
                ) if level < 3 else float("nan"),
            }
        )

print("\nPOST-SHOCK PREDICTION RECOVERY")
print("-" * 96)
print(" level | system                 | integrated error | median recovery | recovered")
for row in recovery_rows:
    print(
        f" {int(row['true_level']):5d} | {row['system']:22s} | "
        f"{row['mean_integrated_postshock_error']:16.3f} | "
        f"{row['median_recovery_time']:15.3f} | "
        f"{100*row['recovery_observed_fraction']:8.2f}%"
        if not np.isnan(row["recovery_observed_fraction"])
        else
        f" {int(row['true_level']):5d} | {row['system']:22s} | "
        f"{row['mean_integrated_postshock_error']:16.3f} | {'n/a':>15s} | {'n/a':>9s}"
    )


# =============================================================================
# 6. LOSS-GATED ESCALATION AND HYSTERESIS SWEEP
# =============================================================================

def simulate_escalation(
    z: np.ndarray,
    mismatch: np.ndarray,
    active_force: np.ndarray,
    event_time: int,
    true_level: int,
    fallback_after: int,
    revision_evidence: float,
    reentry_after: int,
    evidence_decay: float,
) -> Dict[str, float]:
    mode = 0  # 0 machine, 1 fallback, 2 revision
    maximum_depth = 0
    consecutive_surprise = 0
    agreements = 0
    evidence = 0.0
    bad_plan_steps = 0
    unnecessary_fallback_steps = 0
    false_revision = 0
    pre_event_revision = 0
    reset_time: Optional[int] = None
    fallback_time: Optional[int] = None
    revision_time: Optional[int] = None

    for t in range(z.shape[0]):
        mode_before = mode
        surprising = bool(
            mismatch[t] or z[t] > cfg.surprise_z_threshold
        )
        if surprising:
            if reset_time is None and t >= event_time:
                reset_time = t
            maximum_depth = max(maximum_depth, 1)
            consecutive_surprise += 1
            agreements = 0
            evidence += 1.0
            if mode_before == 0:
                bad_plan_steps += 1
            if mode < 1 and consecutive_surprise >= fallback_after:
                mode = 1
                maximum_depth = max(maximum_depth, 2)
                if fallback_time is None and t >= event_time:
                    fallback_time = t
            if mode < 2 and evidence >= revision_evidence:
                mode = 2
                maximum_depth = 3
                if revision_time is None and t >= event_time:
                    revision_time = t
                if t < event_time:
                    pre_event_revision = 1
                if true_level < 3:
                    false_revision = 1
        else:
            consecutive_surprise = 0
            agreements += 1
            evidence = max(0.0, evidence - evidence_decay)
            if mode == 1 and agreements >= reentry_after:
                mode = 0
                evidence = 0.0
                agreements = 0

        if mode_before == 1 and not bool(active_force[t]):
            unnecessary_fallback_steps += 1

    expected_time = (
        reset_time if true_level == 1
        else fallback_time if true_level == 2
        else revision_time
    )
    classification_latency = (
        float(expected_time - event_time) if expected_time is not None else float("nan")
    )
    return {
        "predicted_depth": float(maximum_depth),
        "bad_plan_steps": float(bad_plan_steps),
        "unnecessary_fallback_steps": float(unnecessary_fallback_steps),
        "false_revision": float(false_revision),
        "pre_event_revision": float(pre_event_revision),
        "classification_latency": classification_latency,
        "reached_expected_depth": float(maximum_depth >= true_level),
    }


confusion_rows: List[Dict[str, object]] = []
operating_summary: Dict[str, Dict[str, float]] = {}
for op_name, fallback_after, revision_evidence, reentry_after, evidence_decay in cfg.operating_points:
    all_episode_results: List[Tuple[int, Dict[str, float]]] = []
    matrix = np.zeros((3, 3), dtype=int)
    for level in (1, 2, 3):
        z = analysis_by_level[level]["z"].numpy()
        mismatch = analysis_by_level[level]["mismatch"].numpy()
        active = episodes_by_level[level]["forces"].numpy() != 0
        events = episodes_by_level[level]["event_time"].numpy()
        for i in range(z.shape[0]):
            result = simulate_escalation(
                z[i],
                mismatch[i],
                active[i],
                int(events[i]),
                level,
                fallback_after,
                revision_evidence,
                reentry_after,
                evidence_decay,
            )
            all_episode_results.append((level, result))
            depth = int(result["predicted_depth"])
            if depth >= 1:
                matrix[level - 1, min(depth, 3) - 1] += 1

    for true_level in range(1, 4):
        for predicted_depth in range(1, 4):
            confusion_rows.append(
                {
                    "operating_point": op_name,
                    "true_level": true_level,
                    "maximum_escalation_depth": predicted_depth,
                    "count": int(matrix[true_level - 1, predicted_depth - 1]),
                }
            )
    latencies = [
        r["classification_latency"]
        for _, r in all_episode_results
        if not np.isnan(r["classification_latency"])
    ]
    operating_summary[op_name] = {
        "classification_accuracy": float(np.trace(matrix) / matrix.sum()),
        "mean_bad_plan_steps": float(np.mean([r["bad_plan_steps"] for _, r in all_episode_results])),
        "mean_unnecessary_fallback_steps": float(
            np.mean([r["unnecessary_fallback_steps"] for _, r in all_episode_results])
        ),
        "false_revision_rate_L1_L2": float(
            np.mean([r["false_revision"] for level, r in all_episode_results if level < 3])
        ),
        "pre_event_revision_rate": float(
            np.mean([r["pre_event_revision"] for _, r in all_episode_results])
        ),
        "L3_revision_rate": float(
            np.mean([r["predicted_depth"] >= 3 for level, r in all_episode_results if level == 3])
        ),
        "median_classification_latency": float(np.median(latencies)) if latencies else None,
    }

print("\nESCALATION CONFUSION MATRICES")
print("-" * 96)
for op_name in operating_summary:
    print(f"{op_name}:")
    rows = [r for r in confusion_rows if r["operating_point"] == op_name]
    matrix = np.zeros((3, 3), dtype=int)
    for row in rows:
        matrix[int(row["true_level"]) - 1, int(row["maximum_escalation_depth"]) - 1] = int(row["count"])
    print(matrix)
    print(operating_summary[op_name])


# =============================================================================
# 7. STRUCTURAL REVISION: PASSIVE BUFFER VS ACTIVE RE-EXAMINATION
# =============================================================================

def true_regime_table(wind_direction: int) -> torch.Tensor:
    q = torch.arange(cfg.n_states, device=device).repeat_interleave(cfg.n_actions)
    a = torch.arange(cfg.n_actions, device=device).repeat(cfg.n_states)
    wind = torch.full_like(q, wind_direction)
    return physics_step(q, a, wind).reshape(cfg.n_states, cfg.n_actions)


def patch_table_from_transitions(
    base_table: np.ndarray,
    q_values: np.ndarray,
    actions: np.ndarray,
    successors: np.ndarray,
) -> Tuple[np.ndarray, int]:
    table = base_table.copy()
    seen = set()
    for q, action, successor in zip(q_values, actions, successors):
        table[int(q), int(action)] = int(successor)
        seen.add((int(q), int(action)))
    return table, len(seen)


revision_rng = np.random.default_rng(cfg.seed + 40)
nominal_np = RUNTIME_TABLE.detach().cpu().numpy()
revision_rows: List[Dict[str, float]] = []
for trial in range(cfg.revision_trials):
    wind = int(revision_rng.choice([-1, 1]))
    true_table = true_regime_table(wind).detach().cpu().numpy()

    # Passive: one on-policy trajectory, duplicates allowed.
    passive_q = np.empty(cfg.revision_budget, dtype=np.int64)
    passive_a = revision_rng.integers(0, cfg.n_actions, size=cfg.revision_budget)
    passive_next = np.empty(cfg.revision_budget, dtype=np.int64)
    q_current = int(revision_rng.integers(0, cfg.n_states))
    for t in range(cfg.revision_budget):
        passive_q[t] = q_current
        passive_next[t] = true_table[q_current, passive_a[t]]
        q_current = int(passive_next[t])
    passive_table, passive_coverage = patch_table_from_transitions(
        nominal_np,
        passive_q,
        passive_a,
        passive_next,
    )

    # Active: equal budget, distinct state-action probes.
    all_pairs = np.arange(cfg.n_states * cfg.n_actions)
    active_pairs = revision_rng.choice(
        all_pairs,
        size=min(cfg.revision_budget, all_pairs.size),
        replace=False,
    )
    active_q = active_pairs // cfg.n_actions
    active_a = active_pairs % cfg.n_actions
    active_next = true_table[active_q, active_a]
    active_table, active_coverage = patch_table_from_transitions(
        nominal_np,
        active_q,
        active_a,
        active_next,
    )

    # Independent on-policy evaluation trajectory under the changed regime.
    eval_q = np.empty(cfg.revision_eval_trajectory, dtype=np.int64)
    eval_a = revision_rng.integers(0, cfg.n_actions, size=cfg.revision_eval_trajectory)
    eval_next = np.empty(cfg.revision_eval_trajectory, dtype=np.int64)
    q_eval = int(revision_rng.integers(0, cfg.n_states))
    for t in range(cfg.revision_eval_trajectory):
        eval_q[t] = q_eval
        eval_next[t] = true_table[q_eval, eval_a[t]]
        q_eval = int(eval_next[t])

    nominal_global = float((nominal_np == true_table).mean())
    for method, table, coverage in [
        ("passive-on-policy", passive_table, passive_coverage),
        ("active-distinct-probes", active_table, active_coverage),
    ]:
        revision_rows.append(
            {
                "trial": float(trial),
                "wind_direction": float(wind),
                "method": method,
                "budget": float(cfg.revision_budget),
                "distinct_transition_coverage": float(coverage),
                "nominal_table_global_accuracy": nominal_global,
                "revised_global_accuracy": float((table == true_table).mean()),
                "revised_onpolicy_accuracy": float(
                    (table[eval_q, eval_a] == eval_next).mean()
                ),
            }
        )

revision_summary: Dict[str, Dict[str, float]] = {}
for method in ["passive-on-policy", "active-distinct-probes"]:
    rows = [r for r in revision_rows if r["method"] == method]
    revision_summary[method] = {
        "mean_distinct_coverage": float(np.mean([r["distinct_transition_coverage"] for r in rows])),
        "mean_global_accuracy": float(np.mean([r["revised_global_accuracy"] for r in rows])),
        "mean_onpolicy_accuracy": float(np.mean([r["revised_onpolicy_accuracy"] for r in rows])),
    }

print("\nSTRUCTURAL REVISION CONTRAST")
print("-" * 96)
for method, summary in revision_summary.items():
    print(method, summary)


# =============================================================================
# 8. SAVE ARTIFACTS AND SUMMARY FIGURE
# =============================================================================

config_json = asdict(cfg)
config_json["operating_points"] = [list(x) for x in cfg.operating_points]
results = {
    "interpretation_boundary": (
        "The finite state and action alphabet are supplied. V3 tests runtime state reset, "
        "temporary fallback, escalation, and equal-budget transition-table patching. It does "
        "not solve autonomous guard/state-split proposal or formally verify revised machines."
    ),
    "runtime": {
        "device": str(device),
        "gpu": torch.cuda.get_device_name(0) if is_cuda else None,
        "torch_version": torch.__version__,
        "amp": amp_enabled,
        "train_seconds": train_seconds,
    },
    "crystallization": {
        "transition_accuracy": machine_accuracy,
        "mean_confidence": float(MACHINE_CONFIDENCE.mean().item()),
        "minimum_confidence": float(MACHINE_CONFIDENCE.min().item()),
        "minimum_nuisance_stability": float(MACHINE_STABILITY.min().item()),
    },
    "frozen_surprise_baseline": {
        "mean_nll": float(BASELINE_MEAN.mean().item()),
        "mean_scale": float(BASELINE_SCALE.mean().item()),
        "checksum": frozen_baseline_checksum,
    },
    "recovery": recovery_rows,
    "operating_points": operating_summary,
    "confusion_matrices": confusion_rows,
    "revision": revision_summary,
}

with (out_dir / "config.json").open("w", encoding="utf-8") as f:
    json.dump(config_json, f, indent=2)
with (out_dir / "results.json").open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
with (out_dir / "recovery_metrics.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(recovery_rows[0].keys()))
    writer.writeheader()
    writer.writerows(recovery_rows)
with (out_dir / "confusion_matrices.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(confusion_rows[0].keys()))
    writer.writeheader()
    writer.writerows(confusion_rows)
with (out_dir / "revision_trials.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(revision_rows[0].keys()))
    writer.writeheader()
    writer.writerows(revision_rows)

np.savez_compressed(
    out_dir / "crystallized_machine.npz",
    transition_table=MACHINE_TABLE.detach().cpu().numpy(),
    confidence=MACHINE_CONFIDENCE.detach().cpu().numpy(),
    stability=MACHINE_STABILITY.detach().cpu().numpy(),
    baseline_mean=BASELINE_MEAN.detach().cpu().numpy(),
    baseline_scale=BASELINE_SCALE.detach().cpu().numpy(),
)
torch.save(
    {"state_dict": model.state_dict(), "config": config_json},
    out_dir / "nominal_world_model.pt",
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

# L1 integrated error: exactness of symbolic reset versus neural correction.
l1_rows = [r for r in recovery_rows if int(r["true_level"]) == 1]
axes[0, 0].bar(
    [r["system"].replace("-", "\n") for r in l1_rows],
    [r["mean_integrated_postshock_error"] for r in l1_rows],
    color=["#E45756", "#54A24B", "#F58518", "#4C78A8"],
)
axes[0, 0].set_title("L1 impulse: integrated post-shock error")
axes[0, 0].set_ylabel("incorrect predicted transitions")
axes[0, 0].grid(axis="y", alpha=0.25)

# Balanced confusion matrix.
balanced_rows = [r for r in confusion_rows if r["operating_point"] == "balanced"]
balanced_matrix = np.zeros((3, 3), dtype=int)
for row in balanced_rows:
    balanced_matrix[int(row["true_level"]) - 1, int(row["maximum_escalation_depth"]) - 1] = int(row["count"])
im = axes[0, 1].imshow(balanced_matrix, cmap="Blues")
for i in range(3):
    for j in range(3):
        axes[0, 1].text(j, i, str(balanced_matrix[i, j]), ha="center", va="center")
axes[0, 1].set_xticks(range(3), ["reset", "fallback", "revision"])
axes[0, 1].set_yticks(range(3), ["L1 state", "L2 transient", "L3 permanent"])
axes[0, 1].set_title("Balanced policy: true level vs max escalation")
fig.colorbar(im, ax=axes[0, 1], fraction=0.046)

# Hysteresis trade curve.
op_names = list(operating_summary.keys())
axes[1, 0].scatter(
    [operating_summary[n]["mean_bad_plan_steps"] for n in op_names],
    [
        operating_summary[n]["mean_unnecessary_fallback_steps"]
        + 5.0 * operating_summary[n]["false_revision_rate_L1_L2"]
        for n in op_names
    ],
    s=90,
    color=["#E45756", "#54A24B", "#4C78A8"],
)
for name in op_names:
    axes[1, 0].annotate(
        name,
        (
            operating_summary[name]["mean_bad_plan_steps"],
            operating_summary[name]["mean_unnecessary_fallback_steps"]
            + 5.0 * operating_summary[name]["false_revision_rate_L1_L2"],
        ),
        xytext=(5, 5),
        textcoords="offset points",
    )
axes[1, 0].set_xlabel("mean bad-plan steps (under-response)")
axes[1, 0].set_ylabel("fallback + weighted false-revision cost")
axes[1, 0].set_title("Hysteresis operating trade-off")
axes[1, 0].grid(alpha=0.25)

# Active versus passive structural re-examination.
methods = ["passive-on-policy", "active-distinct-probes"]
axes[1, 1].bar(
    methods,
    [revision_summary[m]["mean_global_accuracy"] for m in methods],
    color=["#F58518", "#54A24B"],
)
axes[1, 1].set_ylim(0, 1.02)
axes[1, 1].set_title(f"L3 table revision with equal budget={cfg.revision_budget}")
axes[1, 1].set_ylabel("global changed-regime transition accuracy")
axes[1, 1].grid(axis="y", alpha=0.25)

fig.suptitle(
    "Homeostatic symbolic recovery after bird-strike interventions\n"
    f"nominal machine={100*machine_accuracy:.1f}% | "
    f"balanced classification={100*operating_summary['balanced']['classification_accuracy']:.1f}% | "
    f"L3 revision={100*operating_summary['balanced']['L3_revision_rate']:.1f}%",
    fontsize=14,
)
fig.savefig(out_dir / "summary.png", dpi=180)
plt.show()

archive_path = out_dir / "run_bundle.zip"
if archive_path.exists():
    archive_path.unlink()
temporary_base = out_dir.parent / f".{out_dir.name}_run_bundle_tmp"
temporary_zip = temporary_base.with_suffix(".zip")
if temporary_zip.exists():
    temporary_zip.unlink()
made_archive = shutil.make_archive(
    str(temporary_base),
    "zip",
    root_dir=out_dir,
    base_dir=".",
)
shutil.move(made_archive, archive_path)

print("\n" + "=" * 96)
print("FINAL V3 SUMMARY")
print("=" * 96)
print(f"nominal machine transition accuracy: {100*machine_accuracy:.2f}%")
for row in l1_rows:
    print(
        f"L1 {row['system']}: integrated error="
        f"{row['mean_integrated_postshock_error']:.3f}, "
        f"median recovery={row['median_recovery_time']:.3f}"
    )
for name, summary in operating_summary.items():
    print(
        f"{name}: classification={100*summary['classification_accuracy']:.2f}% | "
        f"bad steps={summary['mean_bad_plan_steps']:.3f} | "
        f"false revision L1/L2={100*summary['false_revision_rate_L1_L2']:.2f}% | "
        f"L3 revision={100*summary['L3_revision_rate']:.2f}%"
    )
for method, summary in revision_summary.items():
    print(
        f"{method}: coverage={summary['mean_distinct_coverage']:.2f} | "
        f"global accuracy={100*summary['mean_global_accuracy']:.2f}% | "
        f"on-policy accuracy={100*summary['mean_onpolicy_accuracy']:.2f}%"
    )
print(f"artifacts: {out_dir}")
print(f"bundle: {archive_path}")


## Experiment 5: Runtime MDL refinement V4

Original cell `4`.

**Audit note.** The cleaned copy guards the all-NaN summary case that produced a mean-of-empty-slice warning.


In [ ]:
"""
RUNTIME REFINEMENT V4 — RE-GROUNDING + MDL-GATED STRUCTURAL ADOPTION

Standalone deterministic experiment for a Colab or local Python cell. It compares:

1. Naive disagreement refinement: every observed counterexample is made exact by
   adding an exception/clone state and patching the responsible transition.
2. Gated refinement: first re-ground to the observed state, retain a rolling
   transition buffer, and adopt a global dynamics rule only when it is MDL-positive
   and passes independent targeted confirmation queries.

Predeclared predictions:
P1. Under isolated one-step shocks, naive model size grows while the gated model
    retains the 65-state nominal machine.
P2. On clean nominal planning after those shocks, gated planning exceeds naive
    planning because the gated learner refuses to encode transient exceptions.
P3. After a permanent force change, gated refinement adopts one compressed rule
    and restores exact transition accuracy; naive refinement continues accumulating
    exception states.

Important scope boundary: this is an active finite-machine refinement experiment,
not a neural-network training experiment and not a full implementation of L*.
"""

from __future__ import annotations

import csv
import json
import math
import os
import random
import shutil
import time
from collections import deque
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Deque, Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np


FAST_DEV_RUN = os.environ.get("RUNTIME_MDL_V4_FAST_DEV_RUN", "0") == "1"


@dataclass(frozen=True)
class Config:
    seed: int = 4407
    n_pos: int = 13
    v_max: int = 2
    n_actions: int = 3
    horizon: int = 800
    trials: int = 128
    transient_shocks: int = 32
    mixed_transient_shocks: int = 16
    permanent_onset: int = 400
    min_shock_gap: int = 8
    buffer_size: int = 64
    min_buffer: int = 16
    min_candidate_errors: int = 3
    error_code_bits: float = math.log2(65)
    structural_rule_bits: float = 12.0
    mdl_min_gain_bits: float = 12.0
    confirmation_queries: int = 8
    confirmation_required_fraction: float = 1.0
    planning_tasks: int = 64
    planning_max_depth: int = 24
    checkpoints: Tuple[int, ...] = tuple(range(0, 801, 40))
    output_dir: str = "outputs/05-runtime-mdl-refinement-v4/runtime_mdl_refinement_v4_results"

    @property
    def n_vel(self) -> int:
        return 2 * self.v_max + 1

    @property
    def n_states(self) -> int:
        return self.n_pos * self.n_vel


cfg = Config()
if FAST_DEV_RUN:
    cfg = Config(
        horizon=160,
        trials=12,
        transient_shocks=8,
        mixed_transient_shocks=4,
        permanent_onset=80,
        buffer_size=32,
        min_buffer=12,
        confirmation_queries=6,
        planning_tasks=20,
        planning_max_depth=18,
        checkpoints=tuple(range(0, 161, 20)),
        output_dir=str(Path.cwd() / "runtime_mdl_refinement_v4_smoke"),
    )

random.seed(cfg.seed)
np.random.seed(cfg.seed)
rng_master = np.random.default_rng(cfg.seed)
ACTION_VALUES = np.asarray([-1, 0, 1], dtype=np.int64)


def encode_state(position: int, velocity: int) -> int:
    return int(position * cfg.n_vel + (velocity + cfg.v_max))


def decode_state(state: int) -> Tuple[int, int]:
    return int(state // cfg.n_vel), int(state % cfg.n_vel - cfg.v_max)


def physics_step(state: int, action: int, force: int = 0) -> int:
    position, velocity = decode_state(state)
    velocity_next = int(np.clip(velocity + int(ACTION_VALUES[action]) + force, -cfg.v_max, cfg.v_max))
    position_next = position + velocity_next
    if position_next < 0:
        position_next = -position_next
        velocity_next = -velocity_next
    if position_next >= cfg.n_pos:
        position_next = 2 * (cfg.n_pos - 1) - position_next
        velocity_next = -velocity_next
    return encode_state(position_next, velocity_next)


def make_table(force: int) -> np.ndarray:
    table = np.empty((cfg.n_states, cfg.n_actions), dtype=np.int64)
    for q in range(cfg.n_states):
        for a in range(cfg.n_actions):
            table[q, a] = physics_step(q, a, force)
    return table


TABLES = {force: make_table(force) for force in (-1, 0, 1)}


def choose_manifest_force(q: int, a: int, generator: np.random.Generator) -> int:
    candidates = [d for d in (-1, 1) if TABLES[d][q, a] != TABLES[0][q, a]]
    if not candidates:
        return 0
    return int(generator.choice(candidates))


def separated_times(
    count: int,
    low: int,
    high: int,
    gap: int,
    generator: np.random.Generator,
) -> set[int]:
    available = list(range(low, high))
    generator.shuffle(available)
    selected: List[int] = []
    for t in available:
        if all(abs(t - prior) >= gap for prior in selected):
            selected.append(t)
            if len(selected) == count:
                break
    if len(selected) < count:
        raise RuntimeError("Could not schedule separated shocks.")
    return set(selected)


class NaiveExceptionMachine:
    """Exact counterexample patching by monotonic exception-state growth."""

    def __init__(self) -> None:
        self.labels: List[int] = list(range(cfg.n_states))
        self.transitions: List[List[int]] = TABLES[0].astype(int).tolist()
        self.current = 0
        self.refinements = 0
        self.disagreements = 0

    @property
    def state_count(self) -> int:
        return len(self.labels)

    def reset_observed_state(self, q: int) -> None:
        self.current = int(q)

    def step(self, action: int, observed_next: int) -> bool:
        source = self.current
        predicted_internal = self.transitions[source][action]
        predicted_label = self.labels[predicted_internal]
        disagreement = predicted_label != observed_next
        if disagreement:
            self.disagreements += 1
            clone = len(self.labels)
            self.labels.append(int(observed_next))
            # The exception state inherits nominal dynamics for its observed label.
            self.transitions.append(TABLES[0][observed_next].astype(int).tolist())
            self.transitions[source][action] = clone
            self.current = clone
            self.refinements += 1
        else:
            self.current = predicted_internal
        return disagreement

    def predicted_label(self, internal: int, action: int) -> Tuple[int, int]:
        nxt = self.transitions[internal][action]
        return nxt, self.labels[nxt]

    def base_transition_accuracy(self, true_table: np.ndarray) -> float:
        correct = 0
        for q in range(cfg.n_states):
            for a in range(cfg.n_actions):
                _, label = self.predicted_label(q, a)
                correct += int(label == int(true_table[q, a]))
        return correct / (cfg.n_states * cfg.n_actions)


class MDLGatedMachine:
    """Re-ground first; adopt only a compressed rule with independent confirmation."""

    def __init__(self) -> None:
        self.force_rule = 0
        self.table = TABLES[0].copy()
        self.current = 0
        self.buffer: Deque[Tuple[int, int, int]] = deque(maxlen=cfg.buffer_size)
        self.disagreements = 0
        self.regrounds = 0
        self.adoptions = 0
        self.adoption_time: Optional[int] = None
        self.confirmation_queries_used = 0
        self.candidate_tests = 0
        self.rejected_candidates = 0
        self.last_mdl_gain = 0.0

    @property
    def state_count(self) -> int:
        return cfg.n_states

    def reset_observed_state(self, q: int) -> None:
        self.current = int(q)

    def _errors(self, table: np.ndarray) -> int:
        return sum(int(table[q, a] != nxt) for q, a, nxt in self.buffer)

    def _best_candidate(self) -> Tuple[Optional[int], float, int, int]:
        current_errors = self._errors(self.table)
        best_force: Optional[int] = None
        best_gain = -float("inf")
        best_errors = current_errors
        for force in (-1, 0, 1):
            if force == self.force_rule:
                continue
            errors = self._errors(TABLES[force])
            gain = (current_errors - errors) * cfg.error_code_bits - cfg.structural_rule_bits
            if gain > best_gain:
                best_force, best_gain, best_errors = force, gain, errors
        return best_force, best_gain, current_errors, best_errors

    def _confirm(self, candidate_force: int, actual_force: int, generator: np.random.Generator) -> bool:
        differing = np.argwhere(self.table != TABLES[candidate_force])
        if len(differing) < cfg.confirmation_queries:
            return False
        ids = generator.choice(len(differing), size=cfg.confirmation_queries, replace=False)
        correct = 0
        for idx in ids:
            q, a = differing[int(idx)]
            observed = int(TABLES[actual_force][q, a])
            correct += int(observed == int(TABLES[candidate_force][q, a]))
        self.confirmation_queries_used += cfg.confirmation_queries
        return correct / cfg.confirmation_queries >= cfg.confirmation_required_fraction

    def step(
        self,
        action: int,
        observed_next: int,
        actual_force_for_queries: int,
        t: int,
        generator: np.random.Generator,
    ) -> bool:
        source = self.current
        predicted = int(self.table[source, action])
        disagreement = predicted != observed_next
        self.buffer.append((source, int(action), int(observed_next)))
        if disagreement:
            self.disagreements += 1
            self.regrounds += 1
        # Re-grounding is unconditional after observing the next state. It prevents
        # stale-state trajectory error from becoming evidence about delta.
        self.current = int(observed_next)

        if len(self.buffer) >= cfg.min_buffer:
            candidate, gain, current_errors, _ = self._best_candidate()
            self.last_mdl_gain = float(gain)
            if (
                candidate is not None
                and current_errors >= cfg.min_candidate_errors
                and gain >= cfg.mdl_min_gain_bits
            ):
                self.candidate_tests += 1
                if self._confirm(candidate, actual_force_for_queries, generator):
                    self.force_rule = int(candidate)
                    self.table = TABLES[self.force_rule].copy()
                    self.adoptions += 1
                    if self.adoption_time is None:
                        self.adoption_time = int(t)
                    self.buffer.clear()
                else:
                    self.rejected_candidates += 1
        return disagreement

    def base_transition_accuracy(self, true_table: np.ndarray) -> float:
        return float(np.mean(self.table == true_table))


def bfs_plan_naive(
    machine: NaiveExceptionMachine,
    start_q: int,
    goal_q: int,
    max_depth: int,
) -> Optional[List[int]]:
    if start_q == goal_q:
        return []
    queue: Deque[Tuple[int, List[int]]] = deque([(int(start_q), [])])
    visited = {int(start_q)}
    while queue:
        internal, plan = queue.popleft()
        if len(plan) >= max_depth:
            continue
        for action in range(cfg.n_actions):
            nxt, label = machine.predicted_label(internal, action)
            next_plan = plan + [action]
            if label == goal_q:
                return next_plan
            if nxt not in visited:
                visited.add(nxt)
                queue.append((nxt, next_plan))
    return None


def bfs_plan_table(
    table: np.ndarray,
    start_q: int,
    goal_q: int,
    max_depth: int,
) -> Optional[List[int]]:
    if start_q == goal_q:
        return []
    queue: Deque[Tuple[int, List[int]]] = deque([(int(start_q), [])])
    visited = {int(start_q)}
    while queue:
        q, plan = queue.popleft()
        if len(plan) >= max_depth:
            continue
        for action in range(cfg.n_actions):
            nxt = int(table[q, action])
            next_plan = plan + [action]
            if nxt == goal_q:
                return next_plan
            if nxt not in visited:
                visited.add(nxt)
                queue.append((nxt, next_plan))
    return None


def execute_plan(start_q: int, plan: Sequence[int], true_table: np.ndarray) -> int:
    q = int(start_q)
    for action in plan:
        q = int(true_table[q, int(action)])
    return q


def planning_success(
    naive: NaiveExceptionMachine,
    gated: MDLGatedMachine,
    true_force: int,
    generator: np.random.Generator,
) -> Dict[str, float]:
    table = TABLES[true_force]
    results = {"naive": [], "gated": [], "oracle": []}
    attempts = 0
    while len(results["oracle"]) < cfg.planning_tasks and attempts < cfg.planning_tasks * 20:
        attempts += 1
        start = int(generator.integers(0, cfg.n_states))
        goal = int(generator.integers(0, cfg.n_states))
        oracle_plan = bfs_plan_table(table, start, goal, cfg.planning_max_depth)
        if oracle_plan is None:
            continue
        naive_plan = bfs_plan_naive(naive, start, goal, cfg.planning_max_depth)
        gated_plan = bfs_plan_table(gated.table, start, goal, cfg.planning_max_depth)
        results["oracle"].append(float(execute_plan(start, oracle_plan, table) == goal))
        results["naive"].append(float(naive_plan is not None and execute_plan(start, naive_plan, table) == goal))
        results["gated"].append(float(gated_plan is not None and execute_plan(start, gated_plan, table) == goal))
    if not results["oracle"]:
        raise RuntimeError("No reachable planning tasks sampled.")
    return {name: float(np.mean(values)) for name, values in results.items()}


def run_stream(condition: str, trial: int) -> Dict[str, object]:
    generator = np.random.default_rng(cfg.seed + 10000 * (condition == "mixed") + trial)
    if condition == "transient":
        shock_times = separated_times(
            cfg.transient_shocks, 8, cfg.horizon - 8, cfg.min_shock_gap, generator
        )
        permanent_force = 0
        permanent_onset = cfg.horizon + 1
    elif condition == "mixed":
        shock_times = separated_times(
            cfg.mixed_transient_shocks, 8, cfg.permanent_onset - 8, cfg.min_shock_gap, generator
        )
        permanent_force = int(generator.choice([-1, 1]))
        permanent_onset = cfg.permanent_onset
    else:
        raise ValueError(condition)

    naive = NaiveExceptionMachine()
    gated = MDLGatedMachine()
    q = int(generator.integers(0, cfg.n_states))
    naive.reset_observed_state(q)
    gated.reset_observed_state(q)
    checkpoint_rows: List[Dict[str, float]] = []

    def record(t: int, active_force: int) -> None:
        true_table = TABLES[active_force]
        checkpoint_rows.append(
            {
                "condition": condition,
                "trial": float(trial),
                "time": float(t),
                "naive_states": float(naive.state_count),
                "gated_states": float(gated.state_count),
                "naive_transition_accuracy": naive.base_transition_accuracy(true_table),
                "gated_transition_accuracy": gated.base_transition_accuracy(true_table),
            }
        )

    record(0, 0)
    for t in range(cfg.horizon):
        action = int(generator.integers(0, cfg.n_actions))
        regime_force = permanent_force if t >= permanent_onset else 0
        applied_force = regime_force
        if t in shock_times:
            bird_force = choose_manifest_force(q, action, generator)
            if bird_force == 0:
                # Make the shock manifest without leaking any information to a learner.
                alternatives = [a for a in range(cfg.n_actions) if choose_manifest_force(q, a, generator) != 0]
                if alternatives:
                    action = int(generator.choice(alternatives))
                    bird_force = choose_manifest_force(q, action, generator)
            applied_force = bird_force
        q_next = int(TABLES[applied_force][q, action])
        naive.step(action, q_next)
        gated.step(action, q_next, regime_force, t, generator)
        q = q_next
        if (t + 1) in cfg.checkpoints:
            record(t + 1, regime_force)

    final_force = permanent_force if condition == "mixed" else 0
    planning = planning_success(naive, gated, final_force, generator)
    summary = {
        "condition": condition,
        "trial": float(trial),
        "permanent_force": float(permanent_force),
        "naive_final_states": float(naive.state_count),
        "gated_final_states": float(gated.state_count),
        "naive_refinements": float(naive.refinements),
        "gated_adoptions": float(gated.adoptions),
        "gated_adoption_time": float(gated.adoption_time) if gated.adoption_time is not None else float("nan"),
        "gated_adoption_delay": (
            float(gated.adoption_time - permanent_onset)
            if gated.adoption_time is not None and condition == "mixed"
            else float("nan")
        ),
        "gated_regrounds": float(gated.regrounds),
        "gated_candidate_tests": float(gated.candidate_tests),
        "gated_rejected_candidates": float(gated.rejected_candidates),
        "gated_confirmation_queries": float(gated.confirmation_queries_used),
        "naive_final_transition_accuracy": naive.base_transition_accuracy(TABLES[final_force]),
        "gated_final_transition_accuracy": gated.base_transition_accuracy(TABLES[final_force]),
        "naive_planning_success": planning["naive"],
        "gated_planning_success": planning["gated"],
        "oracle_planning_success": planning["oracle"],
    }
    return {"summary": summary, "checkpoints": checkpoint_rows}


def mean_ci(values: Sequence[float]) -> Tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    valid = arr[np.isfinite(arr)]
    if len(valid) == 0:
        return float("nan"), float("nan")
    mean = float(np.mean(valid))
    if len(valid) == 1:
        return mean, float("nan")
    return mean, float(1.96 * np.std(valid, ddof=1) / math.sqrt(len(valid)))


print("=" * 100)
print("RUNTIME REFINEMENT V4 — disagreement, re-grounding, compression gate")
print("=" * 100)
print(f"fast_dev={FAST_DEV_RUN} | trials={cfg.trials} | horizon={cfg.horizon} | states={cfg.n_states}")
print(
    "nominal overlap with permanent rules: "
    f"wind=-1 {np.mean(TABLES[0] == TABLES[-1]):.2%} | "
    f"wind=+1 {np.mean(TABLES[0] == TABLES[1]):.2%}"
)

started = time.perf_counter()
all_summaries: List[Dict[str, float]] = []
all_checkpoints: List[Dict[str, float]] = []
for condition in ("transient", "mixed"):
    for trial in range(cfg.trials):
        result = run_stream(condition, trial)
        all_summaries.append(result["summary"])
        all_checkpoints.extend(result["checkpoints"])
    print(f"finished {condition}: {cfg.trials} trials")
elapsed = time.perf_counter() - started


def summarize_condition(condition: str) -> Dict[str, Dict[str, float]]:
    rows = [r for r in all_summaries if r["condition"] == condition]
    keys = [
        "naive_final_states",
        "gated_final_states",
        "naive_refinements",
        "gated_adoptions",
        "gated_adoption_delay",
        "gated_regrounds",
        "gated_candidate_tests",
        "gated_rejected_candidates",
        "gated_confirmation_queries",
        "naive_final_transition_accuracy",
        "gated_final_transition_accuracy",
        "naive_planning_success",
        "gated_planning_success",
    ]
    result: Dict[str, Dict[str, float]] = {}
    for key in keys:
        mean, ci = mean_ci([float(r[key]) for r in rows])
        result[key] = {"mean": mean, "ci95_half_width": ci}
    if condition == "mixed":
        result["gated_revision_rate"] = {
            "mean": float(np.mean([np.isfinite(float(r["gated_adoption_time"])) for r in rows])),
            "ci95_half_width": float("nan"),
        }
    else:
        result["gated_false_revision_rate"] = {
            "mean": float(np.mean([float(r["gated_adoptions"]) > 0 for r in rows])),
            "ci95_half_width": float("nan"),
        }
    return result


aggregate = {condition: summarize_condition(condition) for condition in ("transient", "mixed")}
print("\nAGGREGATE RESULTS (mean ± 95% CI half-width)")
print("-" * 100)
for condition in ("transient", "mixed"):
    a = aggregate[condition]
    print(condition.upper())
    print(
        f"  states: naive={a['naive_final_states']['mean']:.2f}±{a['naive_final_states']['ci95_half_width']:.2f} | "
        f"gated={a['gated_final_states']['mean']:.2f}±{a['gated_final_states']['ci95_half_width']:.2f}"
    )
    print(
        f"  transition accuracy: naive={a['naive_final_transition_accuracy']['mean']:.2%} | "
        f"gated={a['gated_final_transition_accuracy']['mean']:.2%}"
    )
    print(
        f"  planning success: naive={a['naive_planning_success']['mean']:.2%}±{a['naive_planning_success']['ci95_half_width']:.2%} | "
        f"gated={a['gated_planning_success']['mean']:.2%}±{a['gated_planning_success']['ci95_half_width']:.2%}"
    )
    if condition == "transient":
        print(f"  gated false-revision rate={a['gated_false_revision_rate']['mean']:.2%}")
    else:
        print(
            f"  gated revision rate={a['gated_revision_rate']['mean']:.2%} | "
            f"adoption delay={a['gated_adoption_delay']['mean']:.2f}±{a['gated_adoption_delay']['ci95_half_width']:.2f} steps"
        )

# Audit assertions make silent failures explicit.
transient_rows = [r for r in all_summaries if r["condition"] == "transient"]
mixed_rows = [r for r in all_summaries if r["condition"] == "mixed"]
assert all(r["gated_final_states"] == cfg.n_states for r in all_summaries)
assert all(r["naive_final_states"] >= cfg.n_states for r in all_summaries)
assert all(r["gated_adoptions"] == 0 for r in transient_rows), "False structural adoption under transient shocks."
assert all(np.isfinite(r["gated_adoption_time"]) for r in mixed_rows), "Missed permanent revision."
assert all(r["gated_adoptions"] == 1 for r in mixed_rows), "Permanent regime should require one adoption."
assert all(abs(r["gated_final_transition_accuracy"] - 1.0) < 1e-12 for r in all_summaries)

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
with (out_dir / "config.json").open("w", encoding="utf-8") as f:
    json.dump(asdict(cfg), f, indent=2)
with (out_dir / "results.json").open("w", encoding="utf-8") as f:
    json.dump({"aggregate": aggregate, "elapsed_seconds": elapsed}, f, indent=2, allow_nan=True)
with (out_dir / "trial_results.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(all_summaries[0].keys()))
    writer.writeheader()
    writer.writerows(all_summaries)
with (out_dir / "trajectories.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(all_checkpoints[0].keys()))
    writer.writeheader()
    writer.writerows(all_checkpoints)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = {"naive": "#E45756", "gated": "#54A24B"}
for col, condition in enumerate(("transient", "mixed")):
    rows = [r for r in all_checkpoints if r["condition"] == condition]
    times = sorted({int(r["time"]) for r in rows})
    ax = axes[0, col]
    for method in ("naive", "gated"):
        means, cis = [], []
        for t in times:
            values = [r[f"{method}_states"] for r in rows if int(r["time"]) == t]
            mean, ci = mean_ci(values)
            means.append(mean)
            cis.append(ci)
        means_np, cis_np = np.asarray(means), np.asarray(cis)
        ax.plot(times, means_np, label=method, color=colors[method], lw=2)
        ax.fill_between(times, means_np - cis_np, means_np + cis_np, color=colors[method], alpha=0.18)
    if condition == "mixed":
        ax.axvline(cfg.permanent_onset, ls="--", color="black", lw=1.3, label="permanent change")
    ax.set_title(f"{condition}: machine size")
    ax.set_xlabel("world transitions")
    ax.set_ylabel("symbolic states")
    ax.grid(alpha=0.25)
    ax.legend()

    ax = axes[1, col]
    summaries = [r for r in all_summaries if r["condition"] == condition]
    transition = [
        np.mean([r["naive_final_transition_accuracy"] for r in summaries]),
        np.mean([r["gated_final_transition_accuracy"] for r in summaries]),
    ]
    planning = [
        np.mean([r["naive_planning_success"] for r in summaries]),
        np.mean([r["gated_planning_success"] for r in summaries]),
    ]
    x = np.arange(2)
    width = 0.34
    ax.bar(x - width / 2, transition, width, label="transition accuracy", color="#4C78A8")
    ax.bar(x + width / 2, planning, width, label="planning success", color="#F2CF5B")
    ax.set_xticks(x, ["naive", "gated"])
    ax.set_ylim(0, 1.05)
    ax.set_title(f"{condition}: final fidelity")
    ax.set_ylabel("fraction")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

fig.suptitle(
    "Runtime refinement: counterexample memorization vs re-grounding + MDL adoption",
    fontsize=15,
)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(out_dir / "summary.png", dpi=180, bbox_inches="tight")
plt.show()

bundle_base = out_dir.parent / f"{out_dir.name}_run_bundle"
archive_path = shutil.make_archive(str(bundle_base), "zip", root_dir=out_dir)
target_bundle = out_dir / "run_bundle.zip"
shutil.copy2(archive_path, target_bundle)
Path(archive_path).unlink(missing_ok=True)

print("\n" + "=" * 100)
print("FINAL V4 SUMMARY")
print("=" * 100)
print(f"elapsed={elapsed:.2f}s")
for condition in ("transient", "mixed"):
    a = aggregate[condition]
    print(
        f"{condition}: states naive/gated={a['naive_final_states']['mean']:.2f}/{a['gated_final_states']['mean']:.2f} | "
        f"planning naive/gated={a['naive_planning_success']['mean']:.2%}/{a['gated_planning_success']['mean']:.2%}"
    )
print(f"artifacts: {out_dir}")
print(f"bundle: {target_bundle}")



## Experiment 6: Calibrated compression gate V5

Original cell `5`.


In [ ]:
"""
CALIBRATED COMPRESSION GATE V5 — SINGLE-CELL COLAB EXPERIMENT

Purpose
-------
Calibrate the runtime rule-adoption gate instead of reporting one hand-picked
operating point. The experiment sweeps:

  rolling buffer size      = {24, 48, 96} transitions
  required MDL gain        = {0, 12, 24} bits beyond model cost
  discriminating support   = {4, 12, 24} consecutive net-support events
  confirmation probe count = {0, 4, 8}

against four stress profiles and two conditions (transient-only and a permanent
change after the midpoint). It adds the missing frozen/no-adaptation control.

Predeclared calibration rule
----------------------------
An operating point is feasible only if, in EVERY stress profile:

  transient false-adoption rate <= 2%
  pre-change false-adoption rate <= 2%
  permanent correct-final-rule rate >= 95%

Among feasible points, select the one with the smallest worst-profile mean
adoption delay; break ties using fewer mean confirmation queries, then smaller
buffer, higher MDL threshold, and higher persistence support. This rule is fixed
before results are produced.

Scope boundary
--------------
The candidate rule family is supplied: global force in {-1, 0, +1}. This tests
gate calibration under noise and coherent transients, not autonomous rule invention.

Set CALIBRATED_GATE_V5_FAST_DEV_RUN=1 for a reduced execution-path smoke test.
"""

from __future__ import annotations

import csv
import json
import math
import os
import random
import shutil
import time
from collections import deque
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Deque, Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np


FAST_DEV_RUN = os.environ.get("CALIBRATED_GATE_V5_FAST_DEV_RUN", "0") == "1"


@dataclass(frozen=True)
class Config:
    seed: int = 5507
    n_pos: int = 13
    v_max: int = 2
    n_actions: int = 3
    horizon: int = 600
    permanent_onset: int = 300
    trials: int = 64
    buffer_sizes: Tuple[int, ...] = (24, 48, 96)
    mdl_gain_thresholds: Tuple[float, ...] = (0.0, 12.0, 24.0)
    persistence_support_requirements: Tuple[int, ...] = (4, 12, 24)
    confirmation_probe_budgets: Tuple[int, ...] = (0, 4, 8)
    minimum_buffer_records: int = 12
    minimum_current_errors: int = 3
    structural_rule_bits: float = 12.0
    error_code_bits: float = math.log2(65)
    probe_cooldown: int = 8
    planning_max_depth: int = 24
    maximum_false_adoption_rate: float = 0.02
    minimum_correct_recovery_rate: float = 0.95
    output_dir: str = "outputs/06-calibrated-compression-gate-v5/calibrated_gate_v5_results"

    @property
    def n_vel(self) -> int:
        return 2 * self.v_max + 1

    @property
    def n_states(self) -> int:
        return self.n_pos * self.n_vel


cfg = Config()
if FAST_DEV_RUN:
    cfg = replace(
        cfg,
        horizon=200,
        permanent_onset=100,
        trials=8,
        buffer_sizes=(24, 48),
        mdl_gain_thresholds=(0.0, 24.0),
        persistence_support_requirements=(4, 24),
        confirmation_probe_budgets=(0, 8),
        maximum_false_adoption_rate=0.15,
        minimum_correct_recovery_rate=0.75,
        output_dir=str(Path.cwd() / "calibrated_gate_v5_smoke"),
    )
output_override = os.environ.get("CALIBRATED_GATE_V5_OUTPUT_DIR")
if output_override:
    cfg = replace(cfg, output_dir=output_override)

random.seed(cfg.seed)
np.random.seed(cfg.seed)
ACTION_VALUES = np.asarray([-1, 0, 1], dtype=np.int64)
FORCES = (-1, 0, 1)
FORCE_TO_INDEX = {-1: 0, 0: 1, 1: 2}


@dataclass(frozen=True)
class StressProfile:
    name: str
    local_shocks: int = 0
    global_bursts: int = 0
    burst_min_duration: int = 10
    burst_max_duration: int = 18
    observation_error_probability: float = 0.0
    probe_error_probability: float = 0.0


PROFILES = (
    StressProfile("sparse-impulses", local_shocks=20),
    StressProfile("dense-impulses", local_shocks=60),
    StressProfile("coherent-bursts", global_bursts=8),
    StressProfile(
        "noisy-observer-probes",
        local_shocks=32,
        observation_error_probability=0.02,
        probe_error_probability=0.10,
    ),
)


def encode_state(position: int, velocity: int) -> int:
    return int(position * cfg.n_vel + (velocity + cfg.v_max))


def decode_state(state: int) -> Tuple[int, int]:
    return int(state // cfg.n_vel), int(state % cfg.n_vel - cfg.v_max)


def physics_step(state: int, action: int, force: int = 0) -> int:
    position, velocity = decode_state(state)
    velocity_next = int(
        np.clip(velocity + int(ACTION_VALUES[action]) + int(force), -cfg.v_max, cfg.v_max)
    )
    position_next = position + velocity_next
    if position_next < 0:
        position_next = -position_next
        velocity_next = -velocity_next
    if position_next >= cfg.n_pos:
        position_next = 2 * (cfg.n_pos - 1) - position_next
        velocity_next = -velocity_next
    return encode_state(position_next, velocity_next)


def make_table(force: int) -> np.ndarray:
    return np.asarray(
        [
            [physics_step(q, a, force) for a in range(cfg.n_actions)]
            for q in range(cfg.n_states)
        ],
        dtype=np.int64,
    )


TABLES = {force: make_table(force) for force in FORCES}


def choose_manifest_local_force(q: int, action: int, generator: np.random.Generator) -> int:
    candidates = [force for force in (-1, 1) if TABLES[force][q, action] != TABLES[0][q, action]]
    return int(generator.choice(candidates)) if candidates else 0


def separated_starts(
    count: int,
    low: int,
    high: int,
    minimum_gap: int,
    generator: np.random.Generator,
) -> List[int]:
    candidates = list(range(low, high))
    generator.shuffle(candidates)
    selected: List[int] = []
    for t in candidates:
        if all(abs(t - prior) >= minimum_gap for prior in selected):
            selected.append(int(t))
            if len(selected) == count:
                break
    if len(selected) != count:
        raise RuntimeError(f"Could schedule only {len(selected)}/{count} separated events.")
    return sorted(selected)


def build_schedule(
    profile: StressProfile,
    condition: str,
    generator: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, int]:
    """Return local-event mask, global transient force, persistent force, onset."""
    local = np.zeros(cfg.horizon, dtype=bool)
    transient_global = np.zeros(cfg.horizon, dtype=np.int64)
    if condition == "transient":
        event_end = cfg.horizon - 8
        permanent_force = 0
        onset = cfg.horizon + 1
        fraction = 1.0
    elif condition == "permanent":
        event_end = cfg.permanent_onset - 8
        permanent_force = int(generator.choice([-1, 1]))
        onset = cfg.permanent_onset
        fraction = 0.5
    else:
        raise ValueError(condition)

    scale = 0.25 if FAST_DEV_RUN else 1.0
    local_count = max(1, int(round(profile.local_shocks * fraction * scale))) if profile.local_shocks else 0
    if local_count:
        starts = separated_starts(local_count, 8, event_end, 6, generator)
        local[starts] = True

    burst_count = max(1, int(round(profile.global_bursts * fraction * scale))) if profile.global_bursts else 0
    if burst_count:
        direction = int(generator.choice([-1, 1]))
        max_duration = profile.burst_max_duration
        starts = separated_starts(
            burst_count,
            8,
            max(9, event_end - max_duration),
            max_duration + 8,
            generator,
        )
        for start in starts:
            duration = int(generator.integers(profile.burst_min_duration, profile.burst_max_duration + 1))
            transient_global[start : min(start + duration, event_end)] = direction

    persistent = np.zeros(cfg.horizon, dtype=np.int64)
    if condition == "permanent":
        persistent[cfg.permanent_onset :] = permanent_force
    return local, transient_global, persistent, permanent_force


@dataclass(frozen=True)
class GateParameters:
    buffer_size: int
    mdl_gain_threshold: float
    persistence_support: int
    confirmation_probes: int

    @property
    def name(self) -> str:
        return (
            f"B{self.buffer_size}-G{self.mdl_gain_threshold:g}-"
            f"S{self.persistence_support}-Q{self.confirmation_probes}"
        )


OPERATING_POINTS = tuple(
    GateParameters(buffer_size, gain, support, probes)
    for buffer_size in cfg.buffer_sizes
    for gain in cfg.mdl_gain_thresholds
    for support in cfg.persistence_support_requirements
    for probes in cfg.confirmation_probe_budgets
)


class CalibratedGate:
    def __init__(self, params: GateParameters) -> None:
        self.params = params
        self.force_rule = 0
        self.observed_state = 0
        self.buffer: Deque[np.ndarray] = deque(maxlen=params.buffer_size)
        self.error_sums = np.zeros(3, dtype=np.int64)
        self.adoptions: List[Tuple[int, int, int]] = []
        self.query_count = 0
        self.rejections = 0
        self.regrounds = 0
        self.next_test_time = 0
        self.support_counts = np.zeros(3, dtype=np.int64)

    def reset(self, observed_state: int) -> None:
        self.observed_state = int(observed_state)

    def _append_record(self, q: int, action: int, observed_next: int) -> np.ndarray:
        errors = np.asarray(
            [TABLES[force][q, action] != observed_next for force in FORCES],
            dtype=np.int64,
        )
        if len(self.buffer) == self.params.buffer_size:
            self.error_sums -= self.buffer[0]
        self.buffer.append(errors)
        self.error_sums += errors
        current_error = int(errors[FORCE_TO_INDEX[self.force_rule]])
        for force in FORCES:
            if force == self.force_rule:
                continue
            idx = FORCE_TO_INDEX[force]
            candidate_error = int(errors[idx])
            if current_error == 1 and candidate_error == 0:
                self.support_counts[idx] += 1
            elif current_error == 0 and candidate_error == 1:
                self.support_counts[idx] = 0
        return errors

    def _best_candidate(self) -> Tuple[Optional[int], float, int]:
        current_index = FORCE_TO_INDEX[self.force_rule]
        current_errors = int(self.error_sums[current_index])
        best_force: Optional[int] = None
        best_gain = -float("inf")
        for force in FORCES:
            if force == self.force_rule:
                continue
            errors = int(self.error_sums[FORCE_TO_INDEX[force]])
            gain = (current_errors - errors) * cfg.error_code_bits - cfg.structural_rule_bits
            if gain > best_gain:
                best_force, best_gain = force, float(gain)
        return best_force, best_gain, current_errors

    def _confirm(
        self,
        candidate: int,
        query_force: int,
        probe_error_probability: float,
        generator: np.random.Generator,
    ) -> bool:
        probes = self.params.confirmation_probes
        if probes == 0:
            return True
        differing = np.argwhere(TABLES[self.force_rule] != TABLES[candidate])
        ids = generator.choice(len(differing), size=probes, replace=False)
        self.query_count += probes
        for idx in ids:
            q, action = differing[int(idx)]
            observed = int(TABLES[query_force][q, action])
            if generator.random() < probe_error_probability:
                observed = int(generator.integers(0, cfg.n_states - 1))
                if observed >= int(TABLES[query_force][q, action]):
                    observed += 1
            if observed != int(TABLES[candidate][q, action]):
                return False
        return True

    def step(
        self,
        action: int,
        observed_next: int,
        query_force: int,
        probe_error_probability: float,
        t: int,
        generator: np.random.Generator,
    ) -> None:
        source = self.observed_state
        prediction = int(TABLES[self.force_rule][source, action])
        if prediction != observed_next:
            self.regrounds += 1
        self._append_record(source, action, observed_next)
        self.observed_state = int(observed_next)

        if len(self.buffer) < cfg.minimum_buffer_records or t < self.next_test_time:
            return
        candidate, gain, current_errors = self._best_candidate()
        if (
            candidate is None
            or current_errors < cfg.minimum_current_errors
            or gain < self.params.mdl_gain_threshold
            or self.support_counts[FORCE_TO_INDEX[candidate]] < self.params.persistence_support
        ):
            return
        if self._confirm(candidate, query_force, probe_error_probability, generator):
            old = self.force_rule
            self.force_rule = int(candidate)
            self.adoptions.append((int(t), int(old), int(candidate)))
            self.buffer.clear()
            self.error_sums.fill(0)
            self.support_counts.fill(0)
        else:
            self.rejections += 1
            self.next_test_time = int(t + cfg.probe_cooldown)


def corrupt_observation(true_state: int, probability: float, generator: np.random.Generator) -> int:
    if generator.random() >= probability:
        return int(true_state)
    replacement = int(generator.integers(0, cfg.n_states - 1))
    return replacement + int(replacement >= true_state)


def bfs_plan(table: np.ndarray, start: int, goal: int) -> Optional[List[int]]:
    if start == goal:
        return []
    queue: Deque[Tuple[int, List[int]]] = deque([(int(start), [])])
    visited = {int(start)}
    while queue:
        q, plan = queue.popleft()
        if len(plan) >= cfg.planning_max_depth:
            continue
        for action in range(cfg.n_actions):
            nxt = int(table[q, action])
            next_plan = plan + [action]
            if nxt == goal:
                return next_plan
            if nxt not in visited:
                visited.add(nxt)
                queue.append((nxt, next_plan))
    return None


def execute(start: int, plan: Sequence[int], table: np.ndarray) -> int:
    q = int(start)
    for action in plan:
        q = int(table[q, int(action)])
    return q


def exact_planning_matrix() -> Dict[Tuple[int, int], float]:
    matrix: Dict[Tuple[int, int], float] = {}
    for model_force in FORCES:
        for true_force in FORCES:
            successes: List[float] = []
            for start in range(cfg.n_states):
                for goal in range(cfg.n_states):
                    oracle = bfs_plan(TABLES[true_force], start, goal)
                    if oracle is None:
                        continue
                    plan = bfs_plan(TABLES[model_force], start, goal)
                    successes.append(
                        float(plan is not None and execute(start, plan, TABLES[true_force]) == goal)
                    )
            matrix[(model_force, true_force)] = float(np.mean(successes))
    return matrix


PLANNING_MATRIX = exact_planning_matrix()


def run_trial(
    params: GateParameters,
    profile: StressProfile,
    condition: str,
    trial: int,
) -> Dict[str, object]:
    op_index = OPERATING_POINTS.index(params)
    profile_index = PROFILES.index(profile)
    condition_index = 0 if condition == "transient" else 1
    # World randomness is identical across operating points. Confirmation-probe
    # randomness is separate so query budgets cannot perturb the world stream.
    world_seed = cfg.seed + 10_000 * profile_index + 1_000 * condition_index + trial
    generator = np.random.default_rng(world_seed)
    gate_generator = np.random.default_rng(world_seed + 1_000_000 * (op_index + 1))
    local_events, transient_global, persistent, permanent_force = build_schedule(
        profile, condition, generator
    )
    gate = CalibratedGate(params)
    true_q = int(generator.integers(0, cfg.n_states))
    observed_q = corrupt_observation(true_q, profile.observation_error_probability, generator)
    gate.reset(observed_q)
    false_prechange_adoption = False
    wrong_postchange_adoption = False
    correct_adoption_time: Optional[int] = None
    postchange_invalid_steps = 0
    manifest_local_events = 0

    for t in range(cfg.horizon):
        action = int(generator.integers(0, cfg.n_actions))
        background_force = int(persistent[t])
        global_force = int(transient_global[t])
        path_force = global_force if global_force != 0 else background_force
        query_force = path_force
        if local_events[t]:
            local_force = choose_manifest_local_force(true_q, action, generator)
            if local_force == 0:
                valid_actions = [
                    a for a in range(cfg.n_actions)
                    if choose_manifest_local_force(true_q, a, generator) != 0
                ]
                action = int(generator.choice(valid_actions))
                local_force = choose_manifest_local_force(true_q, action, generator)
            path_force = int(local_force)
            query_force = background_force  # Local bird strike does not affect reset probes.
        true_next = int(TABLES[path_force][true_q, action])
        if local_events[t]:
            assert true_next != int(TABLES[background_force][true_q, action])
            manifest_local_events += 1
        observed_next = corrupt_observation(
            true_next, profile.observation_error_probability, generator
        )
        if condition == "permanent" and t >= cfg.permanent_onset and gate.force_rule != permanent_force:
            postchange_invalid_steps += 1
        adoption_count_before = len(gate.adoptions)
        gate.step(
            action,
            observed_next,
            query_force,
            profile.probe_error_probability,
            t,
            gate_generator,
        )
        if len(gate.adoptions) > adoption_count_before:
            _, _, adopted = gate.adoptions[-1]
            if condition == "transient" or t < cfg.permanent_onset:
                if adopted != 0:
                    false_prechange_adoption = True
            elif adopted != permanent_force:
                wrong_postchange_adoption = True
            elif correct_adoption_time is None:
                correct_adoption_time = int(t)
        true_q = true_next

    final_true_force = 0 if condition == "transient" else permanent_force
    final_accuracy = float(np.mean(TABLES[gate.force_rule] == TABLES[final_true_force]))
    final_planning = PLANNING_MATRIX[(gate.force_rule, final_true_force)]
    frozen_accuracy = float(np.mean(TABLES[0] == TABLES[final_true_force]))
    frozen_planning = PLANNING_MATRIX[(0, final_true_force)]
    return {
        "operating_point": params.name,
        "buffer_size": params.buffer_size,
        "mdl_gain_threshold": params.mdl_gain_threshold,
        "persistence_support": params.persistence_support,
        "confirmation_probes": params.confirmation_probes,
        "profile": profile.name,
        "condition": condition,
        "trial": trial,
        "permanent_force": permanent_force,
        "false_prechange_adoption": float(false_prechange_adoption),
        "wrong_postchange_adoption": float(wrong_postchange_adoption),
        "correct_final_rule": float(gate.force_rule == final_true_force),
        "correct_adoption_delay": (
            float(correct_adoption_time - cfg.permanent_onset)
            if correct_adoption_time is not None and condition == "permanent"
            else float("nan")
        ),
        "postchange_invalid_steps": float(postchange_invalid_steps),
        "adoption_count": float(len(gate.adoptions)),
        "confirmation_queries": float(gate.query_count),
        "rejected_candidates": float(gate.rejections),
        "regrounds": float(gate.regrounds),
        "manifest_local_events": float(manifest_local_events),
        "final_transition_accuracy": final_accuracy,
        "final_planning_success": final_planning,
        "frozen_transition_accuracy": frozen_accuracy,
        "frozen_planning_success": frozen_planning,
    }


def mean_ci(values: Sequence[float]) -> Tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return float("nan"), float("nan")
    mean = float(np.mean(arr))
    if len(arr) == 1:
        return mean, float("nan")
    return mean, float(1.96 * np.std(arr, ddof=1) / math.sqrt(len(arr)))


print("=" * 108)
print("CALIBRATED COMPRESSION GATE V5 — delay, false adoption, and query cost")
print("=" * 108)
print(
    f"fast_dev={FAST_DEV_RUN} | states={cfg.n_states} | operating_points={len(OPERATING_POINTS)} | "
    f"profiles={len(PROFILES)} | trials/cell={cfg.trials}"
)
print(
    "nominal overlap with changed regimes: "
    f"wind=-1 {np.mean(TABLES[0] == TABLES[-1]):.2%} | "
    f"wind=+1 {np.mean(TABLES[0] == TABLES[1]):.2%}"
)

started = time.perf_counter()
trial_rows: List[Dict[str, object]] = []
for op_number, params in enumerate(OPERATING_POINTS, start=1):
    for profile in PROFILES:
        for condition in ("transient", "permanent"):
            for trial in range(cfg.trials):
                trial_rows.append(run_trial(params, profile, condition, trial))
    print(f"completed {op_number:2d}/{len(OPERATING_POINTS)}: {params.name}")
elapsed = time.perf_counter() - started


stress_rows: List[Dict[str, object]] = []
for params in OPERATING_POINTS:
    for profile in PROFILES:
        for condition in ("transient", "permanent"):
            rows = [
                row for row in trial_rows
                if row["operating_point"] == params.name
                and row["profile"] == profile.name
                and row["condition"] == condition
            ]
            delay_mean, delay_ci = mean_ci([float(row["correct_adoption_delay"]) for row in rows])
            stress_rows.append(
                {
                    "operating_point": params.name,
                    "buffer_size": params.buffer_size,
                    "mdl_gain_threshold": params.mdl_gain_threshold,
                    "persistence_support": params.persistence_support,
                    "confirmation_probes": params.confirmation_probes,
                    "profile": profile.name,
                    "condition": condition,
                    "false_prechange_adoption_rate": float(np.mean([row["false_prechange_adoption"] for row in rows])),
                    "wrong_postchange_adoption_rate": float(np.mean([row["wrong_postchange_adoption"] for row in rows])),
                    "correct_final_rule_rate": float(np.mean([row["correct_final_rule"] for row in rows])),
                    "mean_adoption_delay": delay_mean,
                    "adoption_delay_ci95_half_width": delay_ci,
                    "mean_postchange_invalid_steps": float(np.mean([row["postchange_invalid_steps"] for row in rows])),
                    "mean_confirmation_queries": float(np.mean([row["confirmation_queries"] for row in rows])),
                    "mean_final_transition_accuracy": float(np.mean([row["final_transition_accuracy"] for row in rows])),
                    "mean_final_planning_success": float(np.mean([row["final_planning_success"] for row in rows])),
                    "mean_frozen_transition_accuracy": float(np.mean([row["frozen_transition_accuracy"] for row in rows])),
                    "mean_frozen_planning_success": float(np.mean([row["frozen_planning_success"] for row in rows])),
                }
            )


operating_rows: List[Dict[str, object]] = []
for params in OPERATING_POINTS:
    rows = [row for row in stress_rows if row["operating_point"] == params.name]
    transient = [row for row in rows if row["condition"] == "transient"]
    permanent = [row for row in rows if row["condition"] == "permanent"]
    worst_transient_false = max(float(row["false_prechange_adoption_rate"]) for row in transient)
    worst_prechange_false = max(float(row["false_prechange_adoption_rate"]) for row in permanent)
    worst_recovery = min(float(row["correct_final_rule_rate"]) for row in permanent)
    finite_delays = [float(row["mean_adoption_delay"]) for row in permanent if np.isfinite(float(row["mean_adoption_delay"]))]
    worst_delay = max(finite_delays) if len(finite_delays) == len(permanent) else float("inf")
    mean_queries = float(np.mean([row["mean_confirmation_queries"] for row in permanent]))
    feasible = (
        worst_transient_false <= cfg.maximum_false_adoption_rate
        and worst_prechange_false <= cfg.maximum_false_adoption_rate
        and worst_recovery >= cfg.minimum_correct_recovery_rate
        and np.isfinite(worst_delay)
    )
    operating_rows.append(
        {
            "operating_point": params.name,
            "buffer_size": params.buffer_size,
            "mdl_gain_threshold": params.mdl_gain_threshold,
            "persistence_support": params.persistence_support,
            "confirmation_probes": params.confirmation_probes,
            "feasible": bool(feasible),
            "worst_transient_false_adoption_rate": worst_transient_false,
            "worst_prechange_false_adoption_rate": worst_prechange_false,
            "worst_permanent_recovery_rate": worst_recovery,
            "worst_profile_mean_adoption_delay": worst_delay,
            "mean_permanent_confirmation_queries": mean_queries,
            "mean_permanent_planning_success": float(np.mean([row["mean_final_planning_success"] for row in permanent])),
        }
    )

feasible_rows = [row for row in operating_rows if row["feasible"]]
if not feasible_rows:
    raise RuntimeError("No operating point met the predeclared safety/recovery constraints.")
selected = min(
    feasible_rows,
    key=lambda row: (
        float(row["worst_profile_mean_adoption_delay"]),
        float(row["mean_permanent_confirmation_queries"]),
        int(row["buffer_size"]),
        -float(row["mdl_gain_threshold"]),
        -int(row["persistence_support"]),
    ),
)

print("\nOPERATING-POINT CALIBRATION")
print("-" * 108)
print(" point       ok | false(trans/pre) | recovery | worst delay | queries | permanent planning")
for row in sorted(
    operating_rows,
    key=lambda x: (not bool(x["feasible"]), float(x["worst_profile_mean_adoption_delay"])),
):
    delay = row["worst_profile_mean_adoption_delay"]
    delay_text = f"{delay:11.2f}" if np.isfinite(delay) else f"{'miss':>11s}"
    print(
        f" {row['operating_point']:11s} {str(row['feasible']):>4s} | "
        f"{100*row['worst_transient_false_adoption_rate']:5.1f}%/"
        f"{100*row['worst_prechange_false_adoption_rate']:5.1f}% | "
        f"{100*row['worst_permanent_recovery_rate']:7.1f}% | {delay_text} | "
        f"{row['mean_permanent_confirmation_queries']:7.2f} | "
        f"{100*row['mean_permanent_planning_success']:8.2f}%"
    )

print("\nSELECTED GATE")
print("-" * 108)
print(json.dumps(selected, indent=2))
selected_stress = [row for row in stress_rows if row["operating_point"] == selected["operating_point"]]
print("\nSELECTED-GATE STRESS AUDIT")
print("-" * 108)
for row in selected_stress:
    print(
        f"{row['profile']:24s} {row['condition']:9s} | "
        f"false={100*row['false_prechange_adoption_rate']:5.1f}% | "
        f"recover={100*row['correct_final_rule_rate']:6.1f}% | "
        f"delay={row['mean_adoption_delay']:6.2f} | "
        f"plan={100*row['mean_final_planning_success']:6.1f}% | "
        f"frozen-plan={100*row['mean_frozen_planning_success']:6.1f}%"
    )

# Audits.
assert selected["worst_transient_false_adoption_rate"] <= cfg.maximum_false_adoption_rate
assert selected["worst_prechange_false_adoption_rate"] <= cfg.maximum_false_adoption_rate
assert selected["worst_permanent_recovery_rate"] >= cfg.minimum_correct_recovery_rate
assert all(row["manifest_local_events"] > 0 for row in trial_rows if "impulses" in str(row["profile"]) or "noisy" in str(row["profile"]))

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
with (out_dir / "config.json").open("w", encoding="utf-8") as f:
    json.dump({**asdict(cfg), "profiles": [asdict(profile) for profile in PROFILES]}, f, indent=2)
with (out_dir / "results.json").open("w", encoding="utf-8") as f:
    json.dump(
        {
            "selected_gate": selected,
            "selected_stress_audit": selected_stress,
            "operating_points": operating_rows,
            "elapsed_seconds": elapsed,
        },
        f,
        indent=2,
        allow_nan=True,
    )
for filename, rows in (
    ("trial_results.csv", trial_rows),
    ("stress_results.csv", stress_rows),
    ("operating_points.csv", operating_rows),
):
    with (out_dir / filename).open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
planning_rows = [
    {
        "model_force": model_force,
        "true_force": true_force,
        "planning_success": PLANNING_MATRIX[(model_force, true_force)],
        "transition_accuracy": float(np.mean(TABLES[model_force] == TABLES[true_force])),
    }
    for model_force in FORCES
    for true_force in FORCES
]
with (out_dir / "planning_matrix.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(planning_rows[0].keys()))
    writer.writeheader()
    writer.writerows(planning_rows)

# Summary figure.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ax = axes[0, 0]
for probes, color in zip(cfg.confirmation_probe_budgets, ("#E45756", "#F2CF5B", "#54A24B")):
    rows = [row for row in operating_rows if int(row["confirmation_probes"]) == probes]
    ax.scatter(
        [100 * float(row["worst_transient_false_adoption_rate"]) for row in rows],
        [float(row["worst_profile_mean_adoption_delay"]) for row in rows],
        s=[35 + int(row["buffer_size"]) for row in rows],
        alpha=0.8,
        label=f"{probes} probes",
        color=color,
    )
ax.scatter(
    100 * float(selected["worst_transient_false_adoption_rate"]),
    float(selected["worst_profile_mean_adoption_delay"]),
    marker="*",
    s=280,
    color="black",
    label="selected",
    zorder=10,
)
ax.axvline(100 * cfg.maximum_false_adoption_rate, ls="--", color="gray")
ax.set_title("Gate trade surface")
ax.set_xlabel("worst transient false-adoption rate (%)")
ax.set_ylabel("worst-profile mean adoption delay")
ax.grid(alpha=0.25)
ax.legend()

permanent_selected = [row for row in selected_stress if row["condition"] == "permanent"]
names = [str(row["profile"]) for row in permanent_selected]
x = np.arange(len(names))
ax = axes[0, 1]
ax.bar(x - 0.18, [row["mean_final_planning_success"] for row in permanent_selected], 0.36, label="selected gate", color="#54A24B")
ax.bar(x + 0.18, [row["mean_frozen_planning_success"] for row in permanent_selected], 0.36, label="frozen machine", color="#9D755D")
ax.set_xticks(x, names, rotation=18, ha="right")
ax.set_ylim(0, 1.05)
ax.set_title("Permanent change: final planning")
ax.set_ylabel("success fraction")
ax.grid(axis="y", alpha=0.25)
ax.legend()

ax = axes[1, 0]
ax.bar(x, [row["mean_adoption_delay"] for row in permanent_selected], color="#4C78A8")
ax.set_xticks(x, names, rotation=18, ha="right")
ax.set_title("Selected gate: adoption delay")
ax.set_ylabel("world transitions")
ax.grid(axis="y", alpha=0.25)

ax = axes[1, 1]
transient_selected = [row for row in selected_stress if row["condition"] == "transient"]
ax.bar(
    np.arange(len(transient_selected)) - 0.18,
    [100 * row["false_prechange_adoption_rate"] for row in transient_selected],
    0.36,
    label="transient false adoption",
    color="#E45756",
)
ax.bar(
    np.arange(len(permanent_selected)) + 0.18,
    [100 * row["correct_final_rule_rate"] for row in permanent_selected],
    0.36,
    label="permanent recovery",
    color="#54A24B",
)
ax.set_xticks(np.arange(len(names)), names, rotation=18, ha="right")
ax.set_ylim(0, 105)
ax.set_title("Selected gate: safety and recovery")
ax.set_ylabel("rate (%)")
ax.grid(axis="y", alpha=0.25)
ax.legend()

fig.suptitle(
    f"Calibrated compression gate — selected {selected['operating_point']}",
    fontsize=15,
)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(out_dir / "summary.png", dpi=180, bbox_inches="tight")
plt.show()

bundle_base = out_dir.parent / f"{out_dir.name}_run_bundle"
archive_path = shutil.make_archive(str(bundle_base), "zip", root_dir=out_dir)
target_bundle = out_dir / "run_bundle.zip"
shutil.copy2(archive_path, target_bundle)
Path(archive_path).unlink(missing_ok=True)

print("\n" + "=" * 108)
print("FINAL CALIBRATION SUMMARY")
print("=" * 108)
print(f"selected={selected['operating_point']} | elapsed={elapsed:.2f}s")
print(
    f"worst transient false adoption={100*selected['worst_transient_false_adoption_rate']:.2f}% | "
    f"worst permanent recovery={100*selected['worst_permanent_recovery_rate']:.2f}% | "
    f"worst mean delay={selected['worst_profile_mean_adoption_delay']:.2f}"
)
print(f"artifacts: {out_dir}")
print(f"bundle: {target_bundle}")


## Experiment 7: MDL-gated symbolic rule invention V6

Original cell `6`.

**Audit note.** The checked-in source already contains an ordered-union CSV schema fix, but the archived run predates that fix and stopped during export; rerun status is unverified.


In [ ]:
"""
OPTION 2 — MDL-GATED SYMBOLIC RULE INVENTION (V6), SINGLE-CELL COLAB EXPERIMENT

Question
--------
Once the calibrated patience gate decides that a change is persistent, can the
system invent the new finite dynamics rather than select from {-1, 0, +1}?

What is fixed from Option 1
---------------------------
The selected B24-G24-S24-Q0 operating point is frozen:

  buffer                         24 transitions
  required net MDL improvement  24 bits
  discriminating support        24 events
  gate confirmation probes       0

What changes in Option 2
------------------------
There is no hand-supplied list of candidate worlds. A small typed grammar builds
programs compositionally from constants, state/action signs, predicates, guarded
terms, and clipped addition. Programs are deduplicated by their complete 195-entry
transition semantics. MDL chooses among the generated programs.

Four controllers see matched streams:

  frozen   : never changes the original 65-state machine
  lookup   : patches only state-action pairs already contradicted
  passive  : fixed gate + grammar search, no reset queries
  active   : fixed gate + grammar search + <=16 synthesis-only falsification probes

The active probes do NOT help decide whether a change is temporary or permanent;
the frozen gate already made that decision. They only distinguish formulas that
fit the same passive buffer. Active adoption requires one remaining semantic table.

Tests
-----
Six permanent laws are representable by the grammar but never supplied to the
controller by name. Two deterministic laws are deliberately outside the grammar
and are reported as a boundary audit, not counted as successful invention.
Calibration-range transient bursts and local bird strikes occur before the change.

Primary endpoint
----------------
Exact semantic recovery: the invented program must reconstruct every transition
of the hidden 65 x 3 table. Textual AST equality is not required.

Interpretation boundary
-----------------------
This tests symbolic synthesis inside a supplied grammar, not autonomous ontology
discovery. Reset probes assume a simulator or safe experimental interface. The
unrepresentable audit diagnoses the need for Option 3; it does not prove a general
refusal guarantee.

Set OPTION2_V6_FAST_DEV_RUN=1 for a short execution-path smoke test.
"""

from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import random
import shutil
import time
from collections import deque
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Deque, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np


FAST_DEV_RUN = os.environ.get("OPTION2_V6_FAST_DEV_RUN", "0") == "1"


@dataclass(frozen=True)
class Config:
    seed: int = 6207
    n_pos: int = 13
    v_max: int = 2
    n_actions: int = 3
    horizon: int = 650
    permanent_onset: int = 250
    trials: int = 40

    # Frozen Option-1 gate: B24-G24-S24-Q0.
    buffer_size: int = 24
    mdl_gain_threshold_bits: float = 24.0
    persistence_support: int = 24
    gate_confirmation_probes: int = 0
    minimum_buffer_records: int = 12
    minimum_current_errors: int = 3
    error_code_bits: float = math.log2(65)

    # Option-2 synthesis protocol.
    active_probe_budget_per_attempt: int = 16
    active_probe_total_budget: int = 48
    rejection_cooldown: int = 8
    max_program_bits: float = 48.0

    # Pre-change stress remains inside the Option-1 calibration envelope.
    local_bird_strikes: int = 18
    coherent_bursts: int = 6
    burst_min_duration: int = 10
    burst_max_duration: int = 18

    planning_max_depth: int = 24
    maximum_false_prechange_adoption_rate: float = 0.02
    target_active_exact_recovery_rate: float = 0.90
    output_dir: str = "outputs/07-mdl-gated-symbolic-rule-invention-v6/option2_grammar_invention_v6_results"

    @property
    def n_vel(self) -> int:
        return 2 * self.v_max + 1

    @property
    def n_states(self) -> int:
        return self.n_pos * self.n_vel

    @property
    def n_cells(self) -> int:
        return self.n_states * self.n_actions


cfg = Config()
if FAST_DEV_RUN:
    cfg = replace(
        cfg,
        horizon=300,
        permanent_onset=120,
        trials=4,
        local_bird_strikes=6,
        coherent_bursts=2,
        target_active_exact_recovery_rate=0.50,
        maximum_false_prechange_adoption_rate=0.25,
        output_dir=str(Path.cwd() / "option2_grammar_invention_v6_smoke"),
    )
output_override = os.environ.get("OPTION2_V6_OUTPUT_DIR")
if output_override:
    cfg = replace(cfg, output_dir=output_override)

random.seed(cfg.seed)
np.random.seed(cfg.seed)
ACTION_VALUES = np.asarray([-1, 0, 1], dtype=np.int8)


def encode_state(position: int, velocity: int) -> int:
    return int(position * cfg.n_vel + velocity + cfg.v_max)


def decode_state(state: int) -> Tuple[int, int]:
    return int(state // cfg.n_vel), int(state % cfg.n_vel - cfg.v_max)


CELL_Q = np.repeat(np.arange(cfg.n_states, dtype=np.int64), cfg.n_actions)
CELL_A = np.tile(np.arange(cfg.n_actions, dtype=np.int64), cfg.n_states)
CELL_P = CELL_Q // cfg.n_vel
CELL_V = CELL_Q % cfg.n_vel - cfg.v_max
CELL_AV = ACTION_VALUES[CELL_A]


def physics_step(state: int, action: int, residual_force: int = 0) -> int:
    position, velocity = decode_state(state)
    velocity_next = int(
        np.clip(
            velocity + int(ACTION_VALUES[action]) + int(residual_force),
            -cfg.v_max,
            cfg.v_max,
        )
    )
    position_next = position + velocity_next
    if position_next < 0:
        position_next = -position_next
        velocity_next = -velocity_next
    if position_next >= cfg.n_pos:
        position_next = 2 * (cfg.n_pos - 1) - position_next
        velocity_next = -velocity_next
    return encode_state(position_next, velocity_next)


def residual_to_table(residual: np.ndarray) -> np.ndarray:
    flat = np.asarray(
        [
            physics_step(int(q), int(a), int(force))
            for q, a, force in zip(CELL_Q, CELL_A, residual)
        ],
        dtype=np.int16,
    )
    return flat.reshape(cfg.n_states, cfg.n_actions)


BASE_RESIDUAL = np.zeros(cfg.n_cells, dtype=np.int8)
BASE_TABLE = residual_to_table(BASE_RESIDUAL)


@dataclass(frozen=True)
class Node:
    text: str
    bits: float
    values: np.ndarray


def semantic_key(values: np.ndarray) -> bytes:
    return np.asarray(values, dtype=np.int8).tobytes()


def keep_cheapest(store: Dict[bytes, Node], node: Node) -> None:
    key = semantic_key(node.values)
    old = store.get(key)
    if old is None or (node.bits, len(node.text), node.text) < (old.bits, len(old.text), old.text):
        store[key] = node


def clipped_add(left: Node, right: Node) -> Node:
    return Node(
        text=f"clip({left.text} + {right.text}, -2, 2)",
        bits=float(3.0 + left.bits + right.bits),
        values=np.clip(left.values + right.values, -2, 2).astype(np.int8),
    )


def build_grammar() -> Tuple[List[Node], np.ndarray]:
    """Generate programs from syntax, then collapse semantic duplicates."""
    sign_v = np.sign(CELL_V).astype(np.int8)
    sign_a = np.sign(CELL_AV).astype(np.int8)
    terms = [
        Node("-1", 4.0, -np.ones(cfg.n_cells, dtype=np.int8)),
        Node("+1", 4.0, np.ones(cfg.n_cells, dtype=np.int8)),
        Node("sgn(v)", 5.0, sign_v),
        Node("-sgn(v)", 6.0, -sign_v),
        Node("sgn(a)", 5.0, sign_a),
        Node("-sgn(a)", 6.0, -sign_a),
    ]
    predicates: List[Tuple[str, np.ndarray]] = []
    for threshold in (4, 7, 10):
        predicates.append((f"p < {threshold}", CELL_P < threshold))
        predicates.append((f"p >= {threshold}", CELL_P >= threshold))
    predicates.extend(
        [
            ("v < 0", CELL_V < 0),
            ("v == 0", CELL_V == 0),
            ("v > 0", CELL_V > 0),
            ("a < 0", CELL_AV < 0),
            ("a == 0", CELL_AV == 0),
            ("a > 0", CELL_AV > 0),
        ]
    )

    clauses: List[Node] = []
    for predicate_text, mask in predicates:
        for term in terms:
            clauses.append(
                Node(
                    text=f"({term.text} if {predicate_text} else 0)",
                    bits=float(3.0 + 5.0 + term.bits),
                    values=np.where(mask, term.values, 0).astype(np.int8),
                )
            )

    residual_programs: Dict[bytes, Node] = {}
    keep_cheapest(residual_programs, Node("0", 0.0, BASE_RESIDUAL.copy()))
    for node in terms + clauses:
        keep_cheapest(residual_programs, node)
    # Bounded compositional normal form: two atoms, atom+clause, or two clauses.
    for i, left in enumerate(terms):
        for right in terms[i:]:
            keep_cheapest(residual_programs, clipped_add(left, right))
    for left in terms:
        for right in clauses:
            keep_cheapest(residual_programs, clipped_add(left, right))
    for i, left in enumerate(clauses):
        for right in clauses[i:]:
            keep_cheapest(residual_programs, clipped_add(left, right))

    # Different residual programs can still be observationally identical because
    # velocity clipping and wall reflection hide some force differences. Keep the
    # cheapest expression for each complete transition table.
    by_table: Dict[bytes, Tuple[Node, np.ndarray]] = {}
    for node in residual_programs.values():
        if node.bits > cfg.max_program_bits:
            continue
        table = residual_to_table(node.values)
        key = table.astype(np.int16).tobytes()
        old = by_table.get(key)
        if old is None or (node.bits, len(node.text), node.text) < (
            old[0].bits,
            len(old[0].text),
            old[0].text,
        ):
            by_table[key] = (node, table)

    ordered = sorted(
        by_table.values(),
        key=lambda item: (item[0].bits, len(item[0].text), item[0].text),
    )
    programs = [item[0] for item in ordered]
    tables = np.stack([item[1].reshape(-1) for item in ordered]).astype(np.int16)
    return programs, tables


PROGRAMS, CANDIDATE_TABLES = build_grammar()
PROGRAM_BITS = np.asarray([program.bits for program in PROGRAMS], dtype=np.float32)
PROGRAM_TEXT = [program.text for program in PROGRAMS]
BASELINE_INDEX = next(
    i for i in range(len(PROGRAMS)) if np.array_equal(CANDIDATE_TABLES[i], BASE_TABLE.reshape(-1))
)
assert PROGRAM_TEXT[BASELINE_INDEX] == "0"
assert len({table.tobytes() for table in CANDIDATE_TABLES}) == len(CANDIDATE_TABLES)


def clause(mask: np.ndarray, term: np.ndarray) -> np.ndarray:
    return np.where(mask, term, 0).astype(np.int8)


ONES = np.ones(cfg.n_cells, dtype=np.int8)
SIGN_V = np.sign(CELL_V).astype(np.int8)
SIGN_A = np.sign(CELL_AV).astype(np.int8)


@dataclass(frozen=True)
class Regime:
    name: str
    representable: bool
    residual: np.ndarray
    table: np.ndarray
    grammar_index: Optional[int]


def make_regime(name: str, residual: np.ndarray, representable: bool) -> Regime:
    residual = np.clip(np.asarray(residual, dtype=np.int8), -2, 2)
    table = residual_to_table(residual)
    matches = np.flatnonzero(np.all(CANDIDATE_TABLES == table.reshape(1, -1), axis=1))
    grammar_index = int(matches[0]) if len(matches) else None
    if representable:
        assert grammar_index is not None, f"Representable target missing from grammar: {name}"
    else:
        assert grammar_index is None, f"Negative-control target accidentally entered grammar: {name}"
    return Regime(name, representable, residual, table, grammar_index)


def build_regimes() -> Tuple[Regime, ...]:
    center_seek = np.clip(
        clause(CELL_P < 7, ONES) + clause(CELL_P >= 7, -ONES), -2, 2
    )
    velocity_brake = np.clip(
        clause(CELL_V > 0, -ONES) + clause(CELL_V < 0, ONES), -2, 2
    )
    upper_brake = clause(CELL_P >= 7, -ONES)
    coupled = np.clip(SIGN_A - SIGN_V, -2, 2)

    parity = np.where((CELL_P + CELL_V + CELL_AV) % 2 == 0, 1, -1).astype(np.int8)
    random_generator = np.random.default_rng(cfg.seed + 99173)
    random_local = random_generator.choice([-1, 1], size=cfg.n_cells).astype(np.int8)
    regimes = (
        make_regime("global-wind", ONES, True),
        make_regime("velocity-drag", -SIGN_V, True),
        make_regime("action-coupling", SIGN_A, True),
        make_regime("center-seeking", center_seek, True),
        make_regime("upper-half-brake", upper_brake, True),
        make_regime("brake-plus-action", coupled, True),
        make_regime("parity-checkerboard", parity, False),
        make_regime("random-local-table", random_local, False),
    )
    return regimes


REGIMES = build_regimes()


def separated_starts(
    count: int,
    low: int,
    high: int,
    minimum_gap: int,
    generator: np.random.Generator,
) -> List[int]:
    candidates = list(range(low, high))
    generator.shuffle(candidates)
    selected: List[int] = []
    for t in candidates:
        if all(abs(t - previous) >= minimum_gap for previous in selected):
            selected.append(int(t))
            if len(selected) == count:
                break
    if len(selected) != count:
        raise RuntimeError(f"Could schedule only {len(selected)}/{count} events")
    return sorted(selected)


def build_prechange_stress(generator: np.random.Generator) -> Tuple[np.ndarray, np.ndarray]:
    local = np.zeros(cfg.horizon, dtype=bool)
    burst_force = np.zeros(cfg.horizon, dtype=np.int8)
    event_end = cfg.permanent_onset - 6
    if cfg.local_bird_strikes:
        starts = separated_starts(
            cfg.local_bird_strikes, 5, event_end, 5, generator
        )
        local[starts] = True
    if cfg.coherent_bursts:
        starts = separated_starts(
            cfg.coherent_bursts,
            5,
            event_end - cfg.burst_max_duration,
            cfg.burst_max_duration + 5,
            generator,
        )
        direction = int(generator.choice([-1, 1]))
        for start in starts:
            duration = int(
                generator.integers(cfg.burst_min_duration, cfg.burst_max_duration + 1)
            )
            burst_force[start : start + duration] = direction
    return local, burst_force


def transition_with_impulse(
    regime: Regime,
    q: int,
    action: int,
    impulse: int,
) -> int:
    cell = q * cfg.n_actions + action
    return physics_step(q, action, int(regime.residual[cell]) + int(impulse))


class LookupRepair:
    def __init__(self) -> None:
        self.table = BASE_TABLE.copy()
        self.patches = 0

    def observe(self, q: int, action: int, observed_next: int) -> None:
        if int(self.table[q, action]) != int(observed_next):
            self.table[q, action] = int(observed_next)
            self.patches += 1


class GrammarSynthesizer:
    def __init__(self, active: bool) -> None:
        self.active = bool(active)
        self.current_index = BASELINE_INDEX
        self.buffer: Deque[Tuple[np.ndarray, int]] = deque(maxlen=cfg.buffer_size)
        self.error_sums = np.zeros(len(PROGRAMS), dtype=np.int16)
        self.current_error_sum = 0
        self.support = np.zeros(len(PROGRAMS), dtype=np.int16)
        self.adoption_time: Optional[int] = None
        self.adopted_index: Optional[int] = None
        self.trigger_time: Optional[int] = None
        self.query_count = 0
        self.rejections = 0
        self.ambiguous_attempts = 0
        self.next_attempt_time = 0
        self.last_version_size = 0

    @property
    def adopted(self) -> bool:
        return self.adopted_index is not None

    def table(self) -> np.ndarray:
        return CANDIDATE_TABLES[self.current_index].reshape(cfg.n_states, cfg.n_actions)

    def _append(self, errors: np.ndarray, current_error: int) -> None:
        if len(self.buffer) == cfg.buffer_size:
            old_errors, old_current = self.buffer[0]
            self.error_sums -= old_errors
            self.current_error_sum -= int(old_current)
        compact = errors.astype(np.int8, copy=True)
        self.buffer.append((compact, int(current_error)))
        self.error_sums += compact
        self.current_error_sum += int(current_error)

        support_mask = (current_error == 1) & (errors == 0)
        counter_mask = (current_error == 0) & (errors == 1)
        self.support[support_mask] += 1
        self.support[counter_mask] = 0

    def _eligible(self) -> Tuple[np.ndarray, np.ndarray]:
        score = self.error_sums.astype(np.float64) * cfg.error_code_bits + PROGRAM_BITS
        current_score = self.current_error_sum * cfg.error_code_bits
        gain = current_score - score
        eligible = (
            (np.arange(len(PROGRAMS)) != self.current_index)
            & (self.support >= cfg.persistence_support)
            & (gain >= cfg.mdl_gain_threshold_bits)
        )
        return np.flatnonzero(eligible), score

    @staticmethod
    def _best_index(indices: np.ndarray, score: np.ndarray) -> int:
        return int(
            min(
                (int(i) for i in indices),
                key=lambda i: (float(score[i]), float(PROGRAM_BITS[i]), PROGRAM_TEXT[i]),
            )
        )

    def _active_disambiguate(
        self,
        eligible: np.ndarray,
        score: np.ndarray,
        query_table: np.ndarray,
    ) -> Optional[int]:
        # All empirically best-fitting, persistent programs enter the version space;
        # MDL is a tie-breaker after falsification, not an excuse to hide alternatives.
        minimum_errors = int(np.min(self.error_sums[eligible]))
        version = eligible[self.error_sums[eligible] == minimum_errors]
        if len(version) == 0:
            return None
        remaining_total = cfg.active_probe_total_budget - self.query_count
        budget = min(cfg.active_probe_budget_per_attempt, remaining_total)
        queried_cells: set[int] = set()

        for _ in range(max(0, budget)):
            if len(version) <= 1:
                break
            best_cell: Optional[int] = None
            best_partition = -1.0
            for cell in range(cfg.n_cells):
                if cell in queried_cells:
                    continue
                predictions = CANDIDATE_TABLES[version, cell]
                _, counts = np.unique(predictions, return_counts=True)
                if len(counts) <= 1:
                    continue
                probabilities = counts / counts.sum()
                entropy = float(-np.sum(probabilities * np.log2(probabilities)))
                if entropy > best_partition:
                    best_partition = entropy
                    best_cell = cell
            if best_cell is None:
                break
            queried_cells.add(best_cell)
            observed = int(query_table.reshape(-1)[best_cell])
            self.query_count += 1
            keep = CANDIDATE_TABLES[version, best_cell] == observed
            version = version[keep]
            if len(version) == 0:
                self.rejections += 1
                return None

        self.last_version_size = int(len(version))
        if len(version) != 1:
            self.ambiguous_attempts += 1
            return None
        return self._best_index(version, score)

    def observe(
        self,
        q: int,
        action: int,
        observed_next: int,
        query_table: np.ndarray,
        t: int,
    ) -> None:
        if self.adopted:
            return
        cell = q * cfg.n_actions + action
        predictions = CANDIDATE_TABLES[:, cell]
        errors = predictions != int(observed_next)
        current_prediction = int(CANDIDATE_TABLES[self.current_index, cell])
        current_error = int(current_prediction != int(observed_next))
        self._append(errors, current_error)

        if (
            len(self.buffer) < cfg.minimum_buffer_records
            or self.current_error_sum < cfg.minimum_current_errors
            or t < self.next_attempt_time
        ):
            return
        eligible, score = self._eligible()
        if len(eligible) == 0:
            return
        if self.trigger_time is None:
            self.trigger_time = int(t)

        if self.active:
            selected = self._active_disambiguate(eligible, score, query_table)
            if selected is None:
                self.next_attempt_time = int(t + cfg.rejection_cooldown)
                return
        else:
            selected = self._best_index(eligible, score)

        self.current_index = int(selected)
        self.adopted_index = int(selected)
        self.adoption_time = int(t)


def bfs_plan(table: np.ndarray, start: int, goal: int) -> Optional[List[int]]:
    if start == goal:
        return []
    queue: Deque[Tuple[int, List[int]]] = deque([(int(start), [])])
    visited = {int(start)}
    while queue:
        q, plan = queue.popleft()
        if len(plan) >= cfg.planning_max_depth:
            continue
        for action in range(cfg.n_actions):
            nxt = int(table[q, action])
            next_plan = plan + [action]
            if nxt == goal:
                return next_plan
            if nxt not in visited:
                visited.add(nxt)
                queue.append((nxt, next_plan))
    return None


def execute_plan(start: int, plan: Sequence[int], table: np.ndarray) -> int:
    q = int(start)
    for action in plan:
        q = int(table[q, int(action)])
    return q


PLANNING_CACHE: Dict[Tuple[bytes, bytes], float] = {}


def exact_planning_success(model_table: np.ndarray, true_table: np.ndarray) -> float:
    key = (model_table.astype(np.int16).tobytes(), true_table.astype(np.int16).tobytes())
    cached = PLANNING_CACHE.get(key)
    if cached is not None:
        return cached
    outcomes: List[float] = []
    for start in range(cfg.n_states):
        for goal in range(cfg.n_states):
            oracle = bfs_plan(true_table, start, goal)
            if oracle is None:
                continue
            plan = bfs_plan(model_table, start, goal)
            outcomes.append(
                float(
                    plan is not None
                    and execute_plan(start, plan, true_table) == goal
                )
            )
    result = float(np.mean(outcomes))
    PLANNING_CACHE[key] = result
    return result


def choose_manifest_impulse(
    base_regime: Regime,
    q: int,
    action: int,
    generator: np.random.Generator,
) -> Tuple[int, int]:
    expected = int(base_regime.table[q, action])
    choices = []
    for impulse in (-2, -1, 1, 2):
        observed = transition_with_impulse(base_regime, q, action, impulse)
        if observed != expected:
            choices.append((impulse, observed))
    if not choices:
        return 0, expected
    impulse, observed = choices[int(generator.integers(0, len(choices)))]
    return int(impulse), int(observed)


def run_trial(regime: Regime, trial: int) -> Dict[str, object]:
    regime_index = REGIMES.index(regime)
    world_seed = cfg.seed + 100_000 * regime_index + trial
    generator = np.random.default_rng(world_seed)
    local_events, burst_force = build_prechange_stress(generator)

    lookup = LookupRepair()
    passive = GrammarSynthesizer(active=False)
    active = GrammarSynthesizer(active=True)
    q = int(generator.integers(0, cfg.n_states))
    postchange_cells: set[int] = set()
    manifest_bird_strikes = 0

    for t in range(cfg.horizon):
        action = int(generator.integers(0, cfg.n_actions))
        background = regime if t >= cfg.permanent_onset else NOMINAL_REGIME
        query_table = background.table

        if t < cfg.permanent_onset and int(burst_force[t]) != 0:
            force_residual = np.full(cfg.n_cells, int(burst_force[t]), dtype=np.int8)
            path_table = residual_to_table(force_residual)
            q_next = int(path_table[q, action])
        elif local_events[t]:
            impulse, q_next = choose_manifest_impulse(background, q, action, generator)
            if impulse == 0:
                valid_actions = []
                for candidate_action in range(cfg.n_actions):
                    candidate_impulse, _ = choose_manifest_impulse(
                        background, q, candidate_action, generator
                    )
                    if candidate_impulse != 0:
                        valid_actions.append(candidate_action)
                if valid_actions:
                    action = int(generator.choice(valid_actions))
                    impulse, q_next = choose_manifest_impulse(
                        background, q, action, generator
                    )
            if impulse != 0:
                manifest_bird_strikes += 1
        else:
            q_next = int(background.table[q, action])

        if t >= cfg.permanent_onset:
            postchange_cells.add(q * cfg.n_actions + action)

        lookup.observe(q, action, q_next)
        passive.observe(q, action, q_next, query_table, t)
        active.observe(q, action, q_next, query_table, t)
        q = int(q_next)

    controller_tables = {
        "frozen": BASE_TABLE,
        "lookup": lookup.table,
        "passive": passive.table(),
        "active": active.table(),
    }
    row: Dict[str, object] = {
        "regime": regime.name,
        "representable": float(regime.representable),
        "trial": trial,
        "target_expression": (
            PROGRAM_TEXT[regime.grammar_index] if regime.grammar_index is not None else "OUTSIDE_GRAMMAR"
        ),
        "target_program_bits": (
            float(PROGRAM_BITS[regime.grammar_index]) if regime.grammar_index is not None else float("nan")
        ),
        "postchange_coverage": len(postchange_cells) / cfg.n_cells,
        "manifest_bird_strikes": manifest_bird_strikes,
        "lookup_patches": lookup.patches,
    }
    for name, table in controller_tables.items():
        row[f"{name}_transition_accuracy"] = float(np.mean(table == regime.table))
        row[f"{name}_planning_success"] = exact_planning_success(table, regime.table)
        row[f"{name}_exact_semantic_recovery"] = float(np.array_equal(table, regime.table))

    for name, controller in (("passive", passive), ("active", active)):
        row[f"{name}_adopted"] = float(controller.adopted)
        row[f"{name}_false_prechange_adoption"] = float(
            controller.adoption_time is not None
            and controller.adoption_time < cfg.permanent_onset
        )
        row[f"{name}_adoption_delay"] = (
            float(controller.adoption_time - cfg.permanent_onset)
            if controller.adoption_time is not None
            else float("nan")
        )
        row[f"{name}_queries"] = float(controller.query_count)
        row[f"{name}_rejections"] = float(controller.rejections)
        row[f"{name}_ambiguous_attempts"] = float(controller.ambiguous_attempts)
        row[f"{name}_invented_expression"] = (
            PROGRAM_TEXT[controller.adopted_index]
            if controller.adopted_index is not None
            else "ABSTAIN"
        )
        row[f"{name}_invented_program_bits"] = (
            float(PROGRAM_BITS[controller.adopted_index])
            if controller.adopted_index is not None
            else float("nan")
        )
    return row


def mean_ci(values: Iterable[float]) -> Tuple[float, float]:
    arr = np.asarray(list(values), dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return float("nan"), float("nan")
    mean = float(np.mean(arr))
    if len(arr) == 1:
        return mean, float("nan")
    return mean, float(1.96 * np.std(arr, ddof=1) / math.sqrt(len(arr)))


def modal_expression(rows: Sequence[Dict[str, object]], controller: str) -> str:
    counts: Dict[str, int] = {}
    key = f"{controller}_invented_expression"
    for row in rows:
        expression = str(row[key])
        counts[expression] = counts.get(expression, 0) + 1
    return min(counts, key=lambda expression: (-counts[expression], expression))


NOMINAL_REGIME = Regime(
    name="nominal",
    representable=True,
    residual=BASE_RESIDUAL,
    table=BASE_TABLE,
    grammar_index=BASELINE_INDEX,
)

print("=" * 116)
print("OPTION 2 V6 — MDL-GATED SYMBOLIC RULE INVENTION")
print("=" * 116)
print(
    f"fast_dev={FAST_DEV_RUN} | states={cfg.n_states} | transitions={cfg.n_cells} | "
    f"grammar semantic programs={len(PROGRAMS)} | trials/regime={cfg.trials}"
)
print(
    "frozen gate: "
    f"B{cfg.buffer_size}-G{cfg.mdl_gain_threshold_bits:g}-"
    f"S{cfg.persistence_support}-Q{cfg.gate_confirmation_probes}"
)
for regime in REGIMES:
    if regime.grammar_index is None:
        print(f"target {regime.name:20s} | OUTSIDE GRAMMAR")
    else:
        print(
            f"target {regime.name:20s} | bits={PROGRAM_BITS[regime.grammar_index]:4.1f} | "
            f"canonical={PROGRAM_TEXT[regime.grammar_index]}"
        )

started = time.perf_counter()
trial_rows: List[Dict[str, object]] = []
for regime_number, regime in enumerate(REGIMES, start=1):
    for trial in range(cfg.trials):
        trial_rows.append(run_trial(regime, trial))
    print(f"completed {regime_number}/{len(REGIMES)}: {regime.name}")
elapsed = time.perf_counter() - started


rule_rows: List[Dict[str, object]] = []
for regime in REGIMES:
    rows = [row for row in trial_rows if row["regime"] == regime.name]
    summary: Dict[str, object] = {
        "regime": regime.name,
        "representable": regime.representable,
        "target_expression": rows[0]["target_expression"],
        "mean_postchange_coverage": float(np.mean([row["postchange_coverage"] for row in rows])),
    }
    for controller in ("frozen", "lookup", "passive", "active"):
        summary[f"{controller}_exact_recovery_rate"] = float(
            np.mean([row[f"{controller}_exact_semantic_recovery"] for row in rows])
        )
        summary[f"{controller}_mean_transition_accuracy"] = float(
            np.mean([row[f"{controller}_transition_accuracy"] for row in rows])
        )
        summary[f"{controller}_mean_planning_success"] = float(
            np.mean([row[f"{controller}_planning_success"] for row in rows])
        )
    for controller in ("passive", "active"):
        delay, delay_ci = mean_ci(row[f"{controller}_adoption_delay"] for row in rows)
        summary[f"{controller}_adoption_rate"] = float(
            np.mean([row[f"{controller}_adopted"] for row in rows])
        )
        summary[f"{controller}_false_prechange_rate"] = float(
            np.mean([row[f"{controller}_false_prechange_adoption"] for row in rows])
        )
        summary[f"{controller}_mean_adoption_delay"] = delay
        summary[f"{controller}_adoption_delay_ci95_half_width"] = delay_ci
        summary[f"{controller}_mean_queries"] = float(
            np.mean([row[f"{controller}_queries"] for row in rows])
        )
        summary[f"{controller}_modal_expression"] = modal_expression(rows, controller)
    rule_rows.append(summary)


representable_trials = [row for row in trial_rows if bool(row["representable"])]
unrepresentable_trials = [row for row in trial_rows if not bool(row["representable"])]
controller_rows: List[Dict[str, object]] = []
for controller in ("frozen", "lookup", "passive", "active"):
    row = {
        "controller": controller,
        "representable_exact_recovery_rate": float(
            np.mean(
                [
                    trial[f"{controller}_exact_semantic_recovery"]
                    for trial in representable_trials
                ]
            )
        ),
        "representable_mean_transition_accuracy": float(
            np.mean(
                [trial[f"{controller}_transition_accuracy"] for trial in representable_trials]
            )
        ),
        "representable_mean_planning_success": float(
            np.mean([trial[f"{controller}_planning_success"] for trial in representable_trials])
        ),
    }
    if controller in ("passive", "active"):
        row["false_prechange_adoption_rate"] = float(
            np.mean(
                [trial[f"{controller}_false_prechange_adoption"] for trial in trial_rows]
            )
        )
        row["representable_adoption_rate"] = float(
            np.mean([trial[f"{controller}_adopted"] for trial in representable_trials])
        )
        row["unrepresentable_adoption_rate"] = float(
            np.mean([trial[f"{controller}_adopted"] for trial in unrepresentable_trials])
        )
        row["mean_queries"] = float(
            np.mean([trial[f"{controller}_queries"] for trial in trial_rows])
        )
    controller_rows.append(row)


active_summary = next(row for row in controller_rows if row["controller"] == "active")
passive_summary = next(row for row in controller_rows if row["controller"] == "passive")
lookup_summary = next(row for row in controller_rows if row["controller"] == "lookup")
predictions = {
    "P1_active_exact_recovery_at_least_target": bool(
        active_summary["representable_exact_recovery_rate"]
        >= cfg.target_active_exact_recovery_rate
    ),
    "P2_active_beats_passive_exact_recovery": bool(
        active_summary["representable_exact_recovery_rate"]
        > passive_summary["representable_exact_recovery_rate"]
    ),
    "P3_active_beats_lookup_planning": bool(
        active_summary["representable_mean_planning_success"]
        > lookup_summary["representable_mean_planning_success"]
    ),
    "P4_false_prechange_at_most_calibrated_limit": bool(
        active_summary["false_prechange_adoption_rate"]
        <= cfg.maximum_false_prechange_adoption_rate
    ),
}


print("\nREPRESENTABLE-RULE AUDIT")
print("-" * 116)
print(
    " rule                 | active exact | passive exact | lookup trans | "
    "active plan | delay | queries | modal active invention"
)
for row in rule_rows:
    if not bool(row["representable"]):
        continue
    print(
        f" {row['regime']:20s} | "
        f"{100*row['active_exact_recovery_rate']:11.1f}% | "
        f"{100*row['passive_exact_recovery_rate']:12.1f}% | "
        f"{100*row['lookup_mean_transition_accuracy']:11.1f}% | "
        f"{100*row['active_mean_planning_success']:10.1f}% | "
        f"{row['active_mean_adoption_delay']:5.1f} | "
        f"{row['active_mean_queries']:7.1f} | "
        f"{row['active_modal_expression']}"
    )

print("\nOUT-OF-GRAMMAR BOUNDARY AUDIT")
print("-" * 116)
for row in rule_rows:
    if bool(row["representable"]):
        continue
    print(
        f" {row['regime']:20s} | active adopts={100*row['active_adoption_rate']:5.1f}% | "
        f"active accuracy={100*row['active_mean_transition_accuracy']:6.1f}% | "
        f"active planning={100*row['active_mean_planning_success']:6.1f}% | "
        f"modal={row['active_modal_expression']}"
    )

print("\nAGGREGATE CONTROLLER AUDIT — representable targets")
print("-" * 116)
for row in controller_rows:
    print(
        f" {row['controller']:7s} | exact={100*row['representable_exact_recovery_rate']:6.1f}% | "
        f"transition={100*row['representable_mean_transition_accuracy']:6.1f}% | "
        f"planning={100*row['representable_mean_planning_success']:6.1f}%"
    )

print("\nPREDECLARED PREDICTIONS")
print("-" * 116)
for name, passed in predictions.items():
    print(f" {name}: {'PASS' if passed else 'FAIL'}")

# Internal audits are implementation invariants, not research-outcome assertions.
assert all(row["manifest_bird_strikes"] > 0 for row in trial_rows)
assert all(0.0 <= row["active_transition_accuracy"] <= 1.0 for row in trial_rows)
assert all(row["active_queries"] <= cfg.active_probe_total_budget for row in trial_rows)
assert PROGRAM_TEXT[BASELINE_INDEX] == "0"


out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
with (out_dir / "config.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            **asdict(cfg),
            "frozen_gate": "B24-G24-S24-Q0",
            "representable_regimes": [r.name for r in REGIMES if r.representable],
            "unrepresentable_regimes": [r.name for r in REGIMES if not r.representable],
        },
        file,
        indent=2,
    )
with (out_dir / "results.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "predictions": predictions,
            "controllers": controller_rows,
            "rules": rule_rows,
            "grammar_programs": len(PROGRAMS),
            "elapsed_seconds": elapsed,
        },
        file,
        indent=2,
        allow_nan=True,
    )
for filename, rows in (
    ("trial_results.csv", trial_rows),
    ("rule_summary.csv", rule_rows),
    ("controller_summary.csv", controller_rows),
):
    with (out_dir / filename).open("w", newline="", encoding="utf-8") as file:
        # Aggregate controller rows intentionally have controller-specific fields.
        # Use the ordered union so sparse rows serialize as blank cells.
        fieldnames = list(rows[0].keys()) + sorted(
            {key for row in rows for key in row} - set(rows[0].keys())
        )
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

grammar_rows = [
    {
        "program_index": i,
        "program_bits": float(PROGRAM_BITS[i]),
        "expression": PROGRAM_TEXT[i],
        "transition_sha256": hashlib.sha256(
            CANDIDATE_TABLES[i].astype(np.int16).tobytes()
        ).hexdigest(),
    }
    for i in range(len(PROGRAMS))
]
with (out_dir / "grammar.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(grammar_rows[0].keys()))
    writer.writeheader()
    writer.writerows(grammar_rows)


representable_rule_rows = [row for row in rule_rows if bool(row["representable"])]
names = [str(row["regime"]) for row in representable_rule_rows]
x = np.arange(len(names))
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

ax = axes[0, 0]
width = 0.24
ax.bar(
    x - width,
    [row["lookup_mean_transition_accuracy"] for row in representable_rule_rows],
    width,
    label="lookup repair",
    color="#F2CF5B",
)
ax.bar(
    x,
    [row["passive_mean_transition_accuracy"] for row in representable_rule_rows],
    width,
    label="passive grammar",
    color="#4C78A8",
)
ax.bar(
    x + width,
    [row["active_mean_transition_accuracy"] for row in representable_rule_rows],
    width,
    label="active grammar",
    color="#54A24B",
)
ax.set_xticks(x, names, rotation=20, ha="right")
ax.set_ylim(0, 1.05)
ax.set_title("Complete transition-table accuracy")
ax.set_ylabel("accuracy")
ax.grid(axis="y", alpha=0.25)
ax.legend()

ax = axes[0, 1]
ax.bar(
    x - 0.18,
    [row["passive_exact_recovery_rate"] for row in representable_rule_rows],
    0.36,
    label="passive",
    color="#4C78A8",
)
ax.bar(
    x + 0.18,
    [row["active_exact_recovery_rate"] for row in representable_rule_rows],
    0.36,
    label="active falsification",
    color="#54A24B",
)
ax.set_xticks(x, names, rotation=20, ha="right")
ax.set_ylim(0, 1.05)
ax.set_title("Exact semantic rule recovery")
ax.set_ylabel("trial fraction")
ax.grid(axis="y", alpha=0.25)
ax.legend()

ax = axes[1, 0]
controllers = [row["controller"] for row in controller_rows]
ax.bar(
    np.arange(len(controllers)),
    [row["representable_mean_planning_success"] for row in controller_rows],
    color=["#9D755D", "#F2CF5B", "#4C78A8", "#54A24B"],
)
ax.set_xticks(np.arange(len(controllers)), controllers)
ax.set_ylim(0, 1.05)
ax.set_title("Planning after permanent change")
ax.set_ylabel("exact-goal success")
ax.grid(axis="y", alpha=0.25)

ax = axes[1, 1]
boundary_rows = [row for row in rule_rows if not bool(row["representable"])]
boundary_names = [str(row["regime"]) for row in boundary_rows]
bx = np.arange(len(boundary_names))
ax.bar(
    bx - 0.2,
    [row["active_adoption_rate"] for row in boundary_rows],
    0.4,
    label="active adoption",
    color="#E45756",
)
ax.bar(
    bx + 0.2,
    [row["active_mean_transition_accuracy"] for row in boundary_rows],
    0.4,
    label="final accuracy",
    color="#72B7B2",
)
ax.set_xticks(bx, boundary_names, rotation=15, ha="right")
ax.set_ylim(0, 1.05)
ax.set_title("Out-of-grammar boundary (diagnostic only)")
ax.grid(axis="y", alpha=0.25)
ax.legend()

fig.suptitle(
    "Option 2: fixed persistence gate + compositional symbolic rule invention",
    fontsize=15,
)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(out_dir / "summary.png", dpi=180, bbox_inches="tight")
plt.show()

bundle_base = out_dir.parent / f"{out_dir.name}_run_bundle"
archive_path = shutil.make_archive(str(bundle_base), "zip", root_dir=out_dir)
target_bundle = out_dir / "run_bundle.zip"
shutil.copy2(archive_path, target_bundle)
Path(archive_path).unlink(missing_ok=True)

print("\n" + "=" * 116)
print("FINAL OPTION-2 SUMMARY")
print("=" * 116)
print(f"elapsed={elapsed:.2f}s | grammar semantic programs={len(PROGRAMS)}")
print(
    f"active exact recovery={100*active_summary['representable_exact_recovery_rate']:.2f}% | "
    f"passive={100*passive_summary['representable_exact_recovery_rate']:.2f}%"
)
print(
    f"active planning={100*active_summary['representable_mean_planning_success']:.2f}% | "
    f"lookup={100*lookup_summary['representable_mean_planning_success']:.2f}% | "
    f"frozen={100*controller_rows[0]['representable_mean_planning_success']:.2f}%"
)
print(
    f"active prechange false adoption={100*active_summary['false_prechange_adoption_rate']:.2f}% | "
    f"out-of-grammar adoption={100*active_summary['unrepresentable_adoption_rate']:.2f}%"
)
print(f"artifacts: {out_dir}")
print(f"bundle: {target_bundle}")
